# Сравнение детекторов аномалий траектории



In [ ]:
TORCH_CUDA_INDEX = "https://download.pytorch.org/whl/cu124"
ALLOWED_STATUS = ("move", "anomaly")

# Test — ряд от первой точки штатного хода не левее доли TEST_SPLIT ряда; пул обучения — всё до test.
TEST_SPLIT = 0.6
# Обучающий блок пула: начало и длина в долях пула. Найден в train_size.ipynb на data/2.csv: самый короткий
# непрерывный блок с окнами, эпизодами аномалий, распределением признаков и покрытием режимов пула; модели-
# представители семейств на нём не хуже, чем на всём пуле.
TRAIN_BLOCK_START = 0.375
TRAIN_BLOCK_LENGTH = 0.40
MCC_BAND = 0.02
GRID_STEP_SEC = 10.0
TIME_GAP_SEC = 180.0
THRESHOLD_LOG_POINTS = 60
PLOT_TAIL = 50_000
MAP_FIGSIZE = (22.0, 11.0)
# True — строятся все карты; False — только текст и таблицы, в том числе таблицы метрик после каждой модели.
SHOW_PLOTS = True
# True — модели обучаются заново и сохраняются в TRAINED_MODELS_DIR; False — сохранённая модель загружается, если
# она обучена на той же обучающей выборке, иначе обучается и сохраняется. После правок кода модели — True.
FORCE_TRAIN = False
N_WINDOWS = 8_000
# Нейросети (LSTM, TCN, DONUT): ранняя остановка по ошибке на калибровочном блоке — последние окна по времени.
NN_MAX_EPOCHS = 300
NN_PATIENCE = 10
NN_MIN_DELTA = 1e-4
NN_LR_PATIENCE = 3
NN_LR_FACTOR = 0.5
NN_CALIB_FRACTION = 0.15
TORCH_BATCH_SIZE = 256
TORCH_INFER_BATCH = 512
TORCH_LR = 1e-3

## Чтение и предобработка данных

Исходный файл csv задаёт последовательность записей с полями времени, координат и статуса. После чтения строится очищенный ряд наблюдений и две пары выборок «обучение / тест» с разными долями обучения.

После загрузки CSV имеем индексы $j=0,\ldots,M-1$. Сырые поля: $t_j$ (time), $\varphi_j$ (lat), $\lambda_j$ (lon), $l_j$ (status). Допустимые значения метки: $l_j\in\{\mathrm{move},\mathrm{anomaly},\ldots\}$; в эксперименте остаются только $\mathrm{move}$ и $\mathrm{anomaly}$. Для метрик классификации положительный класс — аномалия ($l_i=\mathrm{anomaly}$); в столбце label она кодируется нулём, штатный ход — единицей.

1. Отбор столбцов. Оставляем только
$$(t_i,\varphi_i,\lambda_i,l_i)_{i=0}^{N-1},$$
где $t_i$ — время, $\varphi_i$ — широта, $\lambda_i$ — долгота, $l_i$ — метка класса.

2. Валидные координаты. Исключаются пропуски, нечисловые координаты, выход за допустимые пределы WGS-84 и точки «нулевого» GPS:
$$\mathcal{J}_1=\{i:\ \varphi_i,\lambda_i\ \text{конечны},\ -90\le\varphi_i\le 90,\ -180\le\lambda_i\le 180,\ (\varphi_i,\lambda_i)\neq(0,0)\}.$$

3. Бинарная задача. Остаются только штатный ход и аномалия:
$$\mathcal{J}_2=\{i\in\mathcal{J}_1:\ l_i\in\{\mathrm{move},\mathrm{anomaly}\}\}.$$

4. Упорядочивание. На $\mathcal{J}_2$ задаётся сортировка по возрастанию $t_i$. Индекс $i=0$ — первая валидная точка ряда.

5. Локальная ENU (м). Опорная точка — первая валидная точка: $(\varphi_0,\lambda_0)=(\varphi_{0},\lambda_{0})$ в нумерации после шагов 1–4. Координаты WGS-84 переводятся в плоскость «восток–север» (WGS-84 $\to$ ECEF $\to$ ENU):
$$(x_i,y_i)=(E_i,N_i)=\mathrm{ENU}(\varphi_i,\lambda_i\mid\varphi_0,\lambda_0).$$
Поля $\varphi_i,\lambda_i$ сохраняются; координаты $x_i$, $y_i$ добавляются в таблицу (столбцы x и y) (метка класса по-прежнему $l_i$).

6. Обрезка по крайним $\mathrm{move}$. Пусть
$$\mathcal{I}_{\mathrm{move}}=\{i:\ l_i=\mathrm{move}\},\quad i_{\min}=\min\mathcal{I}_{\mathrm{move}},\quad i_{\max}=\max\mathcal{I}_{\mathrm{move}}.$$
Рабочий ряд — срез $i_{\min}\le i\le i_{\max}$ (первая и последняя точки ряда — $\mathrm{move}$); координаты $(x_i,y_i)$ не пересчитываются.

7. Разбиение train / test. Для доли $\rho\in(0,1)$ (в блокноте $\rho=\texttt{TEST\_SPLIT}=0.6$) при длине ряда $N$ задаётся пороговый индекс
$$k_0=\lfloor\rho N\rfloor.$$
Граница разбиения — первый индекс штатного хода не левее $k_0$:
$$k=\min\{i\ge k_0:\ l_i=\mathrm{move}\}.$$
Если $l_{k_0}=\mathrm{move}$, то $k=k_0$. Тест: индексы $[k,N)$; пул обучения: $[0,k)$. Так граница не попадает внутрь чисто аномального фрагмента без соседнего $\mathrm{move}$.

8. Обучающий блок найден отдельным экспериментом: самый короткий блок, в котором окон для нейросетей и эпизодов аномалий достаточно, распределение признаков штатного хода совпадает с пулом и режимы движения пула покрыты; модели-представители семейств (фильтр Калмана, Isolation Forest, DBSCAN, HMM, MPCT) обучаются на нём не хуже, чем на всём пуле. Обучение и выбор порога на блоке быстрее в 2.4 раза.

In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path
from typing import Callable, Optional


def ensure_cuda_torch(index_url: str) -> None:
    """Ставит сборку torch с CUDA, если есть GPU NVIDIA, а установленный torch собран без CUDA.

    Вызывается до первого импорта torch в ядре: уже загруженный torch в процессе не заменить. Сборка torch
    проверяется в отдельном процессе, чтобы не загружать его в ядро раньше времени.

    Args:
        index_url: Индекс колёс PyTorch с нужной версией CUDA (``TORCH_CUDA_INDEX``).
    """
    if shutil.which("nvidia-smi") is None:
        return
    probe = subprocess.run(
        [sys.executable, "-c", "import torch; print(torch.version.cuda or '')"],
        capture_output=True,
        text=True,
        check=False,
    )
    if probe.returncode == 0 and probe.stdout.strip():
        return
    if "torch" in sys.modules:
        print("torch без CUDA уже загружен в этом ядре: перезапустите ядро и выполните блокнот заново")
        return
    print(f"Найден GPU NVIDIA, torch собран без CUDA: установка сборки с CUDA из {index_url}")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torch"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch", "--index-url", index_url], check=True)


# До импорта torch: при GPU NVIDIA ставится CUDA-сборка, иначе остаётся текущая и ниже выбирается CPU.
ensure_cuda_torch(TORCH_CUDA_INDEX)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from hmmlearn import _hmmc
from IPython.display import display
from numpy.linalg import LinAlgError
from scipy.linalg import solve_triangular
from scipy.optimize import linprog, minimize
from scipy.special import logsumexp
from scipy.stats import chi2
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import NearestNeighbors
from sklearn.svm import OneClassSVM
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.kalman_filter import KalmanFilter
from statsmodels.tsa.statespace.kalman_smoother import KalmanSmoother
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# Блокнот запускается из notebooks/: пакет app импортируется из корня проекта.
path_root = Path.cwd().parent
if str(path_root) not in sys.path:
    sys.path.insert(0, str(path_root))

from app.evaluation import TrajectoryMetrics, TrajectorySeries
from app.viz import plot_track_maps

path_data = path_root / "data" / "test.csv"
TRAINED_MODELS_DIR = path_root / "notebooks" / "trained_models"
if not TRAINED_MODELS_DIR.exists():
    TRAINED_MODELS_DIR.mkdir(parents=True)
    print(f"Создана папка моделей: {TRAINED_MODELS_DIR}")

get_ipython().run_line_magic("matplotlib", "inline")

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False

SEED = 42
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    DEVICE = torch.device("cuda:0")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
else:
    DEVICE = torch.device("cpu")
torch.set_num_threads(max(1, min(8, os.cpu_count() or 1)))

print(f"Корень: {path_root}")
print(f"Файл: {path_data}")
print(f"Модели: {TRAINED_MODELS_DIR}, обучение {'заново (FORCE_TRAIN)' if FORCE_TRAIN else 'при отсутствии файла'}")
print(f"PyTorch {torch.__version__}, устройство {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(DEVICE)}")

# Преобразование координат WGS-84 → ENU

## Геодезические координаты WGS-84

WGS-84 задаёт эллипсоид вращения с большой полуосью $a=6\,378\,137$ м и сжатием $f=1/298{,}257223563$. Положение точки задаётся геодезическими координатами $(\varphi,\lambda,h)$: широтой $\varphi\in[-90^\circ,90^\circ]$, долготой $\lambda\in[-180^\circ,180^\circ]$ и высотой $h$ над эллипсоидом. В GPS-телеметрии приходят $\mathit{lat}_i\approx\varphi_i$ и $\mathit{lon}_i\approx\lambda_i$; высота в данных отсутствует, поэтому принимается $h_i=0$.

## Переход в ECEF

ECEF (Earth-Centered, Earth-Fixed) — декартова система координат с началом в центре масс Земли. Ось $Z$ направлена вдоль оси вращения, оси $X$ и $Y$ лежат в экваториальной плоскости. Для точки $(\varphi_i,\lambda_i,h_i)$ сначала вычисляется радиус кривизны первого вертикала

$$N(\varphi_i)=\frac{a}{\sqrt{1-e^2\sin^2\varphi_i}},\qquad e^2=2f-f^2,$$

а затем координаты ECEF:

$$X_i=(N(\varphi_i)+h_i)\cos\varphi_i\cos\lambda_i,\qquad
Y_i=(N(\varphi_i)+h_i)\cos\varphi_i\sin\lambda_i,\qquad
Z_i=(N(\varphi_i)(1-e^2)+h_i)\sin\varphi_i.$$

Углы в формулы подставляются в радианах: $\varphi_i=\mathit{lat}_i\pi/180$, $\lambda_i=\mathit{lon}_i\pi/180$.

## Переход в ENU

ENU (East–North–Up) — локальная прямоугольная система с началом в опорной точке $(\varphi_0,\lambda_0,h_0)$. Ось $E$ направлена на восток, $N$ — на север, $U$ — вверх по нормали к эллипсоиду. Разность ECEF $\Delta_i=(X_i-X_0,\,Y_i-Y_0,\,Z_i-Z_0)$ переводится в ENU поворотом

$$
\begin{pmatrix}E_i\\N_i\\U_i\end{pmatrix}
=R(\varphi_0,\lambda_0)\,\Delta_i,\qquad
R=\begin{pmatrix}
-\sin\lambda_0 & \cos\lambda_0 & 0\\
-\sin\varphi_0\cos\lambda_0 & -\sin\varphi_0\sin\lambda_0 & \cos\varphi_0\\
\cos\varphi_0\cos\lambda_0 & \cos\varphi_0\sin\lambda_0 & \sin\varphi_0
\end{pmatrix}.
$$

Матрица $R$ ортогональна: она только поворачивает оси ECEF, не меняя масштаб. Для движения судна по поверхности используется горизонтальная проекция: $x_i=E_i$, $l_i=N_i$; компонента $U_i$ отбрасывается. В блокноте опорная точка — первая валидная точка ряда после фильтрации, поэтому все координаты $x_i,l_i$ отсчитываются от неё в метрах. Плоская интерпретация $(x_i,l_i)$ точна в окрестности опоры; при удалении от неё на десятки километров сказываются кривизна Земли и отличие высот.

In [ ]:
_WGS84_A_M = 6_378_137.0
_WGS84_F = 1.0 / 298.257223563
_WGS84_E2 = 2.0 * _WGS84_F - _WGS84_F**2


def _validate_wgs84_coordinates(
    lat_deg: np.ndarray,
    lon_deg: np.ndarray,
) -> None:
    """Проверяет форму и допустимые диапазоны широты и долготы.

    Args:
        lat_deg: Широты, градусы.
        lon_deg: Долготы, градусы.

    Raises:
        ValueError: Если формы не совпадают или координаты выходят за пределы WGS-84.
    """
    if lat_deg.shape != lon_deg.shape:
        raise ValueError("Массивы широт и долгот должны иметь одинаковую форму")
    if not np.all(np.isfinite(lat_deg)) or not np.all(np.isfinite(lon_deg)):
        raise ValueError("Координаты должны быть конечными числами")
    if np.any((lat_deg < -90.0) | (lat_deg > 90.0)):
        raise ValueError("Широта должна лежать в диапазоне [-90, 90]")
    if np.any((lon_deg < -180.0) | (lon_deg > 180.0)):
        raise ValueError("Долгота должна лежать в диапазоне [-180, 180]")


def wgs84_to_ecef(
    lat_deg: np.ndarray,
    lon_deg: np.ndarray,
    height_m: float | np.ndarray = 0.0,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Переводит геодезические координаты WGS-84 в ECEF.

    Args:
        lat_deg: Широта, градусы.
        lon_deg: Долгота, градусы.
        height_m: Высота над эллипсоидом, м; по умолчанию 0.

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray]: X, Y, Z в метрах.

    Raises:
        ValueError: Если координаты имеют разную форму или выходят за пределы WGS-84.
    """
    lat = np.asarray(lat_deg, dtype=np.float64)
    lon = np.asarray(lon_deg, dtype=np.float64)
    _validate_wgs84_coordinates(lat, lon)
    height = np.broadcast_to(np.asarray(height_m, dtype=np.float64), lat.shape)
    phi = np.deg2rad(lat)
    lam = np.deg2rad(lon)
    sin_phi = np.sin(phi)
    cos_phi = np.cos(phi)
    prime_vertical = _WGS84_A_M / np.sqrt(1.0 - _WGS84_E2 * sin_phi**2)
    radius = prime_vertical + height
    x_ecef = radius * cos_phi * np.cos(lam)
    y_ecef = radius * cos_phi * np.sin(lam)
    z_ecef = (prime_vertical * (1.0 - _WGS84_E2) + height) * sin_phi
    return x_ecef, y_ecef, z_ecef


def wgs84_to_enu(
    lat_deg: np.ndarray,
    lon_deg: np.ndarray,
    origin_lat_deg: float,
    origin_lon_deg: float,
    height_m: float | np.ndarray = 0.0,
    origin_height_m: float = 0.0,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Переводит координаты WGS-84 в локальную ENU относительно опорной точки.

    Args:
        lat_deg: Широты точек, градусы.
        lon_deg: Долготы точек, градусы.
        origin_lat_deg: Широта опорной точки, градусы.
        origin_lon_deg: Долгота опорной точки, градусы.
        height_m: Высота точек над эллипсоидом, м.
        origin_height_m: Высота опорной точки, м.

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray]: E, N, U в метрах.
    """
    origin_lat = float(origin_lat_deg)
    origin_lon = float(origin_lon_deg)
    origin_height = float(origin_height_m)
    x_ecef, y_ecef, z_ecef = wgs84_to_ecef(lat_deg, lon_deg, height_m)
    x0, y0, z0 = wgs84_to_ecef(origin_lat, origin_lon, origin_height)
    dx = x_ecef - x0
    dy = y_ecef - y0
    dz = z_ecef - z0
    phi0 = np.deg2rad(origin_lat)
    lam0 = np.deg2rad(origin_lon)
    sin_phi0 = np.sin(phi0)
    cos_phi0 = np.cos(phi0)
    sin_lam0 = np.sin(lam0)
    cos_lam0 = np.cos(lam0)
    east = -sin_lam0 * dx + cos_lam0 * dy
    north = -sin_phi0 * cos_lam0 * dx - sin_phi0 * sin_lam0 * dy + cos_phi0 * dz
    up = cos_phi0 * cos_lam0 * dx + cos_phi0 * sin_lam0 * dy + sin_phi0 * dz
    return east, north, up


def wgs84_to_local_xy(
    lat_deg: np.ndarray,
    lon_deg: np.ndarray,
    origin_lat_deg: float,
    origin_lon_deg: float,
) -> tuple[np.ndarray, np.ndarray]:
    """Переводит WGS-84 в локальные x (восток) и y (север), метры.

    Args:
        lat_deg: Широты точек, градусы.
        lon_deg: Долготы точек, градусы.
        origin_lat_deg: Широта опорной точки, градусы.
        origin_lon_deg: Долгота опорной точки, градусы.

    Returns:
        tuple[np.ndarray, np.ndarray]: x (восток) и y (север) в метрах.
    """
    east, north, _ = wgs84_to_enu(lat_deg, lon_deg, origin_lat_deg, origin_lon_deg)
    return east, north

## Предварительная обработка

In [ ]:
def nfmt(value: int) -> str:
    """Форматирует целое число с разделителем тысяч (пробел).

    Args:
        value: целое значение для вывода.

    Returns:
        str: строка вида «1 234 567».
    """
    return f"{int(value):,}".replace(",", " ")


def fmt_num(value: float, decimals: int = 2) -> str:
    """Число; при |value| >= 1000 — разделитель тысяч «_».

    Args:
        value: Значение для вывода.
        decimals: Число знаков после запятой (0 — целое).

    Returns:
        str: Отформатированная строка или «—» для нечисловых значений.
    """
    if value is None or not np.isfinite(float(value)):
        return "—"
    number = float(value)
    negative = number < 0
    number = abs(number)
    if number < 1000:
        text = f"{number:.{decimals}f}" if decimals else str(int(round(number)))
        if "." in text:
            text = text.rstrip("0").rstrip(".")
        return ("-" if negative else "") + text
    if decimals == 0:
        body = f"{int(round(number)):,}".replace(",", "_")
    else:
        whole = int(number)
        fraction = number - whole
        fraction_text = f"{fraction:.{decimals}f}".split(".")[1].rstrip("0")
        body = f"{whole:,}".replace(",", "_")
        if fraction_text:
            body = f"{body}.{fraction_text}"
    return ("-" if negative else "") + body


def select_trajectory_columns(raw: pd.DataFrame) -> pd.DataFrame:
    """Оставляет столбцы time, lat, lon и переименовывает status в l.

    Args:
        raw: таблица из data/test.csv.

    Returns:
        pd.DataFrame: столбцы time, lat, lon, y.
    """
    return raw.loc[:, ["time", "lat", "lon", "status"]].rename(columns={"status": "l"}).copy()


def coerce_coordinate_columns(frame: pd.DataFrame) -> pd.DataFrame:
    """Приводит lat и lon к числовому типу (нечисловые значения → NaN).

    Args:
        frame: таблица с столбцами lat, lon.

    Returns:
        pd.DataFrame: копия с числовыми lat, lon.
    """
    result = frame.copy()
    result["lat"] = pd.to_numeric(result["lat"], errors="coerce")
    result["lon"] = pd.to_numeric(result["lon"], errors="coerce")
    return result


def filter_finite_coordinates(frame: pd.DataFrame) -> pd.DataFrame:
    """Удаляет строки с пропусками в lat или lon.

    Args:
        frame: таблица с lat, lon.

    Returns:
        pd.DataFrame: только строки с конечными координатами.
    """
    mask = frame["lat"].notna() & frame["lon"].notna()
    return frame.loc[mask].reset_index(drop=True)


def filter_coordinates_wgs84(frame: pd.DataFrame) -> pd.DataFrame:
    """Удаляет строки с координатами вне допустимых пределов WGS-84.

    Args:
        frame: таблица с lat, lon.

    Returns:
        pd.DataFrame: lat в [-90, 90], lon в [-180, 180].
    """
    mask = frame["lat"].between(-90.0, 90.0) & frame["lon"].between(-180.0, 180.0)
    return frame.loc[mask].reset_index(drop=True)


def filter_nonzero_gps(frame: pd.DataFrame) -> pd.DataFrame:
    """Удаляет точки «нулевого» GPS (lat, lon) = (0, 0).

    Args:
        frame: таблица с lat, lon.

    Returns:
        pd.DataFrame: отфильтрованный ряд.
    """
    mask = ~((frame["lat"] == 0.0) & (frame["lon"] == 0.0))
    return frame.loc[mask].reset_index(drop=True)


def filter_allowed_status(frame: pd.DataFrame, allowed: tuple[str, ...]) -> pd.DataFrame:
    """Оставляет только метки из allowed (move, anomaly).

    Args:
        frame: таблица со столбцом l.
        allowed: допустимые строковые значения status.

    Returns:
        pd.DataFrame: отфильтрованный ряд.
    """
    labels = frame["l"].astype(str)
    return frame.loc[labels.isin(allowed)].reset_index(drop=True)


def sort_by_time(frame: pd.DataFrame) -> pd.DataFrame:
    """Сортирует ряд по возрастанию time (стабильная сортировка).

    Args:
        frame: таблица со столбцом time.

    Returns:
        pd.DataFrame: упорядоченный ряд.
    """
    order = np.argsort(pd.to_datetime(frame["time"]).to_numpy(), kind="mergesort")
    return frame.iloc[order].reset_index(drop=True)


def attach_local_enu(frame: pd.DataFrame) -> pd.DataFrame:
    """Добавляет столбцы x, y — локальная ENU (м) от первой точки ряда.

    Args:
        frame: таблица с lat, lon.

    Returns:
        pd.DataFrame: с добавленными x, y.
    """
    result = frame.copy()
    lat = result["lat"].to_numpy(dtype=float)
    lon = result["lon"].to_numpy(dtype=float)
    east, north = wgs84_to_local_xy(lat, lon, lat[0], lon[0])
    result["x"] = east
    result["y"] = north
    return result


def trim_to_extreme_move(frame: pd.DataFrame) -> pd.DataFrame:
    """Обрезает ряд от первой до последней точки со статусом move.

    Args:
        frame: таблица со столбцом l.

    Returns:
        pd.DataFrame: срез между крайними move.

    Raises:
        RuntimeError: если в ряду нет точек move.
    """
    is_move = frame["l"].astype(str).to_numpy() == "move"
    if not np.any(is_move):
        raise RuntimeError("После фильтрации нет точек move.")
    move_idx = np.flatnonzero(is_move)
    return frame.iloc[move_idx[0] : move_idx[-1] + 1].reset_index(drop=True)


def attach_binary_label(frame: pd.DataFrame) -> pd.DataFrame:
    """Добавляет столбец label для TrajectoryMetrics / TrajectorySeries.

    Положительный класс в метриках (MCC, precision, recall) — anomaly. В этом
    проекте он кодируется нулём (не единицей): anomaly → 0, move → 1. См.
    anomaly_mask (label == 0).

    Args:
        frame: таблица со столбцом l (строки move / anomaly).

    Returns:
        pd.DataFrame: с столбцом label (float).
    """
    result = frame.copy()
    result["label"] = np.where(result["l"].astype(str) == "anomaly", 0.0, 1.0)
    return result


def split_train_test_move(frame: pd.DataFrame, train_fraction: float) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Делит ряд на обучение [0, k) и тест [k, N); граница k — первый move с индексом i >= floor(rho*N).

    Args:
        frame: предобработанный ряд со столбцом l (move / anomaly).
        train_fraction: доля обучения rho из (0, 1), как TEST_SPLIT.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: (train, test).

    Raises:
        ValueError: если train_fraction не в (0, 1).
        RuntimeError: если ряд слишком короткий, нет подходящего move или выборка пуста.
    """
    if not 0.0 < train_fraction < 1.0:
        raise ValueError("train_fraction должен быть в (0, 1).")
    n = len(frame)
    if n < 2:
        raise RuntimeError("Слишком короткий ряд для разбиения.")
    k0 = int(np.floor(train_fraction * n))
    k0 = min(max(k0, 1), n - 1)
    labels = frame["l"].astype(str).to_numpy()
    move_from_k0 = np.flatnonzero((np.arange(n) >= k0) & (labels == "move"))
    if move_from_k0.size == 0:
        raise RuntimeError(f"Нет move для границы при rho={train_fraction}.")
    k = int(move_from_k0[0])
    train = frame.iloc[:k].reset_index(drop=True)
    test = frame.iloc[k:].reset_index(drop=True)
    if len(train) == 0 or len(test) == 0:
        raise RuntimeError(f"Пустая выборка при rho={train_fraction}, k={k}, N={n}.")
    return train, test


def train_block_bounds(pool: pd.DataFrame, start_fraction: float, length_fraction: float) -> tuple[int, int]:
    """Первая и последняя строки обучающего блока пула; обе — штатный ход в эталоне.

    Блок задаётся долями пула; начало сдвигается к первой точке штатного хода не левее, конец — к последней
    не правее.

    Args:
        pool: Пул обучения со столбцом l.
        start_fraction: Начало блока, доля пула.
        length_fraction: Длина блока, доля пула.

    Returns:
        tuple[int, int]: Первая и последняя строки блока.

    Raises:
        RuntimeError: В блоке нет точки штатного хода.
    """
    move = np.flatnonzero(pool["l"].astype(str).to_numpy() == "move")
    start = int(start_fraction * len(pool))
    last = start + int(length_fraction * len(pool)) - 1
    after = np.searchsorted(move, start)
    before = np.searchsorted(move, last, side="right") - 1
    if after >= len(move) or before < 0 or move[after] > move[before]:
        raise RuntimeError("В обучающем блоке нет точки штатного хода.")
    return int(move[after]), int(move[before])


def describe_split(name: str, train: pd.DataFrame, test: pd.DataFrame) -> None:
    """Печатает размер и доли классов anomaly и move для обучения и теста.

    Args:
        name: заголовок разбиения.
        train: обучающая выборка.
        test: тестовая выборка.
    """
    for part_name, part in (("обучение", train), ("тест", test)):
        labels = part["l"].astype(str)
        n = len(part)
        p_anom = float((labels == "anomaly").mean())
        p_move = float((labels == "move").mean())
        print(
            f"{name}: {part_name:<9} {nfmt(n):>10} точек   "
            f"anomaly {p_anom:6.1%}   move {p_move:6.1%}"
        )


raw_full = pd.read_csv(path_data)
frame = select_trajectory_columns(raw_full)
frame = coerce_coordinate_columns(frame)
frame = filter_finite_coordinates(frame)
frame = filter_coordinates_wgs84(frame)
frame = filter_nonzero_gps(frame)
frame = filter_allowed_status(frame, ALLOWED_STATUS)
frame = sort_by_time(frame)
frame = attach_local_enu(frame)
frame = trim_to_extreme_move(frame)
frame = attach_binary_label(frame)
train_pool, test = split_train_test_move(frame, TEST_SPLIT)
block_start, block_last = train_block_bounds(train_pool, TRAIN_BLOCK_START, TRAIN_BLOCK_LENGTH)
train = train_pool.iloc[block_start : block_last + 1].reset_index(drop=True)
dataset = {"train": train, "test": test}
DATASET_NAME = f"блок {TRAIN_BLOCK_LENGTH:.0%} пула → test"

print(f"Исходный CSV: {nfmt(len(raw_full))} строк, после предобработки: {nfmt(len(frame))}.")
print(
    f"Пул обучения: {nfmt(len(train_pool))} точек; обучающий блок — строки {nfmt(block_start)}–{nfmt(block_last)} пула."
)
describe_split(DATASET_NAME, train, test)

## Используемые метрики

### Коэффициент корреляции Мэттьюса (MCC)

Коэффициент корреляции Мэттьюса (Matthews correlation coefficient, MCC) — коэффициент корреляции Пирсона между двумя бинарными векторами: эталонной разметкой и предсказанием (каждая компонента кодируется как 0 или 1). Эквивалентно он выражается через четыре числа таблицы сопряжённости $TP$, $FP$, $FN$, $TN$ и принимает значения от $-1$ до $1$.

Используется в задачах бинарной классификации, в том числе при сильном дисбалансе классов: оценка предсказаний структуры белка, медицинская диагностика, обнаружение аномалий, сравнение порогов и моделей, когда accuracy и доля «большого» класса малоинформативны.

Источник: Chicco D., Jurman G. *The advantages of the Matthews correlation coefficient (MCC) over F1 score and accuracy in binary classification evaluation* // BMC Genomics. 2020. Vol. 21, Art. 6. Open access: [10.1186/s12864-019-6413-7](https://link.springer.com/article/10.1186/s12864-019-6413-7). (В статье отмечено, что MCC изначально предложен B. W. Matthews в 1975 г.; в статье приведены математические свойства и сравнение с accuracy и $F_1$.)

Основное преимущество: метрика одновременно учитывает все четыре ячейки таблицы сопряжённости и симметрично отражает цену ложных срабатываний и пропусков; при дисбалансе классов тривиальные предсказания «всегда отрицательный» или «всегда положительный» класс обычно дают MCC около нуля, тогда как accuracy может оставаться высокой.

Для положительного класса (в этом блокноте положительный класс — **anomaly**, $l_i=0$, бинарная метка $a_i=\mathbf{1}[l_i=0]$; предсказание $\hat a_i=\mathbf{1}[\hat l_i=0]$):

$$
\mathrm{MCC}=\frac{TP\cdot TN - FP\cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}},
$$
где:\
$TP=\sum_i \mathbf{1}[a_i=1\land \hat a_i=1]$,\
$FP=\sum_i \mathbf{1}[a_i=0\land \hat a_i=1]$,\
$FN=\sum_i \mathbf{1}[a_i=1\land \hat a_i=0]$,\
$TN=\sum_i \mathbf{1}[a_i=0\land \hat a_i=0]$.

Интерпретация: $\mathrm{MCC}\in[-1,1]$; $1$ — полное совпадение масок; $0$ — согласие на уровне случайного предсказания при тех же маргинальных долях классов; отрицательные значения — систематическое расхождение хуже случайного. Если знаменатель равен нулю (например, в выборке или в предсказании только один класс), MCC в строгом смысле не определён; в коде этого блокнота в таком случае принимается $0$.

In [ ]:
def confusion_counts(
    predicted_anomaly: np.ndarray,
    ground_truth_anomaly: np.ndarray,
) -> tuple[float, float, float, float]:
    """Считает TP, FP, FN, TN для масок anomaly.

    Положительный класс — anomaly (в исходной разметке label == 0).

    Args:
        predicted_anomaly: Булева маска предсказанных anomaly.
        ground_truth_anomaly: Булева маска эталонных anomaly.

    Returns:
        tuple[float, float, float, float]: (TP, FP, FN, TN).

    Raises:
        ValueError: Если длины масок не совпадают.
    """
    predicted = np.asarray(predicted_anomaly, dtype=bool).ravel()
    truth = np.asarray(ground_truth_anomaly, dtype=bool).ravel()
    if predicted.shape != truth.shape:
        raise ValueError("Маски предсказания и эталона должны иметь одинаковую длину.")
    return (
        float(np.count_nonzero(predicted & truth)),
        float(np.count_nonzero(predicted & ~truth)),
        float(np.count_nonzero(~predicted & truth)),
        float(np.count_nonzero(~predicted & ~truth)),
    )


def mcc_from_counts(
    true_positive: float,
    false_positive: float,
    false_negative: float,
    true_negative: float,
) -> float:
    """Считает MCC по элементам таблицы сопряжённости.

    Args:
        true_positive: TP — anomaly верно предсказана.
        false_positive: FP — anomaly предсказана на точке move.
        false_negative: FN — anomaly пропущена.
        true_negative: TN — move верно оставлен move.

    Returns:
        float: MCC в [-1, 1]; 0.0, если знаменатель равен нулю.
    """
    denominator = np.sqrt(
        (true_positive + false_positive)
        * (true_positive + false_negative)
        * (true_negative + false_positive)
        * (true_negative + false_negative)
    )
    if denominator <= 0.0:
        return 0.0
    numerator = true_positive * true_negative - false_positive * false_negative
    return float(numerator / denominator)


def mcc_score(
    predicted_anomaly: np.ndarray,
    ground_truth_anomaly: np.ndarray,
) -> float:
    """Считает MCC по двум маскам anomaly.

    Args:
        predicted_anomaly: Булева маска предсказанных anomaly.
        ground_truth_anomaly: Булева маска эталонных anomaly.

    Returns:
        float: MCC в [-1, 1]; 0.0 при вырожденной таблице сопряжённости.
    """
    true_positive, false_positive, false_negative, true_negative = confusion_counts(
        predicted_anomaly, ground_truth_anomaly
    )
    return mcc_from_counts(true_positive, false_positive, false_negative, true_negative)

### Distance loss

Геопространственная метрика: насколько расходятся длины эталонной и предсказанной траектории (в метрах). Длина ломаной — сумма геодезических звеньев $d_{\mathrm{geo}}$; distance loss — модуль разности:

$$
L(T)=\sum_i d_{\mathrm{geo}}(p_i,p_{i+1}), \qquad DL=\bigl|L_{\mathrm{pred}}-L_{\mathrm{ref}}\bigr|.
$$

$L_{\mathrm{ref}}$ строится по $G$ (штатные точки и линейная интерполяция в аномалиях между соседними штатными). $L_{\mathrm{pred}}$ — по $P$: в ломаную входят только точки, принятые за штатный ход ($\hat l_i\neq 0$); пропуски не интерполируются, соседние принятые точки соединяются напрямую. На тесте $DL$ также считается через на заполненных траекториях; при выборе порога по MCC в полосе близкого значения MCC минимизируется операционный distance loss (длина по принятым move-точкам).

### Выбор порога

На обучающей выборке: сетка кандидатов $\tau$ — квантили оценки аномальности и 60 значений, равномерных в логарифмической шкале между её 1-м перцентилем и максимумом; выбирается максимум MCC по точкам с оценкой, среди кандидатов в полосе MCC (до 0.02 от лучшего) — наименьшая операционная ошибка длины. На тестовой выборке — тот же порог без подбора.

К сетке добавляется канонический кандидат, если у оценки аномальности есть теоретическая граница: $\chi^2_{0.999}(2)$ у моделей с квадратом нормированной инновации (фильтр Калмана, гауссовский процесс, GP-EKF), 3 у фильтра Хэмпела, 0.5 у Isolation Forest и HMM, 0 у One-Class SVM. Собственная сетка вместо порога — у DBSCAN (радиус $\varepsilon$) и MPCT (предел скорости $v_+$); у LOF порог подбирается вместе с числом соседей $k$.

Отвергнутая точка не используется в оценках следующих точек. У моделей прогноза (AR, ARIMA, фильтр Калмана, гауссовский процесс, GP-EKF) она не входит в прогноз; у моделей по признакам (фильтр Хэмпела, DBSCAN, LOF, Isolation Forest, One-Class SVM, Robust PCA, LSTM-автоэнкодер, TCN, DONUT, HMM) решение принимается в два прохода: признаки пересчитываются от последних принятых точек, оценки и решения второго прохода — итоговые. Отвергнутые точки зависят от порога, поэтому замена выполняется заново для каждого кандидата. MPCT замены не требует: скорость считается только между выбранными точками.

### Точки без оценки и покрытие

Точка без оценки — точка, в которой модель не может дать оценку аномальности; она не относится ни к штатному ходу, ни к аномалии. Точность, полнота, F1 и MCC считаются только по точкам с оценкой, у каждой модели — на своём подмножестве. В заполненной траектории точка без оценки восстанавливается интерполяцией по времени, как отвергнутая. Задержка обнаружения отсчитывается от начала эпизода аномалии. Покрытие в итоговых таблицах — доля точек без оценки, доля аномальных точек среди них и доля аномальных точек без оценки.

In [ ]:
def trajectory_series_pair(
    frame: pd.DataFrame, predicted_anomaly: np.ndarray, scored: Optional[np.ndarray] = None
) -> tuple[TrajectorySeries, TrajectorySeries]:
    """Собирает prediction и ground truth для TrajectoryMetrics.

    Args:
        frame: Строки test с полями time, lat, lon, y.
        predicted_anomaly: True, если модель объявила anomaly.
        scored: True в точках с оценкой аномальности; None — оценка есть во всех точках.

    Returns:
        tuple[TrajectorySeries, TrajectorySeries]: (prediction, ground_truth); в prediction точка без оценки — NaN.
    """
    ground_anomaly = frame["l"].astype(str).to_numpy() == "anomaly"
    truth = np.where(ground_anomaly, 0.0, 1.0)
    predicted = np.where(predicted_anomaly, 0.0, 1.0)
    if scored is not None:
        predicted[~np.asarray(scored, dtype=bool)] = np.nan
    time = frame["time"].to_numpy()
    lon = frame["lon"].to_numpy(dtype=float)
    lat = frame["lat"].to_numpy(dtype=float)
    ground_truth = TrajectorySeries(time, lon, lat, truth)
    prediction = TrajectorySeries(time, lon, lat, predicted)
    return prediction, ground_truth


def slice_trajectory_series(series: TrajectorySeries, tail: int) -> TrajectorySeries:
    """Хвост TrajectorySeries для карты (последние tail точек).

    Args:
        series: Исходный ряд.
        tail: Число точек с конца.

    Returns:
        TrajectorySeries: Срез хвоста.
    """
    count = min(int(tail), len(series.time))
    return TrajectorySeries(
        series.time[-count:],
        series.lon[-count:],
        series.lat[-count:],
        series.label[-count:],
    )


def evaluate_test(
    frame: pd.DataFrame, predicted_anomaly: np.ndarray, scored: Optional[np.ndarray] = None
) -> dict[str, float]:
    """Считает на тесте метрики из Отчёта (TrajectoryMetrics.compute_all).

    Args:
        frame: Тестовый DataFrame.
        predicted_anomaly: Маска предсказанных anomaly.
        scored: True в точках с оценкой аномальности; None — оценка есть во всех точках.

    Returns:
        dict[str, float]: Гео-, временные, классификационные метрики и покрытие test.
    """
    prediction, ground_truth = trajectory_series_pair(frame, predicted_anomaly, scored)
    return TrajectoryMetrics.compute_all(prediction, ground_truth)


def time_seconds_from_frame(frame: pd.DataFrame) -> np.ndarray:
    """Время ряда в секундах от эпохи (float64).

    Args:
        frame: DataFrame с колонкой time, приводимой к datetime.

    Returns:
        np.ndarray: Секунды от эпохи Unix, shape (N,).
    """
    stamp = pd.to_datetime(frame["time"])
    return stamp.to_numpy(dtype="datetime64[ns]").astype(np.int64) / 1e9


def enu_xy_from_frame(frame: pd.DataFrame) -> np.ndarray:
    """Координаты ENU (восток, север), м, форма (N, 2).

    Args:
        frame: DataFrame с колонками x (восток) и y (север), м.

    Returns:
        np.ndarray: Матрица (N, 2) координат ENU, м.
    """
    return np.column_stack([frame["x"].to_numpy(dtype=float), frame["y"].to_numpy(dtype=float)])


def wrap_angle(radians: np.ndarray) -> np.ndarray:
    """Перенос угла в интервал (-pi, pi].

    Args:
        radians: Углы, рад.

    Returns:
        np.ndarray: Те же углы в интервале (-pi, pi].
    """
    return (np.asarray(radians, dtype=float) + np.pi) % (2.0 * np.pi) - np.pi


def trajectory_kinematics(frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Скорость, ускорение, поворот курса и Δt; у i=0 нули.

    Args:
        frame: DataFrame с колонками x, y и time.

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]: speed (м/с), acc (м/с²),
        dpsi (рад) и step_dt (с).
    """
    east = frame["x"].to_numpy(dtype=float)
    north = frame["y"].to_numpy(dtype=float)
    time_sec = time_seconds_from_frame(frame)
    count = len(east)
    step_dt = np.ones(count, dtype=float)
    speed = np.zeros(count, dtype=float)
    acc = np.zeros(count, dtype=float)
    dpsi = np.zeros(count, dtype=float)
    if count < 2:
        return speed, acc, dpsi, step_dt
    step_dt[1:] = np.diff(time_sec)
    dist = np.hypot(np.diff(east), np.diff(north))
    speed[1:] = dist / np.clip(step_dt[1:], 1.0, None)
    acc[1:] = np.diff(speed) / np.clip(step_dt[1:], 1.0, None)
    heading = np.zeros(count, dtype=float)
    dx_step = east[1:] - east[:-1]
    dy_step = north[1:] - north[:-1]
    heading[1:] = np.arctan2(dx_step, dy_step)
    if count > 2:
        dpsi[2:] = wrap_angle(np.diff(heading[1:]))
    return speed, acc, dpsi, step_dt


def robust_params(matrix: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Робастная шкала: медиана и межквартильный размах по столбцам.

    Args:
        matrix: Матрица признаков shape (n, d).

    Returns:
        tuple[np.ndarray, np.ndarray]: center (медианы) и scale (IQR); scale < 1e-6 → 1.
    """
    center = np.median(matrix, axis=0)
    scale = np.subtract(*np.percentile(matrix, [75, 25], axis=0))
    scale = np.where(scale < 1e-6, 1.0, scale)
    return center, scale


def apply_robust(matrix: np.ndarray, center: np.ndarray, scale: np.ndarray) -> np.ndarray:
    """Центрирование и деление на scale.

    Args:
        matrix: Матрица признаков shape (n, d).
        center: Центры столбцов из robust_params.
        scale: Масштабы столбцов из robust_params.

    Returns:
        np.ndarray: (matrix - center) / scale.
    """
    return (matrix - center) / scale


def frame_window_arrays(frame: pd.DataFrame) -> dict[str, np.ndarray]:
    """Поля для оконных детекторов: ENU, шаги, кинематика, move.

    Args:
        frame: DataFrame с колонками x, y, time и l.

    Returns:
        dict[str, np.ndarray]: count, xy, dx, dy, speed, dpsi, dt, dist, move.
    """
    east = frame["x"].to_numpy(dtype=float)
    north = frame["y"].to_numpy(dtype=float)
    speed, _, dpsi, step_dt = trajectory_kinematics(frame)
    count = len(east)
    step_dx = np.zeros(count, dtype=float)
    step_dy = np.zeros(count, dtype=float)
    step_dist = np.zeros(count, dtype=float)
    if count > 1:
        step_dx[1:] = np.diff(east)
        step_dy[1:] = np.diff(north)
        step_dist[1:] = np.hypot(step_dx[1:], step_dy[1:])
    return {
        "count": count,
        "xy": np.column_stack([east, north]),
        "dx": step_dx,
        "dy": step_dy,
        "speed": speed,
        "dpsi": dpsi,
        "dt": step_dt,
        "dist": step_dist,
        "move": frame["l"].astype(str).to_numpy() == "move",
    }


def window_starts(
    arrays: dict[str, np.ndarray],
    length: int,
    max_dt: float,
    max_dist: float,
    require_move: bool,
) -> np.ndarray:
    """Индексы окон без большого разрыва по времени или расстоянию.

    Args:
        arrays: Кэш ряда из frame_window_arrays.
        length: Длина окна в точках.
        max_dt: Максимальный Δt между соседними точками, с.
        max_dist: Максимальный шаг между соседними точками, м.
        require_move: Требовать move во всех точках окна.

    Returns:
        np.ndarray: Индексы допустимых стартов; пустой массив, если окон нет.
    """
    count = arrays["count"]
    if count < length:
        return np.empty(0, dtype=int)
    good = np.ones(count, dtype=bool)
    if require_move:
        good &= arrays["move"]
    step_ok = np.ones(count, dtype=bool)
    step_ok[1:] = (arrays["dt"][1:] <= max_dt) & (arrays["dist"][1:] <= max_dist)
    good_cum = np.cumsum(good.astype(np.int32))
    step_cum = np.cumsum(step_ok.astype(np.int32))
    starts = np.arange(count - length + 1)
    good_sum = good_cum[starts + length - 1] - np.where(starts > 0, good_cum[starts - 1], 0)
    step_sum = step_cum[starts + length - 1] - step_cum[starts]
    return starts[(good_sum == length) & (step_sum == length - 1)]


def take_window_starts(starts: np.ndarray, limit: int, seed: int) -> np.ndarray:
    """Случайная подвыборка стартов окон без повторов.

    Args:
        starts: Допустимые индексы начала окна.
        limit: Максимальное число стартов.
        seed: Seed генератора.

    Returns:
        np.ndarray: Отсортированные выбранные старты.
    """
    if len(starts) <= limit:
        return starts
    picked = np.random.default_rng(seed).choice(len(starts), size=limit, replace=False)
    return np.sort(starts[picked])


def batched_model_forward(model: nn.Module, array: np.ndarray, forward_fn, batch_size: int) -> np.ndarray:
    """Пакетный forward PyTorch-модели без градиентов.

    Args:
        model: Модель на DEVICE.
        array: Входной numpy-массив.
        forward_fn: Callable (model, batch_tensor) -> tensor.
        batch_size: Размер пакета.

    Returns:
        np.ndarray: Склеенные выходы по оси 0.
    """
    model.eval()
    pieces: list[np.ndarray] = []
    with torch.no_grad():
        for start in range(0, len(array), batch_size):
            batch = torch.from_numpy(np.ascontiguousarray(array[start : start + batch_size])).to(DEVICE)
            pieces.append(forward_fn(model, batch).detach().cpu().numpy())
    return np.concatenate(pieces, axis=0)


def operational_path_length_m(east_m: np.ndarray, north_m: np.ndarray, move_mask: np.ndarray) -> float:
    """Длина ломаной в ENU только по точкам move (без интерполяции).

    Args:
        east_m: Восток, м.
        north_m: Север, м.
        move_mask: True для штатного хода.

    Returns:
        float: Длина в метрах.
    """
    move = np.asarray(move_mask, dtype=bool)
    east = np.asarray(east_m, dtype=np.float64)[move]
    north = np.asarray(north_m, dtype=np.float64)[move]
    if east.size < 2:
        return 0.0
    return float(np.sum(np.hypot(np.diff(east), np.diff(north))))


def operational_distance_loss_m(
    east_m: np.ndarray,
    north_m: np.ndarray,
    ground_truth_anomaly: np.ndarray,
    predicted_anomaly: np.ndarray,
) -> float:
    """Операционный |L_pred - L_ref| по move-точкам в ENU.

    Args:
        east_m: Восток, м.
        north_m: Север, м.
        ground_truth_anomaly: True для anomaly в разметке.
        predicted_anomaly: True для предсказанной anomaly.

    Returns:
        float: Distance loss в метрах.
    """
    gt_anomaly = np.asarray(ground_truth_anomaly, dtype=bool)
    pred_anomaly = np.asarray(predicted_anomaly, dtype=bool)
    reference_m = operational_path_length_m(east_m, north_m, ~gt_anomaly)
    predicted_m = operational_path_length_m(east_m, north_m, ~pred_anomaly)
    return float(abs(predicted_m - reference_m))


def _threshold_record(
    tau: float,
    predicted_anomaly: np.ndarray,
    frame: pd.DataFrame,
    ground_truth_anomaly: np.ndarray,
    scored: Optional[np.ndarray] = None,
) -> dict[str, float]:
    """MCC, F1 и операционный DL для одного порога score.

    Точки без оценки не входят в MCC, F1, precision и recall; из ломаной операционного DL исключаются,
    как отвергнутые.

    Args:
        tau: Кандидат порога (для записи в результат).
        predicted_anomaly: Маска предсказанных anomaly.
        frame: Ряд с time, lat, lon, x, y, l.
        ground_truth_anomaly: Эталонная маска anomaly.
        scored: True в точках с оценкой аномальности; None — оценка есть во всех точках.

    Returns:
        dict[str, float]: tau, mcc, f1, precision, recall, loss_m.
    """
    east_m = frame["x"].to_numpy(dtype=float)
    north_m = frame["y"].to_numpy(dtype=float)
    predicted = np.asarray(predicted_anomaly, dtype=bool)
    truth = np.asarray(ground_truth_anomaly, dtype=bool)
    scored_mask = np.ones(len(predicted), dtype=bool) if scored is None else np.asarray(scored, dtype=bool)
    true_positive, false_positive, false_negative, true_negative = confusion_counts(
        predicted[scored_mask], truth[scored_mask]
    )
    precision_denominator = true_positive + false_positive
    recall_denominator = true_positive + false_negative
    f1_denominator = 2.0 * true_positive + false_positive + false_negative
    precision = true_positive / precision_denominator if precision_denominator > 0.0 else float("nan")
    recall = true_positive / recall_denominator if recall_denominator > 0.0 else float("nan")
    f1 = 2.0 * true_positive / f1_denominator if f1_denominator > 0.0 else float("nan")
    mcc = mcc_from_counts(true_positive, false_positive, false_negative, true_negative)
    loss_m = operational_distance_loss_m(east_m, north_m, truth, predicted | ~scored_mask)
    return {
        "tau": float(tau),
        "mcc": float(mcc),
        "f1": float(f1),
        "precision": float(precision),
        "recall": float(recall),
        "loss_m": float(loss_m),
    }


def _pick_mcc_band_record(band: list[dict[str, float]], tie_key: str) -> dict[str, float]:
    """Среди записей в полосе MCC — минимальный операционный DL.

    Args:
        band: Записи-кандидаты в полосе MCC.
        tie_key: Имя ключа для детерминированного разрешения ничьей.

    Returns:
        dict[str, float]: Запись с минимальным loss_m, затем с максимальными MCC и F1.
    """
    return sorted(band, key=lambda r: (r["loss_m"], -r["mcc"], -r["f1"], r[tie_key]))[0]


def select_threshold_from_records(records: list[dict[str, float]], mcc_band: float) -> dict[str, float]:
    """Выбирает порог по записям кандидатов: max MCC, в полосе mcc_band — min операционный DL.

    Args:
        records: Записи _threshold_record по кандидатам порога.
        mcc_band: Допуск по MCC относительно лучшего значения.

    Returns:
        dict[str, float]: tau, train_mcc, train_f1, train_dl_m, best_mcc, band_size.

    Raises:
        RuntimeError: Нет допустимых порогов.
    """
    usable = [r for r in records if np.isfinite(r["loss_m"]) and np.isfinite(r["mcc"])]
    if not usable:
        raise RuntimeError("Нет конечного MCC и distance loss для выбора порога.")
    best_mcc = max(r["mcc"] for r in usable)
    band = [r for r in usable if r["mcc"] >= best_mcc - mcc_band - 1e-12]
    chosen = _pick_mcc_band_record(band, "tau")
    return {
        "tau": chosen["tau"],
        "train_mcc": chosen["mcc"],
        "train_f1": chosen["f1"],
        "train_dl_m": chosen["loss_m"],
        "best_mcc": float(best_mcc),
        "band_size": float(len(band)),
    }


def threshold_candidates(scores: np.ndarray, n_quantiles: int = 180, n_log: int = THRESHOLD_LOG_POINTS) -> np.ndarray:
    """Сетка кандидатов порога: квантили score и точки, равномерные в логарифмической шкале.

    Квантили сгущаются там, где много точек; логарифмические точки закрывают провалы распределения
    (score штатного хода и аномалий различаются на порядки). Если score не положителен, вместо
    логарифмической шкалы — равномерная.

    Args:
        scores: Конечные значения score точек с оценкой.
        n_quantiles: Число квантилей на уровнях от 0.01 до 0.995.
        n_log: Число точек между 1-м перцентилем и максимумом score.

    Returns:
        np.ndarray: Отсортированные уникальные кандидаты.
    """
    values = np.asarray(scores, dtype=float)
    quantiles = np.quantile(values, np.linspace(0.01, 0.995, n_quantiles))
    low, high = float(np.quantile(values, 0.01)), float(np.max(values))
    if high <= low:
        return np.unique(quantiles)
    spread = np.geomspace(low, high, n_log) if low > 0.0 else np.linspace(low, high, n_log)
    return np.unique(np.concatenate([quantiles, spread]))


# Порядок и подписи.
TEST_REPORT_METRICS: tuple[tuple[str, str], ...] = (
    ("rmse_m", "RMSE, м"),
    ("max_deviation_m", "Max(d_i), м"),
    ("distance_loss_m", "Distance loss, м"),
    ("delay_median", "Delay median, с"),
    ("delay_mean", "Delay mean, с"),
    ("delay_p95", "Delay p95, с"),
    ("delay_max", "Delay max, с"),
    ("delay_n_on_time", "On-time episodes"),
    ("delay_n_late_or_missed", "Late/missed episodes"),
    ("precision", "Precision"),
    ("recall", "Recall"),
    ("f1", "F1"),
    ("mcc", "MCC"),
    ("unscored_share", "Без оценки, %"),
    ("unscored_anomaly_share", "Из них аномалий, %"),
    ("anomaly_unscored_share", "Аномалий без оценки, %"),
)


def test_report_row(scenario_name: str, metrics: dict[str, float]) -> dict[str, object]:
    """Оставляет только метрики test в фиксированном порядке.

    Args:
        scenario_name: Имя разбиения train / test.
        metrics: Результат evaluate_test / TrajectoryMetrics.compute_all.

    Returns:
        dict[str, object]: Строка сводной таблицы.
    """
    row: dict[str, object] = {"Сценарий": scenario_name}
    for key, _ in TEST_REPORT_METRICS:
        row[key] = metrics.get(key, float("nan"))
    return row


def _format_metric_value(key: str, value) -> str:
    """Форматирует одну метрику test для таблицы отчёта.

    Args:
        key: Имя метрики из TEST_REPORT_METRICS.
        value: Числовое значение или None.

    Returns:
        str: Строка для display (или «—» при пропуске).
    """
    if value is None or (isinstance(value, float) and not np.isfinite(value)):
        return "—"
    if key in {"precision", "recall", "f1", "mcc"}:
        rate = float(value)
        return fmt_num(rate, 4) if abs(rate) >= 1000 else f"{rate:.4f}"
    if key.endswith("_share"):
        return f"{100.0 * float(value):.2f}"
    if key in {"distance_loss_m", "rmse_m", "max_deviation_m"}:
        return fmt_num(float(value), 2)
    if key.startswith("delay_n_"):
        return fmt_num(int(value), 0)
    if key.startswith("delay_"):
        delay = float(value)
        return fmt_num(delay, 2) if abs(delay) >= 1000 else f"{delay:.2f}"
    return str(value)


def display_test_report_table(frame: pd.DataFrame, caption: str = "") -> None:
    """Стилизованная таблица итоговых метрик test (как в Отчёте).

    Args:
        frame: Столбец «Сценарий» и ключи из TEST_REPORT_METRICS.
        caption: Подпись таблицы.
    """
    labels = {key: label for key, label in TEST_REPORT_METRICS}
    renamed = frame.rename(columns=labels)
    for key, label in TEST_REPORT_METRICS:
        if label in renamed.columns:
            renamed[label] = renamed[label].map(lambda v, k=key: _format_metric_value(k, v))
    styled = renamed.style
    if caption:
        styled = styled.set_caption(caption)
    display(styled)


# Модели прогноза (AR, ARIMA): регулярная сетка по времени, оценка по штатному ходу, замена отвергнутых точек.
# Дисперсия диффузного начального состояния, м²: много больше квадрата координат ENU.
_DIFFUSE_VAR_M2 = 1e14


def grid_nodes(time_sec: np.ndarray, step_sec: float) -> tuple[np.ndarray, np.ndarray]:
    """Узлы регулярной сетки по времени и маска точек, оставшихся в своём узле.

    Args:
        time_sec: Время точек по возрастанию, с.
        step_sec: Шаг сетки, с.

    Returns:
        tuple[np.ndarray, np.ndarray]: Номер узла каждой точки и маска первой точки узла
        (остальные точки узла — без оценки).
    """
    node = np.rint((time_sec - time_sec[0]) / step_sec).astype(np.int64)
    first = np.ones(len(node), dtype=bool)
    first[1:] = node[1:] != node[:-1]
    return node, first


def _forecast_model(series: np.ndarray, order: tuple[int, int, int], trend: str) -> ARIMA:
    """Модель statsmodels с ограничениями стационарности и обратимости."""
    return ARIMA(series, order=order, trend=trend, enforce_stationarity=True, enforce_invertibility=True)


# Штраф целевой функции за параметры на границе стационарности или обратимости.
_INADMISSIBLE_PENALTY = 1e300


def _forecast_params_admissible(model: ARIMA, params: np.ndarray) -> bool:
    """Корни AR- и MA-многочленов строго внутри единичного круга: стационарность и обратимость."""
    for prefix, sign in (("ar.", -1.0), ("ma.", 1.0)):
        coefs = np.array([value for name, value in zip(model.param_names, params) if name.startswith(prefix)])
        if coefs.size and np.any(np.abs(np.roots(np.r_[1.0, sign * coefs])) >= 1.0 - 1e-9):
            return False
    return True


def _start_candidates(model: ARIMA, order: tuple[int, int, int]) -> list[np.ndarray]:
    """Начальные параметры оптимизации: из statsmodels и с AR-коэффициентами по МНК.

    Вблизи единичного корня оптимизация из одного старта попадает в локальный максимум; оба старта сжаты
    внутрь области стационарности и обратимости.

    Args:
        model: Модель statsmodels самого длинного отрезка.
        order: Порядок (p, d, q).

    Returns:
        list[np.ndarray]: Допустимые наборы параметров.
    """
    base = np.asarray(model.start_params, dtype=float)
    candidates = [base.copy()]
    target = np.diff(model.endog.ravel()[~np.isnan(model.endog.ravel())], n=order[1])
    p_order = order[0]
    if p_order and len(target) > 2 * p_order:
        lags = np.column_stack([target[p_order - lag : len(target) - lag] for lag in range(1, p_order + 1)])
        ols = np.linalg.lstsq(lags, target[p_order:], rcond=None)[0]
        alternative = base.copy()
        alternative[[name.startswith("ar.") for name in model.param_names]] = ols
        candidates.append(alternative)
    lagged = np.array([name.startswith(("ar.", "ma.")) for name in model.param_names])
    for start in candidates:
        while not _forecast_params_admissible(model, start):
            start[lagged] *= 0.9
    return candidates


def fit_forecast_axis(
    node: np.ndarray, values: np.ndarray, starts: np.ndarray, order: tuple[int, int, int], trend: str
) -> np.ndarray:
    """Параметры ARIMA одной оси: максимум суммы логарифмов правдоподобия отрезков с общими параметрами.

    Отрезки — обучающие отрезки штатного хода (move_segments), разрезанные на каждом шаге времени, отличном от шага
    сетки: пустых узлов внутри отрезка нет.

    Args:
        node: Узлы наблюдений по возрастанию (точки штатного хода).
        values: Координата наблюдений, м.
        starts: Маска первой точки отрезка.
        order: Порядок (p, d, q).
        trend: Член тренда statsmodels.

    Returns:
        np.ndarray: Параметры в порядке param_names statsmodels.

    Raises:
        RuntimeError: Нет отрезка достаточной длины.
    """
    min_points = order[0] + order[1] + 2
    cuts = np.flatnonzero(starts)[1:]
    models: list[ARIMA] = []
    for index in np.split(np.arange(len(node)), cuts):
        if len(index) < min_points:
            continue
        series = np.full(node[index[-1]] - node[index[0]] + 1, np.nan)
        # Уровень отрезка — его первая точка: координата отсчитывается от неё.
        series[node[index] - node[index[0]]] = values[index] - values[index[0]]
        model = _forecast_model(series, order, trend)
        # Условное правдоподобие: диффузное начальное состояние, первые p + d наблюдений отрезка не входят —
        # как точки без оценки в детекторе.
        model.ssm.initialize_approximate_diffuse(_DIFFUSE_VAR_M2)
        model.ssm.loglikelihood_burn = order[0] + order[1]
        models.append(model)
    if not models:
        raise RuntimeError("Нет отрезка штатного хода для оценки параметров.")
    longest = max(models, key=lambda model: model.nobs)

    def negative_loglike(unconstrained: np.ndarray) -> float:
        """Минус сумма логарифмов правдоподобия отрезков; вне допустимой области — штраф."""
        params = longest.transform_params(unconstrained)
        if not _forecast_params_admissible(longest, params):
            return _INADMISSIBLE_PENALTY
        total = sum(model.loglike(params) for model in models)
        return -total if np.isfinite(total) else _INADMISSIBLE_PENALTY

    results = [
        minimize(negative_loglike, longest.untransform_params(start), method="L-BFGS-B")
        for start in _start_candidates(longest, order)
    ]
    return longest.transform_params(min(results, key=lambda result: result.fun).x)


def forecast_matrices(order: tuple[int, int, int], trend: str, params: np.ndarray) -> dict[str, np.ndarray]:
    """Матрицы формы пространства состояний statsmodels для заданных параметров.

    Args:
        order: Порядок (p, d, q).
        trend: Член тренда statsmodels.
        params: Параметры модели.

    Returns:
        dict[str, np.ndarray]: design (k,), transition (k, k), noise (k, k) и кэш горизонтов.
    """
    model = _forecast_model(np.zeros(16), order, trend)
    model.update(params)
    k_states = model.k_states
    selection = np.asarray(model.ssm["selection"], dtype=float).reshape(k_states, -1)
    state_cov = np.asarray(model.ssm["state_cov"], dtype=float).reshape(selection.shape[1], -1)
    return {
        "design": np.asarray(model.ssm["design"], dtype=float).reshape(k_states),
        "transition": np.asarray(model.ssm["transition"], dtype=float).reshape(k_states, k_states),
        "noise": selection @ state_cov @ selection.T,
        "horizons": {},
    }


def _horizon_matrices(axis: dict[str, np.ndarray], horizon: int) -> tuple[np.ndarray, np.ndarray]:
    """Переход на horizon шагов и накопленный шум: T^h и сумма T^m Q T^m' по m < h (с кэшем)."""
    cache = axis["horizons"]
    if horizon not in cache:
        power = np.eye(len(axis["design"]))
        noise = np.zeros_like(power)
        for _ in range(horizon):
            noise = axis["transition"] @ noise @ axis["transition"].T + axis["noise"]
            power = axis["transition"] @ power
        cache[horizon] = (power, noise)
    return cache[horizon]


def fit_forecast_detector(train: pd.DataFrame, order: tuple[int, int, int], trend: str) -> dict[str, object]:
    """Оценивает модель прогноза по точкам штатного хода обучающей выборки, по двум осям ENU.

    Args:
        train: Обучающий DataFrame с колонками x, y, time и l.
        order: Порядок (p, d, q).
        trend: Член тренда statsmodels.

    Returns:
        dict[str, object]: params и axes (по осям x, y), n_unscored — число точек без оценки в начале отрезка,
        speed_limit — предел скорости перезапуска, м/с.
    """
    xy = enu_xy_from_frame(train)
    node, first = grid_nodes(time_seconds_from_frame(train), GRID_STEP_SEC)
    index, starts = segments_layout(move_segments(train))
    # Из точек одного узла остаётся первая; отрезок, потерявший первую точку, начинается со следующей.
    keep = first[index]
    segment = np.cumsum(starts)[keep]
    starts = np.r_[True, segment[1:] != segment[:-1]]
    index = index[keep]
    # Участок рвётся на любом шаге, отличном от шага сетки: по участкам «точка — пропуск — точка» правдоподобие
    # выбирало прогноз через узел, phi ≈ (0, 1), вместо прогноза по предыдущей точке.
    starts = starts | np.r_[True, np.diff(node[index]) != 1]
    params = [fit_forecast_axis(node[index], xy[index, column], starts, order, trend) for column in range(2)]
    return {
        "params": params,
        "axes": [forecast_matrices(order, trend, value) for value in params],
        "n_unscored": order[0] + order[1],
        "speed_limit": restart_speed_limit(train),
    }


# Канонический кандидат порога NIS моделей прогноза.
FORECAST_CHI999 = float(chi2.ppf(0.999, 2))


def forecast_detect(
    frame: pd.DataFrame, detector: dict[str, object], taus: np.ndarray
) -> tuple[np.ndarray, np.ndarray]:
    """Причинный прогноз на сетке с заменой отвергнутых точек, сразу для нескольких порогов.

    Оценка — NIS: сумма по осям квадрата ошибки прогноза, делённого на дисперсию прогноза. Отвергнутая точка
    (оценка больше порога) не обновляет состояние. Прогноз начинается заново после разрыва по времени и после
    серии отвергнутых точек длиннее разрыва, если точка достижима от последней принятой (restart_allowed);
    первые n_unscored точек отрезка — без оценки.

    Args:
        frame: DataFrame с колонками x, y и time.
        detector: Результат fit_forecast_detector.
        taus: Пороги; np.inf — без замены.

    Returns:
        tuple[np.ndarray, np.ndarray]: NIS shape (C, N), NaN — точка без оценки;
        маска отвергнутых точек shape (C, N).
    """
    xy = enu_xy_from_frame(frame)
    time_sec = time_seconds_from_frame(frame)
    node, first = grid_nodes(time_sec, GRID_STEP_SEC)
    thresholds = np.atleast_1d(np.asarray(taus, dtype=float))
    n_cand, count = len(thresholds), len(xy)
    axes = detector["axes"]
    k_states = len(axes[0]["design"])
    scores = np.full((n_cand, count), np.nan, dtype=np.float32)
    rejected = np.zeros((n_cand, count), dtype=bool)
    state = np.zeros((2, n_cand, k_states))
    cov = np.zeros((2, n_cand, k_states, k_states))
    accepted_since = np.zeros(n_cand, dtype=np.int64)
    run_start = np.full(n_cand, np.nan)
    # Последняя принятая точка: от неё проверяется достижимость точки перезапуска.
    last_xy = np.zeros((n_cand, 2))
    last_time = np.zeros(n_cand)
    design = np.stack([axis["design"] for axis in axes])
    # Уровень модели — первая точка отрезка; задаётся при перезапуске прогноза.
    level = np.zeros((2, n_cand))
    prev_node, prev_time = None, None
    for index in np.flatnonzero(first):
        moment = time_sec[index]
        if prev_node is None or moment - prev_time > TIME_GAP_SEC:
            restart = np.ones(n_cand, dtype=bool)
        else:
            restart = restart_allowed(xy[index], moment, last_xy, last_time, run_start, detector["speed_limit"])
            for axis_index, axis in enumerate(axes):
                power, noise = _horizon_matrices(axis, int(node[index] - prev_node))
                state[axis_index] = state[axis_index] @ power.T
                cov[axis_index] = power @ cov[axis_index] @ power.T + noise
        if np.any(restart):
            state[:, restart] = 0.0
            cov[:, restart] = np.eye(k_states) * _DIFFUSE_VAR_M2
            accepted_since[restart] = 0
            level[:, restart] = xy[index][:, None]
            run_start[restart] = np.nan
        innovation = xy[index][:, None] - np.einsum("ack,ak->ac", state, design) - level
        cov_design = np.einsum("ackl,al->ack", cov, design)
        variance = np.einsum("ack,ak->ac", cov_design, design)
        nis = np.sum(innovation**2 / variance, axis=0)
        unscored = accepted_since < detector["n_unscored"]
        reject = ~unscored & (nis > thresholds)
        scores[:, index] = np.where(unscored, np.nan, nis)
        rejected[:, index] = reject
        accept = ~reject
        gain = cov_design / variance[:, :, None]
        state[:, accept] += gain[:, accept] * innovation[:, accept, None]
        cov[:, accept] -= gain[:, accept, :, None] * cov_design[:, accept, None, :]
        accepted_since[accept] += 1
        last_xy[accept] = xy[index]
        last_time[accept] = moment
        run_start = np.where(reject, np.where(np.isnan(run_start), moment, run_start), np.nan)
        prev_node, prev_time = node[index], moment
    return scores, rejected


def select_forecast_threshold(
    frame: pd.DataFrame, detector: dict[str, object], mcc_band: float, n_quantiles: int = 180
) -> dict[str, float]:
    """Подбирает порог модели прогноза: сетка по оценке без замены и χ²₀.₉₉₉(2), прогон с заменой на каждого кандидата.

    Args:
        frame: Обучающий DataFrame (time, lat, lon, x, y, l).
        detector: Результат fit_forecast_detector.
        mcc_band: Допуск по MCC относительно лучшего значения.
        n_quantiles: Число квантилей в сетке порогов (см. threshold_candidates).

    Returns:
        dict[str, float]: tau, train_mcc, train_f1, train_dl_m, best_mcc, band_size.
    """
    base_scores, _ = forecast_detect(frame, detector, np.array([np.inf]))
    finite = base_scores[0][np.isfinite(base_scores[0])]
    candidates = np.unique(np.r_[threshold_candidates(finite, n_quantiles), FORECAST_CHI999])
    scores, rejected = forecast_detect(frame, detector, candidates)
    ground_truth_anomaly = frame["l"].astype(str).to_numpy() == "anomaly"
    records = [
        _threshold_record(float(tau), rejected[row], frame, ground_truth_anomaly, ~np.isnan(scores[row]))
        for row, tau in enumerate(candidates)
    ]
    return select_threshold_from_records(records, mcc_band)


def forecast_predict(
    frame: pd.DataFrame, detector: dict[str, object], tau: float
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Разметка модели прогноза при одном пороге.

    Args:
        frame: DataFrame с колонками x, y и time.
        detector: Результат fit_forecast_detector.
        tau: Порог NIS.

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray]: Оценка аномальности (NaN — без оценки), маска anomaly,
        маска точек с оценкой.
    """
    scores, rejected = forecast_detect(frame, detector, np.array([tau]))
    values = scores[0].astype(np.float64)
    return values, rejected[0], ~np.isnan(values)


def time_segments(frame: pd.DataFrame, mask: np.ndarray) -> list[np.ndarray]:
    """Индексы точек маски, разрезанные по разрывам по времени; из точек с одинаковым временем — первая.

    Args:
        frame: DataFrame с колонкой time.
        mask: Отбираемые точки.

    Returns:
        list[np.ndarray]: Индексы точек каждого отрезка.
    """
    time_sec = time_seconds_from_frame(frame)
    index = np.flatnonzero(mask)
    if index.size == 0:
        return []
    index = index[np.r_[True, np.diff(time_sec[index]) > 0.0]]
    return np.split(index, np.flatnonzero(np.diff(time_sec[index]) > TIME_GAP_SEC) + 1)


def load_or_train(name: str, train: pd.DataFrame, fit: Callable[[], dict[str, object]]) -> dict[str, object]:
    """Обученная модель: из файла TRAINED_MODELS_DIR / name.pt или обучение с сохранением.

    Файл хранит отпечаток обучающей выборки; при другом отпечатке (другие данные или блок) модель обучается заново.
    Изменения кода отпечаток не отражает: после них — FORCE_TRAIN = True.

    Args:
        name: Имя файла модели без расширения.
        train: Обучающая выборка.
        fit: Обучение: состояние модели — всё, что нужно для разметки test и печати.

    Returns:
        dict[str, object]: Состояние модели.
    """
    TRAINED_MODELS_DIR.mkdir(parents=True, exist_ok=True)
    path = TRAINED_MODELS_DIR / f"{name}.pt"
    fingerprint = int(pd.util.hash_pandas_object(train[["time", "x", "y", "l"]], index=False).sum())
    if not FORCE_TRAIN and path.exists():
        saved = torch.load(path, map_location=DEVICE, weights_only=False)
        if saved["fingerprint"] == fingerprint:
            print(f"  модель загружена: {path.name}")
            return saved["state"]
        print(f"  {path.name} обучен на другой обучающей выборке — модель обучается заново")
    state = fit()
    torch.save({"fingerprint": fingerprint, "state": state}, path)
    print(f"  модель сохранена: {path.name}")
    return state


def move_segments(frame: pd.DataFrame) -> list[np.ndarray]:
    """Обучающие отрезки штатного хода: подряд идущие точки штатного хода между разрывами по времени.

    Отрезок рвётся на каждой аномальной точке: точки штатного хода по разные стороны серии аномалий
    не склеиваются, и прыжок через серию не попадает в обучение как обычный переход.

    Args:
        frame: DataFrame с колонками time и l.

    Returns:
        list[np.ndarray]: Индексы точек каждого отрезка.
    """
    move = frame["l"].astype(str).to_numpy() == "move"
    segments = []
    for segment in time_segments(frame, np.ones(len(frame), dtype=bool)):
        cuts = np.flatnonzero(np.diff(move[segment].astype(np.int8)) != 0) + 1
        segments.extend(part for part in np.split(segment, cuts) if move[part[0]])
    return segments


def segments_layout(segments: list[np.ndarray]) -> tuple[np.ndarray, np.ndarray]:
    """Точки отрезков подряд и маска первой точки отрезка.

    Args:
        segments: Индексы точек каждого отрезка.

    Returns:
        tuple[np.ndarray, np.ndarray]: индексы точек отрезков подряд (P,); маска первой точки отрезка (P,).
    """
    if not segments:
        raise RuntimeError("Нет отрезков для признаков.")
    first = np.zeros(sum(len(segment) for segment in segments), dtype=bool)
    first[np.r_[0, np.cumsum([len(segment) for segment in segments])[:-1]]] = True
    return np.concatenate(segments), first


def move_speeds(train: pd.DataFrame) -> np.ndarray:
    """Скорости между соседними точками обучающих отрезков штатного хода, м/с.

    Args:
        train: Обучающий DataFrame с колонками x, y, time и l.

    Returns:
        np.ndarray: Скорости переходов всех отрезков.
    """
    xy = enu_xy_from_frame(train)
    time_sec = time_seconds_from_frame(train)
    speeds = []
    for segment in move_segments(train):
        if len(segment) > 1:
            step = np.diff(xy[segment], axis=0)
            speeds.append(np.hypot(step[:, 0], step[:, 1]) / np.diff(time_sec[segment]))
    if not speeds:
        raise RuntimeError("Нет переходов штатного хода в обучающей выборке.")
    return np.concatenate(speeds)


def restart_speed_limit(train: pd.DataFrame) -> float:
    """Предел скорости перезапуска прогноза — наибольшая скорость штатного хода обучающей выборки, м/с.

    Общий для всех моделей прогноза (AR, ARIMA, Kalman, GP, GP-EKF): одинаковые условия перезапуска.

    Args:
        train: Обучающий DataFrame с колонками x, y, time и l.

    Returns:
        float: Предел скорости, м/с.
    """
    return float(np.max(move_speeds(train)))


def restart_allowed(
    point: np.ndarray,
    moment: float,
    last_xy: np.ndarray,
    last_time: np.ndarray,
    run_start: np.ndarray,
    speed_limit: float,
) -> np.ndarray:
    """Перезапуск прогноза по серии отвергнутых точек, для нескольких порогов сразу.

    Серия отвергнутых точек длиннее разрыва по времени перезапускает прогноз, только если точка достижима от
    последней принятой со скоростью не выше предела: иначе опорной стала бы точка смещённого трека аномалии.

    Args:
        point: Координаты ENU точки (2,), м.
        moment: Время точки, с.
        last_xy: Последние принятые точки (C, 2), м.
        last_time: Их время (C,), с.
        run_start: Начало серии отвергнутых точек (C,), с; NaN — серии нет.
        speed_limit: Предел скорости, м/с (restart_speed_limit).

    Returns:
        np.ndarray: Маска перезапуска (C,).
    """
    step = point[None, :] - last_xy
    speed = np.hypot(step[:, 0], step[:, 1]) / np.maximum(moment - last_time, 1e-9)
    return (moment - run_start > TIME_GAP_SEC) & (speed <= speed_limit)


def feature_keys(before: np.ndarray, rows: np.ndarray) -> np.ndarray:
    """Ключи признаков точек во втором проходе: точка, её предшественник и предшественник предшественника.

    Признаки точки (dbscan_features) определяются этой тройкой: равные ключи — равные признаки.

    Args:
        before: Позиция предыдущей принятой точки для каждой точки; -1 — нет.
        rows: Позиции точек.

    Returns:
        np.ndarray: Ключи (R, 3).
    """
    prev = before[rows]
    return np.column_stack([rows, prev, np.where(prev >= 0, before[np.maximum(prev, 0)], -1)])


def cached_rows(
    keys: np.ndarray, cache: dict[bytes, object], compute: Callable[[np.ndarray], np.ndarray]
) -> np.ndarray:
    """Результаты по ключам: найденные — из кэша, недостающие — compute и в кэш.

    Args:
        keys: Ключи (R, d), целые; в одном вызове не повторяются.
        cache: Кэш ключ → результат строки, общий для кандидатов порога.
        compute: Результаты недостающих строк по маске (R,) — массив (M,) или (M, ...).

    Returns:
        np.ndarray: Результаты (R,) или (R, ...).
    """
    packed = np.ascontiguousarray(keys, dtype=np.int64)
    packed = packed.view(np.dtype((np.void, 8 * packed.shape[1]))).ravel().tolist()
    missing = np.array([key not in cache for key in packed], dtype=bool)
    if np.any(missing):
        values = compute(missing)
        cache.update(zip((key for key, absent in zip(packed, missing) if absent), values))
    return np.array([cache[key] for key in packed])


def dbscan_segments(frame: pd.DataFrame, mask: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Точки маски по отрезкам между разрывами по времени и маска первой точки отрезка.

    Args:
        frame: DataFrame с колонкой time.
        mask: Отбираемые точки.

    Returns:
        tuple[np.ndarray, np.ndarray]: индексы точек отрезков подряд (P,); маска первой точки отрезка (P,).
    """
    return segments_layout(time_segments(frame, mask))


def dbscan_features(xy: np.ndarray, time_sec: np.ndarray, before: np.ndarray) -> np.ndarray:
    """Признаки (v, a_long, a_lat) по предыдущей принятой точке ``before`` и её предыдущей принятой точке.

    Args:
        xy: Координаты ENU точек отрезков подряд (P, 2), м.
        time_sec: Время точек, с.
        before: Позиция предыдущей принятой точки того же отрезка; -1 — нет.

    Returns:
        np.ndarray: (P, 3): скорость, м/с; продольное и поперечное ускорение, м/с²; NaN — точка без оценки.
    """
    features = np.full((len(xy), 3), np.nan)
    prev = before
    prev2 = np.where(prev >= 0, before[np.maximum(prev, 0)], -1)
    valid = (prev >= 0) & (prev2 >= 0)
    cur, one, two = np.flatnonzero(valid), prev[valid], prev2[valid]
    step = xy[cur] - xy[one]
    back = xy[one] - xy[two]
    dt = time_sec[cur] - time_sec[one]
    speed = np.hypot(step[:, 0], step[:, 1]) / dt
    prev_speed = np.hypot(back[:, 0], back[:, 1]) / (time_sec[one] - time_sec[two])
    turn = wrap_angle(np.arctan2(step[:, 0], step[:, 1]) - np.arctan2(back[:, 0], back[:, 1]))
    features[cur] = np.column_stack([speed, (speed - prev_speed) / dt, speed * turn / dt])
    return features


def rpca_window_starts(first: np.ndarray, scored: np.ndarray, length: int) -> np.ndarray:
    """Начала окон из ``length`` подряд идущих точек с признаками внутри одного отрезка.

    Args:
        first: Маска первой точки отрезка (P,).
        scored: Маска точек с признаками (P,).
        length: Длина окна, точки.

    Returns:
        np.ndarray: Позиции начала окон.
    """
    count = len(first)
    if count < length:
        return np.empty(0, dtype=np.int64)
    segment = np.cumsum(first)
    scored_sum = np.r_[0, np.cumsum(scored)]
    starts = np.arange(count - length + 1)
    inside = scored_sum[starts + length] - scored_sum[starts] == length
    return starts[inside & (segment[starts + length - 1] == segment[starts])]


def train_with_early_stopping(model: nn.Module, loader: DataLoader, batch_loss, calib_loss) -> dict[str, object]:
    """Adam до NN_MAX_EPOCHS эпох с ранней остановкой и снижением скорости обучения на плато.

    Эпоха лучше прежней, если ошибка на калибровочном блоке упала больше чем на NN_MIN_DELTA (относительно); после
    NN_PATIENCE эпох без улучшения обучение прекращается, веса — лучшей эпохи. Скорость обучения уменьшается
    в NN_LR_FACTOR раз после NN_LR_PATIENCE эпох без улучшения.

    Args:
        model: Сеть на DEVICE.
        loader: Пакеты обучающих тензоров.
        batch_loss: Функция (сеть, список тензоров пакета на DEVICE) -> скалярная ошибка для градиента.
        calib_loss: Функция (сеть) -> ошибка на калибровочном блоке (float).

    Returns:
        dict[str, object]: epochs, calib_loss — лучшая ошибка калибровки, stopped_by_cap — остановка по лимиту эпох.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=TORCH_LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, factor=NN_LR_FACTOR, patience=NN_LR_PATIENCE, threshold=NN_MIN_DELTA
    )
    best_loss, best_state, stale, epochs = np.inf, None, 0, 0
    for epoch in range(1, NN_MAX_EPOCHS + 1):
        model.train()
        for batch in loader:
            optimizer.zero_grad()
            loss = batch_loss(model, [tensor.to(DEVICE) for tensor in batch])
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            value = float(calib_loss(model))
        scheduler.step(value)
        epochs = epoch
        if best_state is None or value < best_loss - NN_MIN_DELTA * abs(best_loss):
            best_loss, stale = value, 0
            best_state = {key: tensor.detach().clone() for key, tensor in model.state_dict().items()}
        else:
            stale += 1
            if stale >= NN_PATIENCE:
                break
    model.load_state_dict(best_state)
    model.eval()
    stopped_by_cap = stale < NN_PATIENCE
    if stopped_by_cap:
        print(f"  предупреждение: обучение упёрлось в лимит {NN_MAX_EPOCHS} эпох, плато не достигнуто")
    return {"epochs": epochs, "calib_loss": best_loss, "stopped_by_cap": stopped_by_cap}


# Оценка фильтра Калмана постоянной скорости — общая для фильтра Калмана и GP-EKF (сглаживатель, начальный шум).
KALMAN_SIGMA_ACC_START = 0.35
KALMAN_SIGMA_MEAS_START = 40.0
KALMAN_SIGMA_ACC_MIN = 1e-3
KALMAN_SIGMA_MEAS_MIN = 0.5
KALMAN_CHI999 = float(chi2.ppf(0.999, 2))


def kalman_process_noise(dt: np.ndarray, sigma_acc: float) -> np.ndarray:
    """Ковариация шума процесса одной оси при дискретном белом ускорении, shape (2, 2, n).

    Args:
        dt: Шаги времени, с.
        sigma_acc: СКО ускорения, м/с².

    Returns:
        np.ndarray: Матрицы sigma_acc² · [[dt⁴/4, dt³/2], [dt³/2, dt²]].
    """
    dt = np.asarray(dt, dtype=float)
    noise = np.array([[dt**4 / 4.0, dt**3 / 2.0], [dt**3 / 2.0, dt**2]])
    return noise * sigma_acc**2


def kalman_likelihood_data(train: pd.DataFrame) -> dict[str, np.ndarray]:
    """Точки штатного хода обучающей выборки для условного правдоподобия.

    Отрезки — обучающие отрезки штатного хода (move_segments); отрезки короче трёх точек отбрасываются.

    Args:
        train: Обучающий DataFrame с колонками x, y, time и l.

    Returns:
        dict[str, np.ndarray]: xy (n, 2), step — интервал до следующей точки, с; boundary — последняя точка
        отрезка; counted — точки, входящие в правдоподобие (с третьей точки отрезка).
    """
    segments = [segment for segment in move_segments(train) if len(segment) >= 3]
    if not segments:
        raise RuntimeError("Нет отрезка штатного хода из трёх точек для оценки фильтра Калмана.")
    index = np.concatenate(segments)
    time_sec = time_seconds_from_frame(train)[index]
    lengths = np.array([len(segment) for segment in segments])
    ends = np.cumsum(lengths) - 1
    boundary = np.zeros(len(index), dtype=bool)
    boundary[ends] = True
    counted = np.ones(len(index), dtype=bool)
    counted[ends - lengths + 1] = False
    counted[ends - lengths + 2] = False
    return {
        "xy": enu_xy_from_frame(train)[index],
        "step": np.r_[np.diff(time_sec), 0.0],
        "boundary": boundary,
        "counted": counted,
    }


def kalman_loglike(data: dict[str, np.ndarray], sigma_acc: float, sigma_meas: float) -> float:
    """Условное логарифмическое правдоподобие модели постоянной скорости по двум осям (statsmodels).

    На границе отрезков состояние сбрасывается в диффузное; первые две точки отрезка не входят.

    Args:
        data: Результат kalman_likelihood_data.
        sigma_acc: СКО ускорения, м/с².
        sigma_meas: СКО измерения, м.

    Returns:
        float: Сумма логарифмов правдоподобия.
    """
    step = data["step"]
    transition = np.zeros((2, 2, len(step)))
    transition[0, 0] = transition[1, 1] = 1.0
    transition[0, 1] = step
    noise = kalman_process_noise(step, sigma_acc)
    transition[:, :, data["boundary"]] = 0.0
    noise[:, :, data["boundary"]] = np.eye(2)[:, :, None] * _DIFFUSE_VAR_M2
    total = 0.0
    for column in range(2):
        values = data["xy"][:, column]
        kalman = KalmanFilter(k_endog=1, k_states=2, k_posdef=2)
        kalman.bind(values - values[0])
        kalman["design"] = np.array([[1.0, 0.0]])
        kalman["obs_cov"] = np.array([[sigma_meas**2]])
        kalman["transition"] = transition
        kalman["selection"] = np.eye(2)
        kalman["state_cov"] = noise
        kalman.initialize_approximate_diffuse(_DIFFUSE_VAR_M2)
        total += float(np.sum(kalman.loglikeobs()[data["counted"]]))
    return total


def fit_kalman_on_train(train: pd.DataFrame) -> tuple[float, float]:
    """Оценивает sigma_acc и sigma_meas по максимуму условного правдоподобия на штатном ходу.

    Args:
        train: Обучающий DataFrame с колонками x, y, time и l.

    Returns:
        tuple[float, float]: (sigma_acc, м/с²; sigma_meas, м), снизу ограничены KALMAN_SIGMA_ACC_MIN
        и KALMAN_SIGMA_MEAS_MIN.
    """
    data = kalman_likelihood_data(train)
    # Нижние границы: при меньших sigma правдоподобие statsmodels численно вырождается в константу.
    lower = np.log([KALMAN_SIGMA_ACC_MIN, KALMAN_SIGMA_MEAS_MIN])
    result = minimize(
        lambda log_sigma: -kalman_loglike(data, *np.exp(log_sigma)),
        np.log([KALMAN_SIGMA_ACC_START, KALMAN_SIGMA_MEAS_START]),
        method="L-BFGS-B",
        bounds=[(lower[0], None), (lower[1], None)],
    )
    if np.any(np.isclose(result.x, lower)):
        print("  предупреждение: оценка σ фильтра Калмана на нижней границе")
    sigma_acc, sigma_meas = np.exp(result.x)
    return float(sigma_acc), float(sigma_meas)


## MPCT (Maximum Physically Consistent Trajectory)

### Постановка

В каждом отрезке ряда выбирается самая длинная подпоследовательность точек, в которой судно могло пройти от каждой выбранной точки к следующей со скоростью не выше $v_+$. Аномалия — точка, не вошедшая в эту подпоследовательность.

### Модель и решение

Подпоследовательность $i_1<i_2<\ldots<i_m$ точек отрезка допустима, если для соседних выбранных точек

$$\frac{\lVert\mathbf p_{i_{k+1}}-\mathbf p_{i_k}\rVert}{t_{i_{k+1}}-t_{i_k}}\le v_+,\qquad \mathbf p_i=(x_i,y_i).$$

Выбирается допустимая подпоследовательность наибольшей длины, $\hat a_i=1$ для точек вне неё. Она находится динамическим программированием: предшественник ищется среди точек за последние 180 с, из более ранних — только конец самой длинной цепочки и только если он достижим со скоростью не выше $v_+$. Решение бинарное, роль порога играет $v_+$; выброс не оставляет следа на соседних точках.

Свойство модели: проверяется только скорость между выбранными точками, кинематика (ускорение, курс) не учитывается. Смещённый эпизод длиннее 180 с, согласованный внутри, принимается целиком или выбирается вместо штатных точек, если он длиннее их; отсюда большие RMSE и ошибка длины траектории.

### Время и точки без оценки

Отрезки — между разрывами по времени больше 180 с. Из точек с одинаковым временем остаётся первая, остальные — без оценки; все остальные точки, включая первую точку отрезка, получают решение.

### Обучение и выбор порога

Обучающий отрезок — подряд идущие точки штатного хода обучающей выборки; отрезок рвётся на каждой аномальной точке и на разрыве по времени больше 180 с, поэтому прыжок через серию аномалий не попадает в обучение как обычный переход. $v_+$ — из 30 значений, равномерных в логарифмической шкале от 95-го процентиля скоростей между соседними точками обучающих отрезков до трёх их максимумов: ниже 95-го процентиля отвергался бы обычный ход, выше трёх максимумов модель не отвергает ничего. Выбор $v_+$ — по обучающей выборке: наибольший MCC, среди значений в полосе MCC (не дальше 0.02 от лучшего) — наименьшая операционная ошибка длины.

In [ ]:
MPCT_GRID_SIZE = 30
MPCT_GRID_LOW_PERCENTILE = 95.0
MPCT_GRID_HIGH_FACTOR = 3.0


def _mpct_segments(frame: pd.DataFrame) -> tuple[np.ndarray, list[np.ndarray]]:
    """Точки ряда по отрезкам между разрывами по времени (из точек с одинаковым временем — первая)."""
    segments = time_segments(frame, np.ones(len(frame), dtype=bool))
    return np.concatenate(segments), segments


def mpct_select_points(xy: np.ndarray, time_sec: np.ndarray, v_max: np.ndarray) -> np.ndarray:
    """Самая длинная допустимая подпоследовательность одного отрезка для нескольких v_+ (динамическое программирование).

    Соседние выбранные точки допустимы, если скорость между ними в плоскости ENU не выше v_+. Предшественник ищется
    среди точек за последние TIME_GAP_SEC; более ранний — только лучший конец старой цепочки и только если он
    достижим со скоростью не выше v_+, иначе цепочка начинается заново. При равной длине — ближайший по времени
    предшественник и самый поздний конец.

    Args:
        xy: Координаты ENU точек отрезка (P, 2), м.
        time_sec: Время точек, с, строго по возрастанию.
        v_max: Пределы скорости (C,), м/с.

    Returns:
        np.ndarray: Маска выбранных точек (C, P).
    """
    count, n_cand = len(xy), len(v_max)
    best = np.zeros((n_cand, count), dtype=np.int64)
    pred = np.full((n_cand, count), -1, dtype=np.int64)
    run_best = np.zeros(n_cand, dtype=np.int64)
    run_arg = np.full(n_cand, -1, dtype=np.int64)
    rows = np.arange(n_cand)
    oldest = 0
    for index in range(count):
        # Точки старше TIME_GAP_SEC — кандидат один: лучший конец старой цепочки (накопленный максимум).
        while oldest < index and time_sec[index] - time_sec[oldest] > TIME_GAP_SEC:
            better = best[:, oldest] >= run_best
            run_best = np.where(better, best[:, oldest], run_best)
            run_arg = np.where(better, oldest, run_arg)
            oldest += 1
        # Переход к старой цепочке — только со скоростью не выше v_+: иначе опорой стал бы хвост аномального эпизода.
        length = np.ones(n_cand, dtype=np.int64)
        source = np.full(n_cand, -1, dtype=np.int64)
        linked = run_arg >= 0
        if np.any(linked):
            old = run_arg[linked]
            jump = xy[index] - xy[old]
            reachable = np.hypot(jump[:, 0], jump[:, 1]) / (time_sec[index] - time_sec[old]) <= v_max[linked]
            rows_linked = np.flatnonzero(linked)[reachable]
            length[rows_linked] = run_best[rows_linked] + 1
            source[rows_linked] = run_arg[rows_linked]
        if oldest < index:
            step = xy[index] - xy[oldest:index]
            speed = np.hypot(step[:, 0], step[:, 1]) / (time_sec[index] - time_sec[oldest:index])
            window = np.where(speed[None, :] <= v_max[:, None], best[:, oldest:index], 0)
            # Последний максимум — ближайший по времени предшественник.
            last = window.shape[1] - 1 - np.argmax(window[:, ::-1], axis=1)
            top = window[rows, last]
            use = (top > 0) & (top + 1 >= length)
            length = np.where(use, top + 1, length)
            source = np.where(use, oldest + last, source)
        best[:, index] = length
        pred[:, index] = source
    chosen = np.zeros((n_cand, count), dtype=bool)
    for row in range(n_cand):
        cursor = count - 1 - int(np.argmax(best[row, ::-1]))
        while cursor >= 0:
            chosen[row, cursor] = True
            cursor = pred[row, cursor]
    return chosen


def mpct_detect(frame: pd.DataFrame, v_max: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Разметка MPCT для нескольких v_+: аномалия — точка вне самой длинной допустимой подпоследовательности отрезка.

    Args:
        frame: DataFrame с колонками x, y и time.
        v_max: Пределы скорости (C,), м/с.

    Returns:
        tuple[np.ndarray, np.ndarray]: маска аномалий (C, N); маска точек с решением (N,) — без повторов времени.
    """
    limits = np.atleast_1d(np.asarray(v_max, dtype=float))
    xy = enu_xy_from_frame(frame)
    time_sec = time_seconds_from_frame(frame)
    index, segments = _mpct_segments(frame)
    rejected = np.zeros((len(limits), len(frame)), dtype=bool)
    for segment in segments:
        rejected[:, segment] = ~mpct_select_points(xy[segment], time_sec[segment], limits)
    scored = np.zeros(len(frame), dtype=bool)
    scored[index] = True
    return rejected, scored


def mpct_speed_grid(train: pd.DataFrame) -> np.ndarray:
    """Сетка v_+: от 95-го процентиля скоростей между соседними точками штатного хода до трёх их максимумов.

    Args:
        train: Обучающий DataFrame с колонками x, y, time и l.

    Returns:
        np.ndarray: MPCT_GRID_SIZE значений, м/с.
    """
    speeds = move_speeds(train)
    low = np.percentile(speeds, MPCT_GRID_LOW_PERCENTILE)
    return np.geomspace(low, MPCT_GRID_HIGH_FACTOR * speeds.max(), MPCT_GRID_SIZE)


def select_mpct_speed(frame: pd.DataFrame, mcc_band: float) -> dict[str, float]:
    """Подбирает v_+ по сетке: максимум MCC, в полосе — минимальная операционная ошибка длины.

    Args:
        frame: Обучающий DataFrame (time, lat, lon, x, y, l).
        mcc_band: Допуск по MCC относительно лучшего значения.

    Returns:
        dict[str, float]: tau (выбранное v_+), train_mcc, train_f1, train_dl_m, best_mcc, band_size.
    """
    grid = mpct_speed_grid(frame)
    rejected, scored = mpct_detect(frame, grid)
    ground_truth_anomaly = frame["l"].astype(str).to_numpy() == "anomaly"
    records = [
        _threshold_record(float(v_max), rejected[row], frame, ground_truth_anomaly, scored)
        for row, v_max in enumerate(grid)
    ]
    return select_threshold_from_records(records, mcc_band)


mpct_scenarios = (
    (DATASET_NAME, dataset),
)

mpct_rows: list[dict[str, object]] = []
mpct_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in mpct_scenarios:
    train = bundle["train"]
    test = bundle["test"]

    print("=" * 72)
    print(f"MPCT: {scenario_name}")
    started = time.perf_counter()
    def train_model() -> dict[str, object]:
        """Обучение, выбор порога и оценки train для печати медиан."""
        return {"grid": mpct_speed_grid(train), "selection": select_mpct_speed(train, MCC_BAND)}
    state = load_or_train("mpct", train, train_model)
    grid, selection = state["grid"], state["selection"]
    v_max = float(selection["tau"])
    print(
        f"  сетка v_+ {grid[0]:.3g}–{grid[-1]:.3g} м/с ({len(grid)} значений); выбрано v_+ {v_max:.3g} м/с: "
        f"MCC train {selection['train_mcc']:.4f} (лучший {selection['best_mcc']:.4f}, "
        f"в полосе {int(selection['band_size'])}), DL train {selection['train_dl_m'] / 1000:.4f} км"
    )
    test_rejected, scored_test = mpct_detect(test, np.array([v_max]))
    predicted_test = test_rejected[0]
    metrics = evaluate_test(test, predicted_test, scored_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м, без оценки {metrics['unscored_share']:.2%} "
        f"(время расчёта {elapsed:.1f} с)"
    )

    mpct_rows.append(test_report_row(scenario_name, metrics))
    mpct_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "scored_test": scored_test,
            "metrics": metrics,
        }
    )

mpct_summary = pd.DataFrame(mpct_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    mpct_summary.loc[:, report_columns],
    caption="MPCT: сводные метрики на test",
)

if SHOW_PLOTS:
    _mpct_ideal_test = mpct_scenario_results[0]["test"]
    _mpct_map_tail = min(PLOT_TAIL, len(_mpct_ideal_test))
    print(f"\nИдеальная разметка: последние {nfmt(_mpct_map_tail)} точек теста (bbox: move + 5%)")
    _mpct_dummy_move = np.zeros(len(_mpct_ideal_test), dtype=bool)
    _mpct_truth_pred, _mpct_truth_gt = trajectory_series_pair(_mpct_ideal_test, _mpct_dummy_move)
    _mpct_truth_pred = slice_trajectory_series(_mpct_truth_pred, _mpct_map_tail)
    _mpct_truth_gt = slice_trajectory_series(_mpct_truth_gt, _mpct_map_tail)
    plot_track_maps(
        _mpct_truth_pred,
        _mpct_truth_gt,
        map_panel="ground_truth",
        title="MPCT — идеальная разметка",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )

for item in mpct_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    if SHOW_PLOTS:
        print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
        prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"], item["scored_test"])
        prediction = slice_trajectory_series(prediction, tail)
        ground_truth = slice_trajectory_series(ground_truth, tail)
        plot_track_maps(
            prediction,
            ground_truth,
            map_panel="model",
            title=f"MPCT — {item['name']}",
            figsize=MAP_FIGSIZE,
            annotate=False,
        )

## AR(2) для обнаружения аномалий траектории

### Постановка

Авторегрессия второго порядка описывает уровень ряда — плоскую координату — через два предыдущих значения. Аномалия — точка, плохо согласованная с причинным прогнозом.

### Модель и оценка аномальности

Для восточной координаты (аналогично для северной):

$$x_j=\mu_x+\eta_j,\qquad \eta_j=\phi_1\eta_{j-1}+\phi_2\eta_{j-2}+\varepsilon_j,\qquad \varepsilon_j\sim\mathcal N(0,\sigma_x^2),$$

$$\hat x_j=\mu_x+\phi_1(x_{j-1}-\mu_x)+\phi_2(x_{j-2}-\mu_x),$$
где $\mu_x$ — первая точка отрезка.

Оценка аномальности — квадрат нормированной инновации (NIS):

$$s_j=\frac{(x_j-\hat x_j)^2}{D_{x,j}}+\frac{(y_j-\hat y_j)^2}{D_{y,j}},\qquad \hat a_j=\mathbf{1}[s_j>\tau],$$

где $D_{x,j}$, $D_{y,j}$ — дисперсии прогноза осей, растут с горизонтом $h$; при верной модели $s_j\sim\chi^2_2$.

Отвергнутая точка в лаги не входит: следующий прогноз строится от последних принятых точек.

### Время и точки без оценки

Ряд переносится на регулярную сетку по времени с шагом 10 с, $j$ — узел сетки. В узлах без точек (пропусках) прогноз продолжается без обновления, поэтому прогноз точки после пропуска строится на $h$ шагов от последней принятой. Прогноз начинается заново после разрыва по времени больше 180 с, а также после 180 с подряд отвергнутых точек — если точка достижима от последней принятой со скоростью не выше наибольшей скорости штатного хода обучающей выборки (предел общий для всех моделей прогноза); иначе опорной стала бы точка смещённого трека аномального эпизода. Первые две точки нового отрезка — без оценки.

### Обучение и выбор порога

1. Параметры $\mu$, $\phi_1$, $\phi_2$, $\sigma^2$ каждой оси оцениваются по максимуму правдоподобия только по точкам штатного хода обучающей выборки: правдоподобие суммируется по обучающим отрезкам, как у MPCT. Для оценки параметров отрезок рвётся и на любом шаге времени, отличном от шага сетки 10 с: по участкам «точка — пропуск — точка» правдоподобие выбирало бы прогноз через узел, $\phi\approx(0,1)$, вместо прогноза по предыдущей точке.
2. Оценка аномальности на обучающей выборке считается по всем точкам.
3. Сетка кандидатов порога строится по оценке аномальности без замены отвергнутых точек, в неё добавляется $\chi^2_{0.999}(2)$; для каждого кандидата детектор заново проходит обучающую выборку с заменой; выбор порога — как $v_+$ у MPCT.

In [ ]:
AR_ORDER = (2, 0, 0)
AR_TREND = "n"

ar_scenarios = (
    (DATASET_NAME, dataset),
)

ar_rows: list[dict[str, object]] = []
ar_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in ar_scenarios:
    train = bundle["train"]
    test = bundle["test"]
    anomaly_train = train["l"].astype(str).to_numpy() == "anomaly"

    print("=" * 72)
    print(f"AR(2): {scenario_name}")
    started = time.perf_counter()
    def train_model() -> dict[str, object]:
        """Обучение, выбор порога и оценки train для печати медиан."""
        detector = fit_forecast_detector(train, AR_ORDER, AR_TREND)
        selection = select_forecast_threshold(train, detector, MCC_BAND)
        scores = forecast_predict(train, detector, selection["tau"])[0]
        return {"detector": detector, "selection": selection, "train_scores": scores}
    state = load_or_train("ar", train, train_model)
    detector, selection, train_scores = state["detector"], state["selection"], state["train_scores"]
    # Контроль притяжения к уровню — первой точке отрезка: смещение за шаг (1 - phi1 - phi2) * |o - mu|
    # по точкам штатного хода тестовой выборки; прогноз строится от принятых точек, а не от аномальных.
    test_move = test["l"].astype(str).to_numpy() == "move"
    segment_id = np.cumsum(np.r_[0, np.diff(time_seconds_from_frame(test)[test_move]) > TIME_GAP_SEC])
    segment_first = np.r_[0, np.flatnonzero(np.diff(segment_id)) + 1]
    for column, axis_name, params in zip(("x", "y"), ("x", "y"), detector["params"]):
        phi_1, phi_2, sigma2 = params
        pull = 1.0 - phi_1 - phi_2
        axis_values = test[column].to_numpy(dtype=float)[test_move]
        shift_m = pull * np.max(np.abs(axis_values - axis_values[segment_first][segment_id]))
        print(
            f"  ось {axis_name}: φ1 {phi_1:.6f}, φ2 {phi_2:.6f}, σ {np.sqrt(sigma2):.2f} м, "
            f"1-φ1-φ2 {pull:.2e}, наибольшее смещение прогноза за шаг на test {shift_m:.3f} м"
        )
    if np.any(~anomaly_train) and np.any(anomaly_train):
        print(
            f"  медиана score на train: move {np.nanmedian(train_scores[~anomaly_train]):.4g}, "
            f"anomaly {np.nanmedian(train_scores[anomaly_train]):.4g}"
        )
    print(
        f"  χ²₀.₉₉₉(df=2) = {FORECAST_CHI999:.2f}; порог {selection['tau']:.4g}: MCC train {selection['train_mcc']:.4f} "
        f"(лучший {selection['best_mcc']:.4f}, в полосе {int(selection['band_size'])}), "
        f"DL train {selection['train_dl_m'] / 1000:.4f} км"
    )

    test_scores, predicted_test, scored_test = forecast_predict(test, detector, selection["tau"])
    metrics = evaluate_test(test, predicted_test, scored_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м, без оценки {metrics['unscored_share']:.2%} "
        f"(время расчёта {elapsed:.1f} с)"
    )

    ar_rows.append(test_report_row(scenario_name, metrics))
    ar_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "scored_test": scored_test,
            "metrics": metrics,
        }
    )

ar_summary = pd.DataFrame(ar_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    ar_summary.loc[:, report_columns],
    caption="AR(2): сводные метрики на test",
)

if SHOW_PLOTS:
    _ar_ideal_test = ar_scenario_results[0]["test"]
    _ar_map_tail = min(PLOT_TAIL, len(_ar_ideal_test))
    print(f"\nИдеальная разметка: последние {nfmt(_ar_map_tail)} точек теста (bbox: move + 5%)")
    _ar_dummy_move = np.zeros(len(_ar_ideal_test), dtype=bool)
    _ar_truth_pred, _ar_truth_gt = trajectory_series_pair(_ar_ideal_test, _ar_dummy_move)
    _ar_truth_pred = slice_trajectory_series(_ar_truth_pred, _ar_map_tail)
    _ar_truth_gt = slice_trajectory_series(_ar_truth_gt, _ar_map_tail)
    plot_track_maps(
        _ar_truth_pred,
        _ar_truth_gt,
        map_panel="ground_truth",
        title="AR(2) — идеальная разметка",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )

for item in ar_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    if SHOW_PLOTS:
        print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
        prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"], item["scored_test"])
        prediction = slice_trajectory_series(prediction, tail)
        ground_truth = slice_trajectory_series(ground_truth, tail)
        plot_track_maps(
            prediction,
            ground_truth,
            map_panel="model",
            title=f"AR(2) — {item['name']}",
            figsize=MAP_FIGSIZE,
            annotate=False,
        )

## ARIMA(2,1,1) для обнаружения аномалий траектории

Как у AR(2).

### Постановка

Приращения координаты описываются процессом авторегрессии и скользящего среднего; уровень — их накопление.

### Модель и оценка аномальности

Для восточной координаты (аналогично для северной):

$$\Delta x_j=\phi_1\Delta x_{j-1}+\phi_2\Delta x_{j-2}+\varepsilon_j+\theta_1\varepsilon_{j-1},\qquad \varepsilon_j\sim\mathcal N(0,\sigma_x^2),$$

где $\Delta x_j=x_j-x_{j-1}$. Дрейф не вводится: средняя скорость по обучающей выборке физического смысла не имеет. Член $\theta_1\varepsilon_{j-1}$ описывает шум GPS: ошибка измерения входит в два соседних приращения с разными знаками.

### Время и точки без оценки

Первые три точки отрезка — без оценки: одна на приращение, две на лаги.

In [ ]:
ARIMA_ORDER = (2, 1, 1)
ARIMA_TREND = "n"

arima_scenarios = (
    (DATASET_NAME, dataset),
)

arima_rows: list[dict[str, object]] = []
arima_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in arima_scenarios:
    train = bundle["train"]
    test = bundle["test"]
    anomaly_train = train["l"].astype(str).to_numpy() == "anomaly"

    print("=" * 72)
    print(f"ARIMA(2,1,1): {scenario_name}")
    started = time.perf_counter()
    def train_model() -> dict[str, object]:
        """Обучение, выбор порога и оценки train для печати медиан."""
        detector = fit_forecast_detector(train, ARIMA_ORDER, ARIMA_TREND)
        selection = select_forecast_threshold(train, detector, MCC_BAND)
        scores = forecast_predict(train, detector, selection["tau"])[0]
        return {"detector": detector, "selection": selection, "train_scores": scores}
    state = load_or_train("arima", train, train_model)
    detector, selection, train_scores = state["detector"], state["selection"], state["train_scores"]
    for axis_name, params in zip(("x", "y"), detector["params"]):
        phi_1, phi_2, theta_1, sigma2 = params
        print(f"  ось {axis_name}: φ1 {phi_1:.6f}, φ2 {phi_2:.6f}, θ1 {theta_1:.6f}, σ {np.sqrt(sigma2):.2f} м")
    if np.any(~anomaly_train) and np.any(anomaly_train):
        print(
            f"  медиана score на train: move {np.nanmedian(train_scores[~anomaly_train]):.4g}, "
            f"anomaly {np.nanmedian(train_scores[anomaly_train]):.4g}"
        )
    print(
        f"  χ²₀.₉₉₉(df=2) = {FORECAST_CHI999:.2f}; порог {selection['tau']:.4g}: MCC train {selection['train_mcc']:.4f} "
        f"(лучший {selection['best_mcc']:.4f}, в полосе {int(selection['band_size'])}), "
        f"DL train {selection['train_dl_m'] / 1000:.4f} км"
    )

    test_scores, predicted_test, scored_test = forecast_predict(test, detector, selection["tau"])
    metrics = evaluate_test(test, predicted_test, scored_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м, без оценки {metrics['unscored_share']:.2%} "
        f"(время расчёта {elapsed:.1f} с)"
    )

    arima_rows.append(test_report_row(scenario_name, metrics))
    arima_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "scored_test": scored_test,
            "metrics": metrics,
        }
    )

arima_summary = pd.DataFrame(arima_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    arima_summary.loc[:, report_columns],
    caption="ARIMA(2,1,1): сводные метрики на test",
)

if SHOW_PLOTS:
    _arima_ideal_test = arima_scenario_results[0]["test"]
    _arima_map_tail = min(PLOT_TAIL, len(_arima_ideal_test))
    print(f"\nИдеальная разметка: последние {nfmt(_arima_map_tail)} точек теста (bbox: move + 5%)")
    _arima_dummy_move = np.zeros(len(_arima_ideal_test), dtype=bool)
    _arima_truth_pred, _arima_truth_gt = trajectory_series_pair(_arima_ideal_test, _arima_dummy_move)
    _arima_truth_pred = slice_trajectory_series(_arima_truth_pred, _arima_map_tail)
    _arima_truth_gt = slice_trajectory_series(_arima_truth_gt, _arima_map_tail)
    plot_track_maps(
        _arima_truth_pred,
        _arima_truth_gt,
        map_panel="ground_truth",
        title="ARIMA(2,1,1) — идеальная разметка",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )

for item in arima_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    if SHOW_PLOTS:
        print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
        prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"], item["scored_test"])
        prediction = slice_trajectory_series(prediction, tail)
        ground_truth = slice_trajectory_series(ground_truth, tail)
        plot_track_maps(
            prediction,
            ground_truth,
            map_panel="model",
            title=f"ARIMA(2,1,1) — {item['name']}",
            figsize=MAP_FIGSIZE,
            annotate=False,
        )

## Фильтр Калмана (постоянная скорость)

### Постановка

Скрытое состояние — положение и скорость в плоскости ENU, измерение — положение. Аномалия — точка, плохо согласованная с причинным прогнозом фильтра.

### Модель и оценка аномальности

$$\mathbf x_i=(x_i,y_i,v^x_i,v^y_i)^\top,\qquad \mathbf x_i=F_i\mathbf x_{i-1}+\mathbf w_i,\qquad \mathbf z_i=H\mathbf x_i+\boldsymbol\nu_i,$$

$$F_i=\begin{pmatrix}I&\Delta t_i I\\0&I\end{pmatrix},\qquad Q_i=\sigma_{\mathrm{acc}}^2\begin{pmatrix}\tfrac{\Delta t_i^4}{4}I&\tfrac{\Delta t_i^3}{2}I\\\tfrac{\Delta t_i^3}{2}I&\Delta t_i^2 I\end{pmatrix},\qquad R=\sigma_{\mathrm{meas}}^2 I,$$

где $I$ — единичная матрица $2\times2$, $H=(I\;0)$, $\Delta t_i=t_i-t_{i-1}$.

Оценка аномальности — квадрат нормированной инновации:

$$s_i=\mathbf r_i^\top S_i^{-1}\mathbf r_i,\qquad \mathbf r_i=\mathbf z_i-H\hat{\mathbf x}_{i\mid i-1},\qquad S_i=HP_{i\mid i-1}H^\top+R,\qquad \hat a_i=\mathbf{1}[s_i>\tau].$$

Отвергнутая точка состояние не обновляет: следующий прогноз строится от последней принятой точки. Аномальная точка, принятая после серии отвергнутых, искажает оценку скорости: следующие точки отвергаются до перезапуска прогноза.

### Время и точки без оценки

Как у AR(2): прогноз начинается заново после разрыва по времени больше 180 с и после 180 с подряд отвергнутых точек, если точка достижима от последней принятой; первые две точки отрезка — без оценки; из точек с одинаковым временем остаётся первая. Начальное состояние — по первым двум точкам отрезка:

$$\hat{\mathbf x}=\Bigl(\mathbf z_2,\ \frac{\mathbf z_2-\mathbf z_1}{\Delta t}\Bigr),\qquad P=\begin{pmatrix}R&R/\Delta t\\R/\Delta t&2R/\Delta t^2\end{pmatrix}.$$

### Обучение и выбор порога

1. $\sigma_{\mathrm{acc}}$ и $\sigma_{\mathrm{meas}}$ оцениваются по максимуму правдоподобия только по точкам штатного хода обучающей выборки: правдоподобие суммируется по обучающим отрезкам, как у MPCT, первые две точки отрезка в него не входят. Нижние границы — $\sigma_{\mathrm{acc}}\ge10^{-3}$ м/с², $\sigma_{\mathrm{meas}}\ge0.5$ м: при меньших значениях правдоподобие численно вырождается.
2. Оценка аномальности и подбор порога — как у AR(2); в сетку кандидатов добавляется $\chi^2_{0.999}(2)$.

In [ ]:
def kalman_detect(
    frame: pd.DataFrame, sigma_acc: float, sigma_meas: float, speed_limit: float, taus: np.ndarray
) -> tuple[np.ndarray, np.ndarray]:
    """NIS фильтра постоянной скорости с заменой отвергнутых точек, сразу для нескольких порогов.

    Отвергнутая точка состояние не обновляет. Прогноз начинается заново после разрыва по времени и после
    серии отвергнутых точек длиннее разрыва, если точка достижима от последней принятой (restart_allowed);
    начальное состояние — по первым двум точкам отрезка, они без оценки. Из точек с одинаковым временем остаётся первая, остальные — без оценки.

    Args:
        frame: DataFrame с колонками x, y и time.
        sigma_acc: СКО ускорения, м/с².
        sigma_meas: СКО измерения, м.
        speed_limit: Предел скорости перезапуска, м/с (restart_speed_limit).
        taus: Пороги NIS; np.inf — без замены.

    Returns:
        tuple[np.ndarray, np.ndarray]: NIS shape (C, N), NaN — точка без оценки; маска отвергнутых точек.
    """
    xy = enu_xy_from_frame(frame)
    time_sec = time_seconds_from_frame(frame)
    thresholds = np.atleast_1d(np.asarray(taus, dtype=float))
    n_cand, count = len(thresholds), len(xy)
    meas_cov = np.eye(2) * sigma_meas**2
    scores = np.full((n_cand, count), np.nan, dtype=np.float32)
    rejected = np.zeros((n_cand, count), dtype=bool)
    state = np.zeros((n_cand, 4))
    cov = np.zeros((n_cand, 4, 4))
    first_point = np.zeros((n_cand, 2))
    first_time = np.zeros(n_cand)
    phase = np.zeros(n_cand, dtype=np.int64)
    run_start = np.full(n_cand, np.nan)
    # Последняя принятая точка: от неё проверяется достижимость точки перезапуска.
    last_xy = np.zeros((n_cand, 2))
    last_time = np.zeros(n_cand)
    transition = np.eye(4)
    prev_time = None
    for index in range(count):
        moment = time_sec[index]
        if prev_time is not None and moment - prev_time <= 0.0:
            continue
        point = xy[index]
        if prev_time is None or moment - prev_time > TIME_GAP_SEC:
            restart = np.ones(n_cand, dtype=bool)
        else:
            restart = restart_allowed(point, moment, last_xy, last_time, run_start, speed_limit)
            dt = moment - prev_time
            transition[0, 2] = transition[1, 3] = dt
            axis_noise = kalman_process_noise(np.array([dt]), sigma_acc)[:, :, 0]
            process = np.zeros((4, 4))
            process[np.ix_([0, 2], [0, 2])] = axis_noise
            process[np.ix_([1, 3], [1, 3])] = axis_noise
            state = state @ transition.T
            cov = transition @ cov @ transition.T + process
        phase[restart] = 0
        run_start[restart] = np.nan
        start, second, active = phase == 0, phase == 1, phase >= 2
        first_point[start] = point
        first_time[start] = moment
        if np.any(second):
            pair_dt = moment - first_time[second]
            state[second, :2] = point
            state[second, 2:] = (point - first_point[second]) / pair_dt[:, None]
            cov[second] = 0.0
            cov[second, :2, :2] = meas_cov
            cov[second, :2, 2:] = cov[second, 2:, :2] = meas_cov / pair_dt[:, None, None]
            cov[second, 2:, 2:] = 2.0 * meas_cov / pair_dt[:, None, None] ** 2
        innovation = point - state[:, :2]
        innovation_cov = cov[:, :2, :2] + meas_cov
        solved = np.linalg.solve(innovation_cov, innovation[:, :, None])[:, :, 0]
        nis = np.einsum("ck,ck->c", innovation, solved)
        reject = active & (nis > thresholds)
        scores[:, index] = np.where(active, nis, np.nan)
        rejected[:, index] = reject
        update = active & ~reject
        gain = cov[update][:, :, :2] @ np.linalg.inv(innovation_cov[update])
        state[update] += np.einsum("ckl,cl->ck", gain, innovation[update])
        cov[update] -= gain @ cov[update][:, :2, :]
        phase[start | second] += 1
        last_xy[~reject] = point
        last_time[~reject] = moment
        run_start = np.where(
            reject, np.where(np.isnan(run_start), moment, run_start), np.where(active, np.nan, run_start)
        )
        prev_time = moment
    return scores, rejected


def select_kalman_threshold(
    frame: pd.DataFrame, sigma_acc: float, sigma_meas: float, speed_limit: float, mcc_band: float
) -> dict[str, float]:
    """Подбирает порог NIS: сетка по NIS без замены и χ²₀.₉₉₉(2), прогон с заменой на каждого кандидата.

    Args:
        frame: Обучающий DataFrame (time, lat, lon, x, y, l).
        sigma_acc: СКО ускорения, м/с².
        sigma_meas: СКО измерения, м.
        speed_limit: Предел скорости перезапуска, м/с.
        mcc_band: Допуск по MCC относительно лучшего значения.

    Returns:
        dict[str, float]: tau, train_mcc, train_f1, train_dl_m, best_mcc, band_size.
    """
    base_scores, _ = kalman_detect(frame, sigma_acc, sigma_meas, speed_limit, np.array([np.inf]))
    finite = base_scores[0][np.isfinite(base_scores[0])]
    candidates = np.unique(np.r_[threshold_candidates(finite), KALMAN_CHI999])
    scores, rejected = kalman_detect(frame, sigma_acc, sigma_meas, speed_limit, candidates)
    ground_truth_anomaly = frame["l"].astype(str).to_numpy() == "anomaly"
    records = [
        _threshold_record(float(tau), rejected[row], frame, ground_truth_anomaly, ~np.isnan(scores[row]))
        for row, tau in enumerate(candidates)
    ]
    return select_threshold_from_records(records, mcc_band)


kalman_scenarios = (
    (DATASET_NAME, dataset),
)

kalman_rows: list[dict[str, object]] = []
kalman_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in kalman_scenarios:
    train = bundle["train"]
    test = bundle["test"]
    anomaly_train = train["l"].astype(str).to_numpy() == "anomaly"

    print("=" * 72)
    print(f"Kalman CV: {scenario_name}")
    started = time.perf_counter()
    def train_model() -> dict[str, object]:
        """Обучение, выбор порога и оценки train для печати медиан."""
        sigma_acc, sigma_meas = fit_kalman_on_train(train)
        speed_limit = restart_speed_limit(train)
        selection = select_kalman_threshold(train, sigma_acc, sigma_meas, speed_limit, MCC_BAND)
        scores = kalman_detect(train, sigma_acc, sigma_meas, speed_limit, np.array([selection["tau"]]))[0]
        return {"sigma": (sigma_acc, sigma_meas), "speed_limit": speed_limit, "selection": selection, "train_scores": scores}
    state = load_or_train("kalman", train, train_model)
    (sigma_acc, sigma_meas), speed_limit = state["sigma"], state["speed_limit"]
    selection, train_scores = state["selection"], state["train_scores"]
    print(f"  σ_acc {sigma_acc:.4f} м/с², σ_meas {sigma_meas:.2f} м; предел скорости перезапуска {speed_limit:.2f} м/с")
    if np.any(~anomaly_train) and np.any(anomaly_train):
        print(
            f"  медиана NIS на train: move {np.nanmedian(train_scores[0][~anomaly_train]):.4g}, "
            f"anomaly {np.nanmedian(train_scores[0][anomaly_train]):.4g}"
        )
    print(
        f"  χ²₀.₉₉₉(df=2) = {KALMAN_CHI999:.2f}; порог NIS {selection['tau']:.4g}: "
        f"MCC train {selection['train_mcc']:.4f} "
        f"(лучший {selection['best_mcc']:.4f}, в полосе {int(selection['band_size'])}), "
        f"DL train {selection['train_dl_m'] / 1000:.4f} км"
    )

    test_scores, test_rejected = kalman_detect(test, sigma_acc, sigma_meas, speed_limit, np.array([selection["tau"]]))
    predicted_test = test_rejected[0]
    scored_test = ~np.isnan(test_scores[0])
    metrics = evaluate_test(test, predicted_test, scored_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м, без оценки {metrics['unscored_share']:.2%} "
        f"(время расчёта {elapsed:.1f} с)"
    )

    kalman_rows.append(test_report_row(scenario_name, metrics))
    kalman_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "scored_test": scored_test,
            "metrics": metrics,
        }
    )

kalman_summary = pd.DataFrame(kalman_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    kalman_summary.loc[:, report_columns],
    caption="Kalman CV: сводные метрики на test",
)

if SHOW_PLOTS:
    _kalman_ideal_test = kalman_scenario_results[0]["test"]
    _kalman_map_tail = min(PLOT_TAIL, len(_kalman_ideal_test))
    print(f"\nИдеальная разметка: последние {nfmt(_kalman_map_tail)} точек теста (bbox: move + 5%)")
    _kalman_dummy_move = np.zeros(len(_kalman_ideal_test), dtype=bool)
    _kalman_truth_pred, _kalman_truth_gt = trajectory_series_pair(_kalman_ideal_test, _kalman_dummy_move)
    _kalman_truth_pred = slice_trajectory_series(_kalman_truth_pred, _kalman_map_tail)
    _kalman_truth_gt = slice_trajectory_series(_kalman_truth_gt, _kalman_map_tail)
    plot_track_maps(
        _kalman_truth_pred,
        _kalman_truth_gt,
        map_panel="ground_truth",
        title="Kalman CV — идеальная разметка",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )

for item in kalman_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    if SHOW_PLOTS:
        print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
        prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"], item["scored_test"])
        prediction = slice_trajectory_series(prediction, tail)
        ground_truth = slice_trajectory_series(ground_truth, tail)
        plot_track_maps(
            prediction,
            ground_truth,
            map_panel="model",
            title=f"Kalman CV — {item['name']}",
            figsize=MAP_FIGSIZE,
            annotate=False,
        )

## Гауссовский процесс

### Постановка

Гауссовский процесс по окну последних принятых точек прогнозирует положение следующей точки. Аномалия — точка, плохо согласованная с прогнозом.

### Модель и оценка аномальности

Для восточной координаты (аналогично для северной) на окне из последних 18 принятых точек отрезка:

$$x(t)=\beta_0+\beta_1(t-\bar t)+f(t),\qquad f\sim\mathcal{GP}\bigl(0,k(t,t')\bigr)$$

$$k(t,t')=\sigma_{\mathrm{rbf}}^2\exp\Bigl(-\frac{(t-t')^2}{2\ell^2}\Bigr)+\sigma_{\mathrm n}^2\,\delta_{tt'},$$

где $\bar t$ — среднее время точек окна. Линейный тренд описывает движение с постоянной скоростью; $\beta_0$, $\beta_1$ оцениваются обобщённым МНК по окну без априорного распределения. RBF описывает плавный манёвр, $\sigma_{\mathrm n}^2$ — шум GPS.

Прогноз в момент $t_i$ — апостериорные среднее $\mu_i$ и дисперсия $\sigma_i^2$, включающая неопределённость оценки тренда. Оценка аномальности — квадрат нормированной ошибки по двум осям:

$$s_i=\sum_{\text{оси}}\frac{(z_i-\mu_i)^2}{\sigma_i^2+\sigma_{\mathrm n}^2},\qquad \hat a_i=\mathbf{1}[s_i>\tau].$$

Отвергнутая точка в окно не входит. Свойства модели: аномальная точка, принятая в окно, искажает тренд, пока не выйдет из окна; после паузы, близкой к разрыву по времени, прогноз продолжает тренд окна без учёта изменения скорости за паузу. В обоих случаях следующие точки могут отвергаться до перезапуска прогноза.

### Время и точки без оценки

Как у AR(2): прогноз начинается заново после разрыва по времени больше 180 с и после 180 с подряд отвергнутых точек, если точка достижима от последней принятой; из точек с одинаковым временем остаётся первая. Первые три точки отрезка — без оценки; пока в окне меньше 18 точек, прогноз строится по имеющимся.

### Обучение и выбор порога

1. Гиперпараметры $\sigma_{\mathrm{rbf}}$, $\ell$, $\sigma_{\mathrm n}$ — общие для обеих осей; оцениваются по максимуму суммы логарифмов прогнозной плотности следующей точки на окнах обучающих отрезков, как у MPCT.
2. Оценка аномальности и подбор порога — как у AR(2); в сетку кандидатов добавляется $\chi^2_{0.999}(2)$.

In [ ]:
GP_WINDOW = 18
GP_MIN_POINTS = 3
GP_FIT_TARGETS = 4_000
# Начальные значения и нижние границы гиперпараметров: sigma_rbf, м; ell, с; sigma_noise, м.
GP_HYPER_START = (20.0, 60.0, 5.0)
GP_HYPER_MIN = (1e-2, 1.0, 0.5)
GP_CHI999 = float(chi2.ppf(0.999, 2))


def gp_window_prediction(
    window_time: np.ndarray,
    window_xy: np.ndarray,
    valid: np.ndarray,
    query_time: np.ndarray,
    hyper: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Прогноз по окнам, пакетно: линейный тренд по обобщённому МНК плюс гауссовский процесс (RBF и шум).

    Универсальный кригинг: коэффициенты тренда β₀ + β₁(t - t̄) без априорного распределения, их
    неопределённость входит в дисперсию прогноза. Недействительные элементы окна отделяются единичной
    диагональю и в прогноз не входят.

    Args:
        window_time: Время точек окон, с, shape (B, W).
        window_xy: Координаты ENU точек окон, м, shape (B, W, 2).
        valid: Действительные элементы окон, shape (B, W).
        query_time: Время прогнозируемой точки, с, shape (B,).
        hyper: (sigma_rbf, ell, sigma_noise).

    Returns:
        tuple[np.ndarray, np.ndarray]: Среднее shape (B, 2), м, и дисперсия прогноза без шума shape (B,), м².
    """
    sigma_rbf, ell, sigma_noise = hyper
    weight = valid.astype(float)
    mean_time = (window_time * weight).sum(axis=1) / np.maximum(weight.sum(axis=1), 1.0)
    trend = np.stack([weight, (window_time - mean_time[:, None]) * weight], axis=2)
    gap = window_time[:, :, None] - window_time[:, None, :]
    gram = sigma_rbf**2 * np.exp(-0.5 * (gap / ell) ** 2) * weight[:, :, None] * weight[:, None, :]
    gram += np.eye(window_time.shape[1]) * np.where(valid, sigma_noise**2, 1.0)[:, :, None]
    cross = sigma_rbf**2 * np.exp(-0.5 * ((window_time - query_time[:, None]) / ell) ** 2) * weight
    query_trend = np.stack([np.ones_like(query_time), query_time - mean_time], axis=1)
    target = window_xy * weight[:, :, None]
    solved = np.linalg.solve(gram, np.concatenate([target, cross[:, :, None], trend], axis=2))
    solved_target, solved_cross, solved_trend = solved[:, :, :2], solved[:, :, 2], solved[:, :, 3:]
    normal = np.einsum("bwi,bwj->bij", trend, solved_trend)
    beta = np.linalg.solve(normal, np.einsum("bwi,bwk->bik", trend, solved_target))
    residual_weights = solved_target - np.einsum("bwi,bik->bwk", solved_trend, beta)
    mean = np.einsum("bi,bik->bk", query_trend, beta) + np.einsum("bw,bwk->bk", cross, residual_weights)
    lever = query_trend - np.einsum("bwi,bw->bi", trend, solved_cross)
    trend_variance = np.einsum("bi,bi->b", lever, np.linalg.solve(normal, lever[:, :, None])[:, :, 0])
    variance = sigma_rbf**2 - np.einsum("bw,bw->b", cross, solved_cross) + trend_variance
    return mean, np.maximum(variance, 0.0)


def gp_fit_targets(train: pd.DataFrame) -> dict[str, np.ndarray]:
    """Окна штатного хода обучающей выборки и следующие за ними точки для оценки гиперпараметров.

    Отрезки — обучающие отрезки штатного хода (move_segments); цель — точка не раньше четвёртой в отрезке,
    окно — до GP_WINDOW предыдущих точек отрезка.

    Args:
        train: Обучающий DataFrame с колонками x, y, time и l.

    Returns:
        dict[str, np.ndarray]: window_time, window_xy, valid, query_time, query_xy.
    """
    xy = enu_xy_from_frame(train)
    time_sec = time_seconds_from_frame(train)
    windows: list[np.ndarray] = []
    queries: list[int] = []
    for segment in move_segments(train):
        for position in range(GP_MIN_POINTS, len(segment)):
            window = segment[max(0, position - GP_WINDOW) : position]
            windows.append(np.pad(window, (GP_WINDOW - len(window), 0), constant_values=-1))
            queries.append(int(segment[position]))
    if not queries:
        raise RuntimeError("Нет окон штатного хода для оценки гауссовского процесса.")
    picked = np.random.default_rng(SEED).permutation(len(queries))[:GP_FIT_TARGETS]
    index = np.asarray(windows)[picked]
    valid = index >= 0
    safe = np.where(valid, index, 0)
    return {
        "window_time": time_sec[safe],
        "window_xy": xy[safe],
        "valid": valid,
        "query_time": time_sec[np.asarray(queries)[picked]],
        "query_xy": xy[np.asarray(queries)[picked]],
    }


def gp_log_predictive(targets: dict[str, np.ndarray], hyper: np.ndarray) -> float:
    """Сумма логарифмов прогнозной плотности следующих точек по двум осям.

    Args:
        targets: Результат gp_fit_targets.
        hyper: (sigma_rbf, ell, sigma_noise).

    Returns:
        float: Сумма логарифмов плотности.
    """
    mean, variance = gp_window_prediction(
        targets["window_time"], targets["window_xy"], targets["valid"], targets["query_time"], hyper
    )
    total_variance = variance + hyper[2] ** 2
    residual = targets["query_xy"] - mean
    return float(-0.5 * np.sum(np.log(2.0 * np.pi * total_variance)[:, None] + residual**2 / total_variance[:, None]))


def fit_gp_on_train(train: pd.DataFrame) -> np.ndarray:
    """Оценивает гиперпараметры по максимуму прогнозной плотности на окнах штатного хода обучающей выборки.

    Args:
        train: Обучающий DataFrame с колонками x, y, time и l.

    Returns:
        np.ndarray: (sigma_rbf, м; ell, с; sigma_noise, м).
    """
    targets = gp_fit_targets(train)
    lower = np.log(GP_HYPER_MIN)
    result = minimize(
        lambda log_hyper: -gp_log_predictive(targets, np.exp(log_hyper)),
        np.log(GP_HYPER_START),
        method="L-BFGS-B",
        bounds=[(value, None) for value in lower],
    )
    if np.any(np.isclose(result.x, lower)):
        print("  предупреждение: гиперпараметр гауссовского процесса на нижней границе")
    return np.exp(result.x)


def gp_detect(
    frame: pd.DataFrame, hyper: np.ndarray, speed_limit: float, taus: np.ndarray
) -> tuple[np.ndarray, np.ndarray]:
    """Скользящий гауссовский процесс с заменой отвергнутых точек, сразу для нескольких порогов.

    Отвергнутая точка в окно не входит. Прогноз начинается заново после разрыва по времени и после серии
    отвергнутых точек длиннее разрыва, если точка достижима от последней принятой (restart_allowed); первые
    GP_MIN_POINTS точек отрезка — без оценки. Из точек с одинаковым
    временем остаётся первая.

    Args:
        frame: DataFrame с колонками x, y и time.
        hyper: (sigma_rbf, ell, sigma_noise).
        speed_limit: Предел скорости перезапуска, м/с (restart_speed_limit).
        taus: Пороги; np.inf — без замены.

    Returns:
        tuple[np.ndarray, np.ndarray]: Оценка аномальности shape (C, N), NaN — точка без оценки;
        маска отвергнутых точек.
    """
    xy = enu_xy_from_frame(frame)
    time_sec = time_seconds_from_frame(frame)
    thresholds = np.atleast_1d(np.asarray(taus, dtype=float))
    n_cand, count = len(thresholds), len(xy)
    scores = np.full((n_cand, count), np.nan, dtype=np.float32)
    rejected = np.zeros((n_cand, count), dtype=bool)
    window_time = np.zeros((n_cand, GP_WINDOW))
    window_xy = np.zeros((n_cand, GP_WINDOW, 2))
    filled = np.zeros(n_cand, dtype=np.int64)
    run_start = np.full(n_cand, np.nan)
    # Последняя принятая точка: от неё проверяется достижимость точки перезапуска.
    last_xy = np.zeros((n_cand, 2))
    last_time = np.zeros(n_cand)
    positions = np.arange(GP_WINDOW)
    prev_time = None
    for index in range(count):
        moment = time_sec[index]
        if prev_time is not None and moment - prev_time <= 0.0:
            continue
        if prev_time is None or moment - prev_time > TIME_GAP_SEC:
            restart = np.ones(n_cand, dtype=bool)
        else:
            restart = restart_allowed(xy[index], moment, last_xy, last_time, run_start, speed_limit)
        filled[restart] = 0
        run_start[restart] = np.nan
        active = filled >= GP_MIN_POINTS
        valid = positions[None, :] >= GP_WINDOW - filled[:, None]
        nis = np.full(n_cand, np.nan)
        # Тренд по окну определён не меньше чем по двум точкам: прогноз — только для окон с оценкой.
        if np.any(active):
            mean, variance = gp_window_prediction(
                window_time[active], window_xy[active], valid[active], np.full(int(active.sum()), moment), hyper
            )
            nis[active] = np.sum((xy[index] - mean) ** 2, axis=1) / (variance + hyper[2] ** 2)
        reject = active & (nis > thresholds)
        scores[:, index] = np.where(active, nis, np.nan)
        rejected[:, index] = reject
        accept = ~reject
        window_time[accept] = np.roll(window_time[accept], -1, axis=1)
        window_xy[accept] = np.roll(window_xy[accept], -1, axis=1)
        window_time[accept, -1] = moment
        window_xy[accept, -1] = xy[index]
        filled[accept] = np.minimum(filled[accept] + 1, GP_WINDOW)
        last_xy[accept] = xy[index]
        last_time[accept] = moment
        run_start = np.where(
            reject, np.where(np.isnan(run_start), moment, run_start), np.where(active, np.nan, run_start)
        )
        prev_time = moment
    return scores, rejected


def select_gp_threshold(
    frame: pd.DataFrame, hyper: np.ndarray, speed_limit: float, mcc_band: float
) -> dict[str, float]:
    """Подбирает порог: общая сетка по оценке без замены и χ²₀.₉₉₉(2), прогон с заменой на каждого кандидата.

    Args:
        frame: Обучающий DataFrame (time, lat, lon, x, y, l).
        hyper: (sigma_rbf, ell, sigma_noise).
        speed_limit: Предел скорости перезапуска, м/с.
        mcc_band: Допуск по MCC относительно лучшего значения.

    Returns:
        dict[str, float]: tau, train_mcc, train_f1, train_dl_m, best_mcc, band_size.
    """
    base_scores, _ = gp_detect(frame, hyper, speed_limit, np.array([np.inf]))
    finite = base_scores[0][np.isfinite(base_scores[0])]
    candidates = np.unique(np.r_[threshold_candidates(finite), GP_CHI999])
    scores, rejected = gp_detect(frame, hyper, speed_limit, candidates)
    ground_truth_anomaly = frame["l"].astype(str).to_numpy() == "anomaly"
    records = [
        _threshold_record(float(tau), rejected[row], frame, ground_truth_anomaly, ~np.isnan(scores[row]))
        for row, tau in enumerate(candidates)
    ]
    return select_threshold_from_records(records, mcc_band)


gp_scenarios = (
    (DATASET_NAME, dataset),
)

gp_rows: list[dict[str, object]] = []
gp_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in gp_scenarios:
    train = bundle["train"]
    test = bundle["test"]
    anomaly_train = train["l"].astype(str).to_numpy() == "anomaly"

    print("=" * 72)
    print(f"Gaussian process: {scenario_name}")
    started = time.perf_counter()
    def train_model() -> dict[str, object]:
        """Обучение, выбор порога и оценки train для печати медиан."""
        gp_hyper = fit_gp_on_train(train)
        speed_limit = restart_speed_limit(train)
        selection = select_gp_threshold(train, gp_hyper, speed_limit, MCC_BAND)
        scores = gp_detect(train, gp_hyper, speed_limit, np.array([selection["tau"]]))[0]
        return {"hyper": gp_hyper, "speed_limit": speed_limit, "selection": selection, "train_scores": scores}
    state = load_or_train("gp", train, train_model)
    gp_hyper, speed_limit = state["hyper"], state["speed_limit"]
    selection, train_scores = state["selection"], state["train_scores"]
    print(
        f"  σ_rbf {gp_hyper[0]:.2f} м, ℓ {gp_hyper[1]:.1f} с, σ_noise {gp_hyper[2]:.2f} м; "
        f"предел скорости перезапуска {speed_limit:.2f} м/с"
    )
    if np.any(~anomaly_train) and np.any(anomaly_train):
        print(
            f"  медиана score на train: move {np.nanmedian(train_scores[0][~anomaly_train]):.4g}, "
            f"anomaly {np.nanmedian(train_scores[0][anomaly_train]):.4g}"
        )
    print(
        f"  χ²₀.₉₉₉(df=2)={GP_CHI999:.2f}; порог {selection['tau']:.4g}: "
        f"MCC train {selection['train_mcc']:.4f} "
        f"(лучший {selection['best_mcc']:.4f}, в полосе {int(selection['band_size'])}), "
        f"DL train {selection['train_dl_m'] / 1000:.4f} км"
    )

    test_scores, test_rejected = gp_detect(test, gp_hyper, speed_limit, np.array([selection["tau"]]))
    predicted_test = test_rejected[0]
    scored_test = ~np.isnan(test_scores[0])
    metrics = evaluate_test(test, predicted_test, scored_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м, без оценки {metrics['unscored_share']:.2%} "
        f"(время расчёта {elapsed:.1f} с)"
    )

    gp_rows.append(test_report_row(scenario_name, metrics))
    gp_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "scored_test": scored_test,
            "metrics": metrics,
        }
    )

gp_summary = pd.DataFrame(gp_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    gp_summary.loc[:, report_columns],
    caption="Gaussian process: сводные метрики на test",
)

if SHOW_PLOTS:
    _gp_ideal_test = gp_scenario_results[0]["test"]
    _gp_map_tail = min(PLOT_TAIL, len(_gp_ideal_test))
    print(f"\nИдеальная разметка: последние {nfmt(_gp_map_tail)} точек теста (bbox: move + 5%)")
    _gp_dummy_move = np.zeros(len(_gp_ideal_test), dtype=bool)
    _gp_truth_pred, _gp_truth_gt = trajectory_series_pair(_gp_ideal_test, _gp_dummy_move)
    _gp_truth_pred = slice_trajectory_series(_gp_truth_pred, _gp_map_tail)
    _gp_truth_gt = slice_trajectory_series(_gp_truth_gt, _gp_map_tail)
    plot_track_maps(
        _gp_truth_pred,
        _gp_truth_gt,
        map_panel="ground_truth",
        title="GP — идеальная разметка",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )

for item in gp_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    if SHOW_PLOTS:
        print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
        prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"], item["scored_test"])
        prediction = slice_trajectory_series(prediction, tail)
        ground_truth = slice_trajectory_series(ground_truth, tail)
        plot_track_maps(
            prediction,
            ground_truth,
            map_panel="model",
            title=f"GP — {item['name']}",
            figsize=MAP_FIGSIZE,
            annotate=False,
        )

## GP-EKF (фильтр Калмана с гауссовским процессом)

Как у фильтра Калмана (постоянная скорость).

### Постановка

Модель постоянной скорости дополняется ускорением-поправкой, которая зависит от скорости и обучается гауссовским процессом по штатному ходу. Аномалия — точка, плохо согласованная с причинным прогнозом фильтра.

### Модель и оценка аномальности

$$\mathbf x_i=F_i\mathbf x_{i-1}+B_i\,\mathbf g(\mathbf v_{i-1})+\mathbf w_i,\qquad B_i=\begin{pmatrix}\tfrac{\Delta t_i^2}{2}I\\\Delta t_i I\end{pmatrix},\qquad \mathbf v_i=(v^x_i,v^y_i)^\top,$$

где $\mathbf g=(g^x,g^y)^\top$ — ускорение-поправка, м/с². По каждой оси — независимый гауссовский процесс от скорости; для восточной оси (аналогично для северной):

$$g^x\sim\mathcal{GP}\bigl(0,k^x(\mathbf v,\mathbf v')\bigr),\qquad k^x(\mathbf v,\mathbf v')=\sigma_{f,x}^2\exp\Bigl(-\frac12\sum_{m\in\{x,y\}}\frac{(v^m-v'^m)^2}{\ell_{x,m}^2}\Bigr),$$

где $\sigma_{f,x}$ — амплитуда, $\ell_{x,m}$ — своя длина корреляции по каждой составляющей скорости.

Прогноз — расширенный фильтр Калмана (extended Kalman filter, EKF):

$$\hat{\mathbf x}_{i\mid i-1}=F_i\hat{\mathbf x}_{i-1}+B_i\boldsymbol\mu_i,\qquad G_i=F_i+B_i\,(0\;\;J_i),\qquad P_{i\mid i-1}=G_iP_{i-1}G_i^\top+B_i\Sigma_iB_i^\top,$$

где $\boldsymbol\mu_i$ и $\sigma_{g,x}^2$, $\sigma_{g,y}^2$ — апостериорные среднее и дисперсии гауссовских процессов при скорости $\hat{\mathbf v}_{i-1}$, $J_i=\partial\boldsymbol\mu/\partial\mathbf v$ — якобиан среднего, $\Sigma_i=\mathrm{diag}\bigl(\sigma_{g,x}^2+\sigma_{\mathrm{acc},x}^2,\ \sigma_{g,y}^2+\sigma_{\mathrm{acc},y}^2\bigr)$, $\sigma_{\mathrm{acc},x}$, $\sigma_{\mathrm{acc},y}$ — шум ускорения.

Шум процесса растёт с $\Delta t_i$, как у фильтра Калмана, и больше при скоростях, где мало опорных точек. При $\mathbf g\equiv0$ и $\sigma_{\mathrm{acc},x}=\sigma_{\mathrm{acc},y}$ модель совпадает с фильтром Калмана; вне диапазона скоростей опорных точек $\boldsymbol\mu_i\to0$.

Свойство модели: сглаживатель притягивает обучающие переходы к модели постоянной скорости, поэтому поправка занижена; невоспроизведённая часть ускорения входит в шум ускорения $\sigma_{\mathrm{acc},x}$, $\sigma_{\mathrm{acc},y}$.

Оценка аномальности, решение и обновление состояния — как у фильтра Калмана.

### Обучение и выбор порога

1. Опорные точки. Сглаживатель Рауха — Тунга — Штрибеля (RTS smoother) с моделью и оценками $\sigma_{\mathrm{acc}}$, $\sigma_{\mathrm{meas}}$ фильтра Калмана восстанавливает состояния $\tilde{\mathbf x}_j$ на отрезках штатного хода обучающей выборки. Опорная точка — переход: вход $\tilde{\mathbf v}_{j-1}$, цель $(\tilde{\mathbf v}_j-\tilde{\mathbf v}_{j-1})/\Delta t_j$. Цель по сырым разностям координат определялась бы шумом GPS. Опорные точки разнесены по времени не меньше чем на 120 с, из них берётся до 1000 случайных: ошибки сглаживания соседних переходов коррелированы и принимались бы за зависимость от скорости.
2. $\sigma_f$ и $\ell$ — по максимуму маргинального правдоподобия (marginal likelihood) на опорных точках; шум цели оценивается вместе с ними и в $\Sigma_i$ не входит.
3. $\sigma_{\mathrm{acc},x}$, $\sigma_{\mathrm{acc},y}$, $\sigma_{\mathrm{meas}}$ — по максимуму правдоподобия фильтра при обученных гауссовских процессах, как $\sigma_{\mathrm{acc}}$ и $\sigma_{\mathrm{meas}}$ у фильтра Калмана, с теми же нижними границами. Сглаживатель убирает часть шума процесса, поэтому оценка шума ускорения по опорным точкам занижена.
4. Оценка аномальности и подбор порога — как у фильтра Калмана.

При прогоне выводится доля дисперсии ускорения штатного хода, объяснённая средним $\boldsymbol\mu$, на обучающей и тестовой выборках — мера отличия модели от фильтра Калмана.

In [ ]:
GP_EKF_SUPPORT = 1000
# Ошибки ускорения сглаженных переходов коррелированы во времени: опорные точки разносятся по времени.
GP_EKF_SUPPORT_GAP_SEC = 120.0
GP_EKF_SEED = SEED
# Сетка скоростей для апостериорных величин GP: шаг — доля наименьшей длины корреляции, поле по каждой составляющей
# скорости — с запасом в несколько наибольших её длин (не больше разброса скоростей), где GP возвращается
# к априорному распределению. Длина на верхней границе — зависимости нет, в шаг и запас не входит.
GP_EKF_GRID_STEP_FRACTION = 0.25
GP_EKF_GRID_MARGIN = 4.0
GP_EKF_GRID_MAX_NODES = 301
GP_EKF_GRID_CHUNK = 8_000
# Границы гиперпараметров GP: длина корреляции, м/с; амплитуда и шум цели, м/с².
GP_EKF_LENGTH_BOUNDS = (1e-2, 1e2)
GP_EKF_SIGMA_BOUNDS = (1e-6, 1e1)
# Верхняя граница СКО измерения при оценке шума фильтра, м: без неё оптимизатор уходит в переполнение exp.
GP_EKF_SIGMA_MEAS_MAX = 1e3
GP_EKF_FD_STEP = 1e-4


def _gp_ekf_segments(frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """Точки обучающих отрезков штатного хода (move_segments) не короче трёх точек.

    Args:
        frame: DataFrame с колонками x, y, time и l.

    Returns:
        tuple[np.ndarray, np.ndarray]: индексы точек отрезков подряд; маска начала отрезка.
    """
    segments = [segment for segment in move_segments(frame) if len(segment) >= 3]
    if not segments:
        raise RuntimeError("Нет отрезка штатного хода из трёх точек для GP-EKF.")
    starts = np.zeros(sum(len(segment) for segment in segments), dtype=bool)
    starts[np.r_[0, np.cumsum([len(segment) for segment in segments])[:-1]]] = True
    return np.concatenate(segments), starts


def gp_ekf_smoothed_transitions(
    frame: pd.DataFrame, sigma_acc: float, sigma_meas: float
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Переходы сглаженного состояния (сглаживатель RTS модели постоянной скорости) на штатном ходу.

    Args:
        frame: DataFrame с колонками x, y, time и l.
        sigma_acc: СКО ускорения фильтра Калмана, м/с².
        sigma_meas: СКО измерения фильтра Калмана, м.

    Returns:
        tuple[np.ndarray, np.ndarray]: вход — сглаженная скорость перед переходом (n, 2), м/с;
        цель — приращение сглаженной скорости, делённое на шаг (n, 2), м/с²; время начала перехода (n,), с.
    """
    index, starts = _gp_ekf_segments(frame)
    time_sec = time_seconds_from_frame(frame)[index]
    xy = enu_xy_from_frame(frame)[index]
    boundary = np.r_[starts[1:], True]
    step = np.r_[np.diff(time_sec), 0.0]
    transition = np.zeros((2, 2, len(step)))
    transition[0, 0] = transition[1, 1] = 1.0
    transition[0, 1] = step
    noise = kalman_process_noise(step, sigma_acc)
    transition[:, :, boundary] = 0.0
    noise[:, :, boundary] = np.eye(2)[:, :, None] * _DIFFUSE_VAR_M2
    velocity = np.zeros((len(step), 2))
    for column in range(2):
        smoother = KalmanSmoother(k_endog=1, k_states=2, k_posdef=2)
        smoother.bind(xy[:, column] - xy[0, column])
        smoother["design"] = np.array([[1.0, 0.0]])
        smoother["obs_cov"] = np.array([[sigma_meas**2]])
        smoother["transition"] = transition
        smoother["selection"] = np.eye(2)
        smoother["state_cov"] = noise
        smoother.initialize_approximate_diffuse(_DIFFUSE_VAR_M2)
        velocity[:, column] = smoother.smooth().smoothed_state[1]
    inner = ~boundary[:-1]
    inputs = velocity[:-1][inner]
    targets = (velocity[1:][inner] - inputs) / step[:-1][inner, None]
    return inputs, targets, time_sec[:-1][inner]


def _gp_ekf_kernel(left: np.ndarray, right: np.ndarray, amplitude: float, lengths: np.ndarray) -> np.ndarray:
    """Ядро RBF со своей длиной корреляции по каждой составляющей скорости."""
    scaled = (left[:, None, :] - right[None, :, :]) / lengths
    return amplitude**2 * np.exp(-0.5 * np.sum(scaled * scaled, axis=2))


def _gp_ekf_neg_log_marginal(log_params: np.ndarray, inputs: np.ndarray, target: np.ndarray) -> float:
    """Минус логарифм маргинального правдоподобия одной оси; параметры — log (σ_f, ℓ_o, ℓ_n, σ_y)."""
    amplitude, length_o, length_n, noise = np.exp(log_params)
    gram = _gp_ekf_kernel(inputs, inputs, amplitude, np.array([length_o, length_n]))
    gram[np.diag_indices_from(gram)] += noise**2
    try:
        chol = np.linalg.cholesky(gram)
    except LinAlgError:
        return 1e30
    alpha = solve_triangular(chol, target, lower=True, check_finite=False)
    return float(0.5 * alpha @ alpha + np.sum(np.log(np.diag(chol))) + 0.5 * len(target) * np.log(2.0 * np.pi))


def _gp_ekf_fit_axis(inputs: np.ndarray, target: np.ndarray) -> dict[str, object]:
    """Гиперпараметры GP одной оси по максимуму маргинального правдоподобия и разложение Холецкого."""
    bounds = np.log([GP_EKF_SIGMA_BOUNDS, GP_EKF_LENGTH_BOUNDS, GP_EKF_LENGTH_BOUNDS, GP_EKF_SIGMA_BOUNDS])
    spread = float(np.std(target))
    start = np.clip(np.log(np.r_[spread, np.std(inputs, axis=0), 0.5 * spread] + 1e-12), bounds[:, 0], bounds[:, 1])
    result = minimize(
        _gp_ekf_neg_log_marginal, start, args=(inputs, target), method="L-BFGS-B", bounds=[tuple(b) for b in bounds]
    )
    amplitude, length_o, length_n, noise = np.exp(result.x)
    lengths = np.array([length_o, length_n])
    gram = _gp_ekf_kernel(inputs, inputs, amplitude, lengths)
    gram[np.diag_indices_from(gram)] += noise**2
    chol = np.linalg.cholesky(gram)
    alpha = solve_triangular(chol.T, solve_triangular(chol, target, lower=True), lower=False)
    return {
        "amplitude": float(amplitude),
        "lengths": lengths,
        "target_noise": float(noise),
        "chol": chol,
        "alpha": alpha,
    }


def gp_ekf_posterior(axis_model: dict[str, object], inputs: np.ndarray, velocity: np.ndarray) -> tuple[np.ndarray, ...]:
    """Апостериорные среднее, дисперсия функции (без шума цели) и градиент среднего по скорости.

    Args:
        axis_model: GP одной оси из _gp_ekf_fit_axis.
        inputs: Скорости опорных точек (n, 2), м/с.
        velocity: Скорости, в которых считается прогноз (m, 2), м/с.

    Returns:
        tuple[np.ndarray, ...]: среднее (m,), м/с²; дисперсия (m,), (м/с²)²; градиент (m, 2), 1/с.
    """
    lengths = axis_model["lengths"]
    cross = _gp_ekf_kernel(velocity, inputs, axis_model["amplitude"], lengths)
    mean = cross @ axis_model["alpha"]
    solved = solve_triangular(axis_model["chol"], cross.T, lower=True, check_finite=False)
    variance = np.maximum(axis_model["amplitude"] ** 2 - np.sum(solved * solved, axis=0), 0.0)
    weighted = cross * axis_model["alpha"]
    diff = (velocity[:, None, :] - inputs[None, :, :]) / lengths**2
    gradient = -np.einsum("mk,mkd->md", weighted, diff)
    return mean, variance, gradient


def gp_ekf_support_index(start_sec: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Опорные переходы: через не меньше GP_EKF_SUPPORT_GAP_SEC со случайным началом, затем до GP_EKF_SUPPORT случайных.

    Args:
        start_sec: Время начала переходов по возрастанию, с.
        rng: Генератор случайных чисел.

    Returns:
        np.ndarray: Индексы выбранных переходов по возрастанию.
    """
    chosen = []
    next_time = start_sec[0] + rng.uniform(0.0, GP_EKF_SUPPORT_GAP_SEC)
    for index, moment in enumerate(start_sec):
        if moment >= next_time:
            chosen.append(index)
            next_time = moment + GP_EKF_SUPPORT_GAP_SEC
    chosen = np.asarray(chosen, dtype=np.int64)
    if len(chosen) > GP_EKF_SUPPORT:
        chosen = np.sort(rng.choice(chosen, GP_EKF_SUPPORT, replace=False))
    return chosen


def fit_gp_ekf_correction(inputs: np.ndarray, targets: np.ndarray, start_sec: np.ndarray) -> dict[str, object]:
    """Обучает ускорение-поправку по опорным точкам и табулирует её на сетке скоростей.

    Args:
        inputs: Сглаженные скорости перед переходом (n, 2), м/с.
        targets: Ускорения переходов (n, 2), м/с².
        start_sec: Время начала переходов по возрастанию, с.

    Returns:
        dict[str, object]: опорные точки, GP по осям, узлы сетки скоростей и значения на них: mean (2, G_o, G_n),
        variance (2, G_o, G_n), gradient (2, 2, G_o, G_n) — производная оси по составляющей скорости.
    """
    chosen = gp_ekf_support_index(start_sec, np.random.default_rng(GP_EKF_SEED))
    inputs, targets = inputs[chosen], targets[chosen]
    axes = [_gp_ekf_fit_axis(inputs, targets[:, axis]) for axis in range(2)]
    upper = GP_EKF_LENGTH_BOUNDS[1] * (1.0 - 1e-6)
    lengths = np.array([model["lengths"] for model in axes])
    free = lengths < upper
    step = GP_EKF_GRID_STEP_FRACTION * float(lengths[free].min()) if np.any(free) else np.inf
    nodes = []
    for column in range(2):
        scales = lengths[free[:, column], column]
        spread = float(np.ptp(inputs[:, column]))
        margin = GP_EKF_GRID_MARGIN * min(float(scales.max()), spread) if len(scales) else 0.0
        low = inputs[:, column].min() - margin
        high = inputs[:, column].max() + margin
        count = int(np.ceil((high - low) / step)) + 1 if np.isfinite(step) and high > low else 2
        if count > GP_EKF_GRID_MAX_NODES:
            print(f"  предупреждение: сетка скоростей GP-EKF ограничена {GP_EKF_GRID_MAX_NODES} узлами по оси")
            count = GP_EKF_GRID_MAX_NODES
        nodes.append(np.linspace(low, high, count))
    grid_o, grid_n = np.meshgrid(nodes[0], nodes[1], indexing="ij")
    flat = np.column_stack([grid_o.ravel(), grid_n.ravel()])
    shape = grid_o.shape
    mean = np.zeros((2, len(flat)))
    variance = np.zeros((2, len(flat)))
    gradient = np.zeros((2, 2, len(flat)))
    for start in range(0, len(flat), GP_EKF_GRID_CHUNK):
        part = slice(start, start + GP_EKF_GRID_CHUNK)
        for axis, axis_model in enumerate(axes):
            mean[axis, part], variance[axis, part], grad = gp_ekf_posterior(axis_model, inputs, flat[part])
            gradient[axis, :, part] = grad.T
    return {
        "inputs": inputs,
        "axes": axes,
        "nodes": nodes,
        "mean": mean.reshape(2, *shape),
        "variance": variance.reshape(2, *shape),
        "gradient": gradient.reshape(2, 2, *shape),
    }


def gp_ekf_lookup(correction: dict[str, object], velocity: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Билинейная интерполяция апостериорных величин GP по сетке скоростей; вне сетки — крайние узлы.

    Args:
        correction: Результат fit_gp_ekf_correction.
        velocity: Скорости (C, 2), м/с.

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray]: среднее (C, 2), дисперсия (C, 2), якобиан (C, 2, 2).
    """
    cells, weights = [], []
    for column, nodes in enumerate(correction["nodes"]):
        position = np.clip((velocity[:, column] - nodes[0]) / (nodes[1] - nodes[0]), 0.0, len(nodes) - 1.0)
        cell = np.minimum(position.astype(np.int64), len(nodes) - 2)
        cells.append(cell)
        weights.append(position - cell)
    (cell_o, cell_n), (weight_o, weight_n) = cells, weights

    def interpolate(table: np.ndarray) -> np.ndarray:
        return (
            table[..., cell_o, cell_n] * (1 - weight_o) * (1 - weight_n)
            + table[..., cell_o + 1, cell_n] * weight_o * (1 - weight_n)
            + table[..., cell_o, cell_n + 1] * (1 - weight_o) * weight_n
            + table[..., cell_o + 1, cell_n + 1] * weight_o * weight_n
        )

    mean = interpolate(correction["mean"]).T
    variance = interpolate(correction["variance"]).T
    jacobian = np.moveaxis(interpolate(correction["gradient"]), -1, 0)
    return mean, variance, jacobian


def _gp_ekf_filter(
    xy: np.ndarray,
    time_sec: np.ndarray,
    starts: np.ndarray,
    correction: dict[str, object],
    sigma_acc: np.ndarray,
    sigma_meas: np.ndarray,
    speed_limit: float,
    taus: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """EKF с ускорением-поправкой и заменой отвергнутых точек для нескольких конфигураций сразу.

    Прогноз начинается заново в начале отрезка и после серии отвергнутых точек длиннее разрыва по времени,
    если точка достижима от последней принятой (restart_allowed);
    начальное состояние — по первым двум точкам отрезка, они без оценки. Точки с тем же временем, что
    у предыдущей, пропускаются (без оценки).

    Args:
        xy: Координаты ENU (N, 2), м.
        time_sec: Время точек, с.
        starts: Маска начала отрезка.
        correction: Результат fit_gp_ekf_correction.
        sigma_acc: Шум ускорения по осям (C, 2), м/с².
        sigma_meas: СКО измерения (C,), м.
        speed_limit: Предел скорости перезапуска, м/с (restart_speed_limit).
        taus: Пороги NIS (C,); np.inf — без замены.

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray]: NIS (C, N), NaN — точка без оценки; маска отвергнутых
        точек (C, N); сумма логарифмов плотности инновации по точкам с оценкой (C,).
    """
    n_cand, count = len(taus), len(xy)
    meas_cov = np.eye(2)[None] * (sigma_meas**2)[:, None, None]
    acc_var = sigma_acc**2
    scores = np.full((n_cand, count), np.nan)
    rejected = np.zeros((n_cand, count), dtype=bool)
    loglike = np.zeros(n_cand)
    state = np.zeros((n_cand, 4))
    cov = np.zeros((n_cand, 4, 4))
    first_point = np.zeros((n_cand, 2))
    first_time = np.zeros(n_cand)
    phase = np.zeros(n_cand, dtype=np.int64)
    run_start = np.full(n_cand, np.nan)
    # Последняя принятая точка: от неё проверяется достижимость точки перезапуска.
    last_xy = np.zeros((n_cand, 2))
    last_time = np.zeros(n_cand)
    transition = np.eye(4)
    prev_time = None
    for index in range(count):
        moment = time_sec[index]
        if prev_time is not None and moment - prev_time <= 0.0:
            continue
        point = xy[index]
        if prev_time is None or starts[index]:
            restart = np.ones(n_cand, dtype=bool)
        else:
            restart = restart_allowed(point, moment, last_xy, last_time, run_start, speed_limit)
            dt = moment - prev_time
            transition[0, 2] = transition[1, 3] = dt
            mean, variance, jacobian = gp_ekf_lookup(correction, state[:, 2:])
            control = np.array([[dt**2 / 2.0, 0.0], [0.0, dt**2 / 2.0], [dt, 0.0], [0.0, dt]])
            linear = np.broadcast_to(transition, (n_cand, 4, 4)).copy()
            linear[:, :, 2:] += control @ jacobian
            state = state @ transition.T + mean @ control.T
            noise = control[None] * (variance + acc_var)[:, None, :]
            cov = linear @ cov @ linear.transpose(0, 2, 1) + noise @ control.T
        phase[restart] = 0
        run_start[restart] = np.nan
        start, second, active = phase == 0, phase == 1, phase >= 2
        first_point[start] = point
        first_time[start] = moment
        if np.any(second):
            pair_dt = moment - first_time[second]
            pair_cov = meas_cov[second]
            state[second, :2] = point
            state[second, 2:] = (point - first_point[second]) / pair_dt[:, None]
            cov[second] = 0.0
            cov[second, :2, :2] = pair_cov
            cov[second, :2, 2:] = cov[second, 2:, :2] = pair_cov / pair_dt[:, None, None]
            cov[second, 2:, 2:] = 2.0 * pair_cov / pair_dt[:, None, None] ** 2
        innovation = point - state[:, :2]
        innovation_cov = cov[:, :2, :2] + meas_cov
        solved = np.linalg.solve(innovation_cov, innovation[:, :, None])[:, :, 0]
        nis = np.einsum("ck,ck->c", innovation, solved)
        reject = active & (nis > taus)
        scores[:, index] = np.where(active, nis, np.nan)
        rejected[:, index] = reject
        if np.any(active):
            log_det = np.linalg.slogdet(innovation_cov[active])[1]
            loglike[active] -= 0.5 * (nis[active] + log_det + 2.0 * np.log(2.0 * np.pi))
        update = active & ~reject
        gain = cov[update][:, :, :2] @ np.linalg.inv(innovation_cov[update])
        state[update] += np.einsum("ckl,cl->ck", gain, innovation[update])
        cov[update] -= gain @ cov[update][:, :2, :]
        phase[start | second] += 1
        last_xy[~reject] = point
        last_time[~reject] = moment
        run_start = np.where(
            reject, np.where(np.isnan(run_start), moment, run_start), np.where(active, np.nan, run_start)
        )
        prev_time = moment
    return scores, rejected, loglike


def fit_gp_ekf_noise(
    train: pd.DataFrame, correction: dict[str, object], sigma_acc: float, sigma_meas: float
) -> tuple[np.ndarray, float]:
    """Шум ускорения по осям и СКО измерения по максимуму правдоподобия фильтра на штатном ходу.

    Правдоподобие условное: первые две точки отрезка задают начальное состояние и в него не входят.
    Градиент — конечными разностями по логарифмам параметров, все конфигурации — за один проход фильтра.

    Args:
        train: Обучающий DataFrame с колонками x, y, time и l.
        correction: Результат fit_gp_ekf_correction.
        sigma_acc: Начальное значение шума ускорения (оценка фильтра Калмана), м/с².
        sigma_meas: Начальное значение СКО измерения (оценка фильтра Калмана), м.

    Returns:
        tuple[np.ndarray, float]: шум ускорения по осям (2,), м/с²; СКО измерения, м.
    """
    index, starts = _gp_ekf_segments(train)
    xy = enu_xy_from_frame(train)[index]
    time_sec = time_seconds_from_frame(train)[index]
    taus = np.full(4, np.inf)

    def objective(log_sigma: np.ndarray) -> tuple[float, np.ndarray]:
        configs = np.vstack([log_sigma, log_sigma + GP_EKF_FD_STEP * np.eye(3)])
        sigma = np.exp(configs)
        _, _, loglike = _gp_ekf_filter(xy, time_sec, starts, correction, sigma[:, :2], sigma[:, 2], np.inf, taus)
        if not np.all(np.isfinite(loglike)):
            return _INADMISSIBLE_PENALTY, np.zeros(3)
        return -loglike[0], -(loglike[1:] - loglike[0]) / GP_EKF_FD_STEP

    lower = np.log([KALMAN_SIGMA_ACC_MIN, KALMAN_SIGMA_ACC_MIN, KALMAN_SIGMA_MEAS_MIN])
    upper = np.log([GP_EKF_SIGMA_BOUNDS[1], GP_EKF_SIGMA_BOUNDS[1], GP_EKF_SIGMA_MEAS_MAX])
    start = np.clip(np.log([sigma_acc, sigma_acc, sigma_meas]), lower, upper)
    result = minimize(objective, start, jac=True, method="L-BFGS-B", bounds=list(zip(lower, upper)))
    if np.any(np.isclose(result.x, lower)) or np.any(np.isclose(result.x, upper)):
        print("  предупреждение: оценка σ GP-EKF на границе")
    sigma = np.exp(result.x)
    return sigma[:2], float(sigma[2])


def gp_ekf_detect(frame: pd.DataFrame, model: dict[str, object], taus: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """NIS GP-EKF с заменой отвергнутых точек, сразу для нескольких порогов.

    Args:
        frame: DataFrame с колонками x, y и time.
        model: Словарь с correction, sigma_acc (2,), м/с², sigma_meas, м, и speed_limit, м/с.
        taus: Пороги NIS; np.inf — без замены.

    Returns:
        tuple[np.ndarray, np.ndarray]: NIS shape (C, N), NaN — точка без оценки; маска отвергнутых точек.
    """
    xy = enu_xy_from_frame(frame)
    time_sec = time_seconds_from_frame(frame)
    thresholds = np.atleast_1d(np.asarray(taus, dtype=float))
    starts = np.r_[True, np.diff(time_sec) > TIME_GAP_SEC]
    n_cand = len(thresholds)
    scores, rejected, _ = _gp_ekf_filter(
        xy,
        time_sec,
        starts,
        model["correction"],
        np.tile(model["sigma_acc"], (n_cand, 1)),
        np.full(n_cand, model["sigma_meas"]),
        model["speed_limit"],
        thresholds,
    )
    return scores, rejected


def select_gp_ekf_threshold(frame: pd.DataFrame, model: dict[str, object], mcc_band: float) -> dict[str, float]:
    """Подбирает порог NIS: сетка по NIS без замены и χ²₀.₉₉₉(2), прогон с заменой на каждого кандидата.

    Args:
        frame: Обучающий DataFrame (time, lat, lon, x, y, l).
        model: Словарь модели GP-EKF.
        mcc_band: Допуск по MCC относительно лучшего значения.

    Returns:
        dict[str, float]: tau, train_mcc, train_f1, train_dl_m, best_mcc, band_size.
    """
    base_scores, _ = gp_ekf_detect(frame, model, np.array([np.inf]))
    finite = base_scores[0][np.isfinite(base_scores[0])]
    candidates = np.unique(np.r_[threshold_candidates(finite), KALMAN_CHI999])
    scores, rejected = gp_ekf_detect(frame, model, candidates)
    ground_truth_anomaly = frame["l"].astype(str).to_numpy() == "anomaly"
    records = [
        _threshold_record(float(tau), rejected[row], frame, ground_truth_anomaly, ~np.isnan(scores[row]))
        for row, tau in enumerate(candidates)
    ]
    return select_threshold_from_records(records, mcc_band)


def gp_ekf_explained_share(frame: pd.DataFrame, model: dict[str, object]) -> float:
    """Доля дисперсии ускорения сглаженных переходов штатного хода, объяснённая средним GP.

    Args:
        frame: DataFrame с колонками x, y, time и l.
        model: Словарь модели GP-EKF с kalman_sigma — оценками фильтра Калмана для сглаживателя.

    Returns:
        float: 1 − Σ(a − μ)² / Σ(a − ā)² по обеим осям; отрицательное значение — поправка хуже константы.
    """
    inputs, targets, _ = gp_ekf_smoothed_transitions(frame, *model["kalman_sigma"])
    mean = gp_ekf_lookup(model["correction"], inputs)[0]
    residual = np.sum((targets - mean) ** 2)
    return float(1.0 - residual / np.sum((targets - targets.mean(axis=0)) ** 2))


def gp_ekf_target_autocorrelation(start_sec: np.ndarray, targets: np.ndarray, lag_sec: float) -> float:
    """Автокорреляция ускорения сглаженных переходов на лаге ``lag_sec`` (пары со сдвигом в пределах ±5 с).

    Args:
        start_sec: Время начала переходов по возрастанию, с.
        targets: Ускорения переходов (n, 2), м/с².
        lag_sec: Сдвиг по времени, с.

    Returns:
        float: Коэффициент корреляции по обеим осям; NaN — нет пар.
    """
    partner = np.searchsorted(start_sec, start_sec + lag_sec)
    valid = partner < len(start_sec)
    valid[valid] = np.abs(start_sec[partner[valid]] - start_sec[valid] - lag_sec) <= 5.0
    if not np.any(valid):
        return float("nan")
    centered = targets - targets.mean(axis=0)
    left, right = centered[valid], centered[partner[valid]]
    return float(np.sum(left * right) / np.sqrt(np.sum(left * left) * np.sum(right * right)))


def fit_gp_ekf_on_train(train: pd.DataFrame) -> dict[str, object]:
    """Обучает GP-EKF: фильтр Калмана → сглаженные переходы → GP по осям → шум по правдоподобию фильтра.

    Args:
        train: Обучающий DataFrame с колонками x, y, time и l.

    Returns:
        dict[str, object]: correction, sigma_acc (2,), м/с², sigma_meas, м, kalman_sigma, speed_limit, м/с.
    """
    kalman_sigma = fit_kalman_on_train(train)
    inputs, targets, start_sec = gp_ekf_smoothed_transitions(train, *kalman_sigma)
    correction = fit_gp_ekf_correction(inputs, targets, start_sec)
    sigma_acc, sigma_meas = fit_gp_ekf_noise(train, correction, *kalman_sigma)
    return {
        "correction": correction,
        "sigma_acc": sigma_acc,
        "sigma_meas": sigma_meas,
        "kalman_sigma": kalman_sigma,
        "speed_limit": restart_speed_limit(train),
        "target_acf": gp_ekf_target_autocorrelation(start_sec, targets, GP_EKF_SUPPORT_GAP_SEC),
    }


gp_ekf_scenarios = (
    (DATASET_NAME, dataset),
)

gp_ekf_rows: list[dict[str, object]] = []
gp_ekf_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in gp_ekf_scenarios:
    train = bundle["train"]
    test = bundle["test"]
    anomaly_train = train["l"].astype(str).to_numpy() == "anomaly"

    print("=" * 72)
    print(f"GP-EKF: {scenario_name}")
    started = time.perf_counter()
    def train_model() -> dict[str, object]:
        """Обучение, выбор порога и оценки train для печати медиан."""
        model = fit_gp_ekf_on_train(train)
        selection = select_gp_ekf_threshold(train, model, MCC_BAND)
        scores = gp_ekf_detect(train, model, np.array([selection["tau"]]))[0]
        return {"model": model, "selection": selection, "train_scores": scores}
    state = load_or_train("gp_ekf", train, train_model)
    gp_ekf_model, selection, train_scores = state["model"], state["selection"], state["train_scores"]
    correction = gp_ekf_model["correction"]
    print(
        f"  фильтр Калмана для сглаживателя: σ_acc {gp_ekf_model['kalman_sigma'][0]:.4f} м/с², "
        f"σ_meas {gp_ekf_model['kalman_sigma'][1]:.2f} м; опорных точек {len(correction['inputs'])}, "
        f"ACF ускорения на {GP_EKF_SUPPORT_GAP_SEC:.0f} с {gp_ekf_model['target_acf']:.2f}"
    )
    for axis_name, axis_model in zip(("x", "y"), correction["axes"]):
        print(
            f"  GP {axis_name}: σ_f {axis_model['amplitude']:.3g} м/с², "
            f"ℓ ({axis_model['lengths'][0]:.3g}, {axis_model['lengths'][1]:.3g}) м/с, "
            f"шум цели {axis_model['target_noise']:.3g} м/с²"
        )
    print(
        f"  σ_acc ({gp_ekf_model['sigma_acc'][0]:.4f}, {gp_ekf_model['sigma_acc'][1]:.4f}) м/с², "
        f"σ_meas {gp_ekf_model['sigma_meas']:.2f} м; доля дисперсии ускорения, объяснённая средним GP: "
        f"train {gp_ekf_explained_share(train, gp_ekf_model):.3f}, "
        f"test {gp_ekf_explained_share(test, gp_ekf_model):.3f}"
    )
    if np.any(~anomaly_train) and np.any(anomaly_train):
        print(
            f"  медиана NIS на train: move {np.nanmedian(train_scores[0][~anomaly_train]):.4g}, "
            f"anomaly {np.nanmedian(train_scores[0][anomaly_train]):.4g}"
        )
    print(
        f"  χ²₀.₉₉₉(df=2) = {KALMAN_CHI999:.2f}; порог NIS {selection['tau']:.4g}: "
        f"MCC train {selection['train_mcc']:.4f} "
        f"(лучший {selection['best_mcc']:.4f}, в полосе {int(selection['band_size'])}), "
        f"DL train {selection['train_dl_m'] / 1000:.4f} км"
    )

    test_scores, test_rejected = gp_ekf_detect(test, gp_ekf_model, np.array([selection["tau"]]))
    predicted_test = test_rejected[0]
    scored_test = ~np.isnan(test_scores[0])
    metrics = evaluate_test(test, predicted_test, scored_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м, без оценки {metrics['unscored_share']:.2%} "
        f"(время расчёта {elapsed:.1f} с)"
    )

    gp_ekf_rows.append(test_report_row(scenario_name, metrics))
    gp_ekf_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "scored_test": scored_test,
            "metrics": metrics,
        }
    )

gp_ekf_summary = pd.DataFrame(gp_ekf_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    gp_ekf_summary.loc[:, report_columns],
    caption="GP-EKF: сводные метрики на test",
)

if SHOW_PLOTS:
    _gp_ekf_ideal_test = gp_ekf_scenario_results[0]["test"]
    _gp_ekf_map_tail = min(PLOT_TAIL, len(_gp_ekf_ideal_test))
    print(f"\nИдеальная разметка: последние {nfmt(_gp_ekf_map_tail)} точек теста (bbox: move + 5%)")
    _gp_ekf_dummy_move = np.zeros(len(_gp_ekf_ideal_test), dtype=bool)
    _gp_ekf_truth_pred, _gp_ekf_truth_gt = trajectory_series_pair(_gp_ekf_ideal_test, _gp_ekf_dummy_move)
    _gp_ekf_truth_pred = slice_trajectory_series(_gp_ekf_truth_pred, _gp_ekf_map_tail)
    _gp_ekf_truth_gt = slice_trajectory_series(_gp_ekf_truth_gt, _gp_ekf_map_tail)
    plot_track_maps(
        _gp_ekf_truth_pred,
        _gp_ekf_truth_gt,
        map_panel="ground_truth",
        title="GP-EKF — идеальная разметка",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )

for item in gp_ekf_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    if SHOW_PLOTS:
        print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
        prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"], item["scored_test"])
        prediction = slice_trajectory_series(prediction, tail)
        ground_truth = slice_trajectory_series(ground_truth, tail)
        plot_track_maps(
            prediction,
            ground_truth,
            map_panel="model",
            title=f"GP-EKF — {item['name']}",
            figsize=MAP_FIGSIZE,
            annotate=False,
        )

## Скрытая марковская модель (HMM)

### Постановка

Ряд признаков описывается одной скрытой марковской цепью, в которой есть режимы штатного хода и режимы аномалии. Переходы между режимами, в том числе между классами, выучиваются по разметке обучающей выборки. Аномалия — точка, в которой вероятнее режим аномалии.

### Модель и оценка аномальности

Точка описывается кинематическими признаками с робастной нормировкой:

$$\mathbf f_i=\bigl(v_i,\ a^{\mathrm{long}}_i,\ a^{\mathrm{lat}}_i\bigr),\qquad a^{\mathrm{long}}_i=\frac{v_i-v_{i-1}}{\Delta t_i},\qquad a^{\mathrm{lat}}_i=\frac{v_i\,\Delta\psi_i}{\Delta t_i},\qquad \mathbf u_i=\frac{\mathbf f_i-m}{\mathrm{IQR}},$$

где $v_i$ — скорость на шаге от предыдущей точки, $\Delta\psi_i$ — изменение курса между соседними шагами, $m$ и $\mathrm{IQR}$ — медиана и межквартильный размах признаков на штатном ходу обучающей выборки. Признаки не зависят от курса и от $\Delta t$; поперечное ускорение мало на стоянке, где курс случаен.

$$S_i\in\mathcal S_{\mathrm{move}}\cup\mathcal S_{\mathrm{anom}},\qquad |\mathcal S_{\mathrm{move}}|=|\mathcal S_{\mathrm{anom}}|=3$$

$$P(S_1=k)=\pi_k,\qquad P(S_i=k\mid S_{i-1}=j)=T_{jk},\qquad \mathbf u_i\mid S_i=k\sim\mathcal N(\mathbf m_k,D_k),$$

где $\mathcal S_{\mathrm{move}}$, $\mathcal S_{\mathrm{anom}}$ — режимы штатного хода и аномалии, $\mathbf m_k$ и $D_k$ — среднее и диагональная ковариация признаков в режиме $k$.

Оценка аномальности — причинная апостериорная вероятность режима аномалии:

$$f_i(k)\propto\mathcal N(\mathbf u_i;\mathbf m_k,D_k)\sum_j f_{i-1}(j)\,T_{jk},\qquad s_i=\sum_{k\in\mathcal S_{\mathrm{anom}}}f_i(k),\qquad \hat a_i=\mathbf{1}[s_i>\tau],$$

где $f_i(k)=P(S_i=k\mid\mathbf u_{1:i})$, в начале отрезка $f(k)\propto\pi_k\,\mathcal N(\mathbf u;\mathbf m_k,D_k)$.

Вероятности перехода между классами задают типичную длительность эпизодов: внутри длинного эпизода аномалии цепь остаётся в режимах аномалии, даже если признаки отдельной точки похожи на штатный ход. В отличие от остальных моделей, HMM учитывает длительность эпизодов и использует разметку при обучении. Свойство модели: после одиночного выброса цепь некоторое время остаётся в режимах аномалии, и несколько следующих точек тоже размечаются аномалией.

Выброс искажает признаки следующих точек. Поэтому решение принимается в два прохода: после первого признаки пересчитываются от последних принятых точек (отвергнутые точки пропускаются), оценки и решения второго прохода — итоговые.

### Время и точки без оценки

Фильтр начинается заново в каждом отрезке между разрывами по времени больше 180 с. Без оценки: первые две точки отрезка (для ускорения нужны две скорости); из точек с одинаковым временем остаётся первая.

### Обучение и выбор порога

1. $m$ и $\mathrm{IQR}$ — по точкам штатного хода обучающих отрезков, как у MPCT. Признаки при обучении считаются так же, как в первом проходе обнаружения, — по предыдущей точке ряда: режимы аномалии учат и вход в эпизод, и движение внутри него. При признаках по последней точке штатного хода каждая аномальная точка при обучении выглядела бы прыжком от трека, а при обнаружении внутри длинного эпизода — обычным ходом, и цепь не уходила бы в режимы аномалии. У первой точки штатного хода после серии аномалий и у следующей признаков нет — прыжок выхода не попадает в обучение штатного хода; цепь при этом не рвётся: в этих точках режим без наблюдения.
2. $\pi$, $T$, $\mathbf m_k$, $D_k$ — алгоритмом EM (Баума — Уэлча) по всем отрезкам обучающей выборки; в размеченной точке вероятность режимов чужого класса равна нулю. Итерации — до относительного прироста логарифма правдоподобия меньше $10^{-4}$, не больше 200; из 5 запусков с фиксированными начальными значениями генератора выбирается лучший по правдоподобию. Число режимов не подбирается.
3. Подбор порога — как у AR(2); в сетку кандидатов добавляется 0.5 — граница «аномалия вероятнее штатного хода»; второй проход выполняется для каждого кандидата.

In [ ]:
HMM_STATES_PER_CLASS = 3
HMM_MAX_ITER = 200
HMM_TOL = 1e-4
HMM_RESTARTS = 5
# Нижняя граница дисперсии признака в режиме: без неё EM схлопывает режим на совпадающих точках.
HMM_VAR_MIN = 1e-3
# Нижняя граница начальных и переходных вероятностей: без неё отрезок, начинающийся в режиме, ни разу не встреченном
# в начале отрезка обучения, получает вероятность 0 и параметры EM становятся NaN.
HMM_PROB_MIN = 1e-6
# Граница «аномалия вероятнее штатного хода».
HMM_CANONICAL_THRESHOLD = 0.5


def _hmm_log_emission(features: np.ndarray, model: dict[str, np.ndarray]) -> np.ndarray:
    """Логарифм плотности признаков в каждом режиме (n, K), диагональная гауссовская плотность."""
    diff = features[:, None, :] - model["means"][None, :, :]
    var = model["vars"][None, :, :]
    return -0.5 * np.sum(diff * diff / var + np.log(2.0 * np.pi * var), axis=2)


def _hmm_runs(first: np.ndarray, scored: np.ndarray) -> list[np.ndarray]:
    """Позиции непрерывных участков точек с признаками внутри отрезка — по участку на перезапуск фильтра."""
    segment = np.cumsum(first)
    breaks = np.flatnonzero(np.diff(scored.astype(np.int8)) != 0) + 1
    breaks = np.union1d(breaks, np.flatnonzero(first))
    runs = np.split(np.arange(len(first)), breaks)
    return [run for run in runs if len(run) and scored[run[0]] and segment[run[0]] == segment[run[-1]]]


def fit_hmm_on_train(train: pd.DataFrame) -> dict[str, object]:
    """EM для HMM с режимами штатного хода и аномалии; в размеченной точке режимы чужого класса запрещены.

    Признаки — по предыдущей точке ряда, как в первом проходе обнаружения; нормировка — по штатному ходу, как у DBSCAN.
    Точка штатного хода после серии аномалий и следующая — без признаков, цепь не рвётся (режим без наблюдения). Из HMM_RESTARTS
    запусков с фиксированными seed выбирается лучший по логарифму правдоподобия.

    Args:
        train: Обучающий DataFrame с колонками x, y, time и l.

    Returns:
        dict[str, object]: center, scale, startprob, transmat, means, vars (K, d), loglik, iterations, converged.
    """
    labels = train["l"].astype(str).to_numpy()
    index, first = dbscan_segments(train, np.ones(len(train), dtype=bool))
    move = labels[index] == "move"
    position = np.arange(len(index))
    # Признаки — как в первом проходе обнаружения: по предыдущей точке ряда. Точка штатного хода после серии
    # аномалий — без предшественника: прыжок выхода не попадает в признаки штатного хода.
    restart = move & ~first & ~np.r_[True, move[:-1]]
    before = np.where(first | restart, -1, position - 1)
    raw = dbscan_features(enu_xy_from_frame(train)[index], time_seconds_from_frame(train)[index], before)
    scored = ~np.isnan(raw[:, 0])
    # Она и следующая точка без признаков, но цепь не рвётся: в них режим без наблюдения.
    blank = (restart | np.r_[False, restart[:-1]]) & ~first & ~scored
    if not np.any(scored & move) or not np.any(scored & ~move):
        raise RuntimeError("Для HMM нужны точки штатного хода и аномалии на обучающей выборке.")
    center, scale = robust_params(raw[scored & move])
    features = apply_robust(raw, center, scale)
    runs = _hmm_runs(first, scored | blank)
    n_states = 2 * HMM_STATES_PER_CLASS
    anomaly_state = np.arange(n_states) >= HMM_STATES_PER_CLASS
    # Запрет режимов чужого класса: log 0 в логарифме плотности.
    forbid = np.where(move[:, None], np.where(anomaly_state, -np.inf, 0.0), np.where(anomaly_state, 0.0, -np.inf))
    best = None
    for restart in range(HMM_RESTARTS):
        rng = np.random.default_rng(SEED + restart)
        means = np.vstack(
            [
                features[rng.choice(np.flatnonzero(scored & move), HMM_STATES_PER_CLASS, replace=False)],
                features[rng.choice(np.flatnonzero(scored & ~move), HMM_STATES_PER_CLASS, replace=False)],
            ]
        )
        variances = np.vstack(
            [
                np.tile(np.maximum(features[scored & move].var(axis=0), HMM_VAR_MIN), (HMM_STATES_PER_CLASS, 1)),
                np.tile(np.maximum(features[scored & ~move].var(axis=0), HMM_VAR_MIN), (HMM_STATES_PER_CLASS, 1)),
            ]
        )
        model = {
            "startprob": np.full(n_states, 1.0 / n_states),
            "transmat": np.full((n_states, n_states), 0.1 / (n_states - 1))
            + np.eye(n_states) * (0.9 - 0.1 / (n_states - 1)),
            "means": means,
            "vars": variances,
        }
        previous, converged = -np.inf, False
        for iteration in range(1, HMM_MAX_ITER + 1):
            start_count = np.zeros(n_states)
            trans_count = np.zeros((n_states, n_states))
            weight = np.zeros(n_states)
            first_moment = np.zeros_like(means)
            second_moment = np.zeros_like(means)
            loglik = 0.0
            for run in runs:
                observed = scored[run]
                log_emission = np.zeros((len(run), n_states))
                log_emission[observed] = _hmm_log_emission(features[run][observed], model)
                log_frame = np.ascontiguousarray(log_emission + forbid[run])
                log_prob, forward = _hmmc.forward_log(model["startprob"], model["transmat"], log_frame)
                backward = _hmmc.backward_log(model["startprob"], model["transmat"], log_frame)
                posterior = np.exp(forward + backward - log_prob)
                loglik += log_prob
                start_count += posterior[0]
                if len(run) > 1:
                    # compute_log_xi_sum уже нормирует на правдоподобие участка.
                    trans_count += np.exp(_hmmc.compute_log_xi_sum(forward, model["transmat"], backward, log_frame))
                weight += posterior[observed].sum(axis=0)
                first_moment += posterior[observed].T @ features[run][observed]
                second_moment += posterior[observed].T @ features[run][observed] ** 2
            start_count += HMM_PROB_MIN
            model["startprob"] = start_count / start_count.sum()
            trans_count += HMM_PROB_MIN
            model["transmat"] = trans_count / trans_count.sum(axis=1, keepdims=True)
            model["means"] = first_moment / weight[:, None]
            model["vars"] = np.maximum(second_moment / weight[:, None] - model["means"] ** 2, HMM_VAR_MIN)
            if abs(loglik - previous) < HMM_TOL * abs(loglik):
                converged = True
                break
            previous = loglik
        if not converged:
            print(f"  предупреждение: EM HMM не сошёлся за {HMM_MAX_ITER} итераций (запуск {restart + 1})")
        if best is None or loglik > best["loglik"]:
            best = {**model, "loglik": loglik, "iterations": iteration, "converged": converged}
    return {"center": center, "scale": scale, **best}


def hmm_filter_scores(features: np.ndarray, first: np.ndarray, model: dict[str, object]) -> np.ndarray:
    """Причинная апостериорная вероятность режима аномалии; фильтр начинается заново на каждом участке.

    Args:
        features: Нормированные признаки (P, d), NaN — точка без признаков.
        first: Маска первой точки отрезка (P,).
        model: Результат fit_hmm_on_train.

    Returns:
        np.ndarray: Оценка (P,), NaN — точка без оценки.
    """
    scored = ~np.isnan(features[:, 0])
    result = np.full(len(features), np.nan)
    anomaly_state = np.arange(len(model["startprob"])) >= HMM_STATES_PER_CLASS
    for run in _hmm_runs(first, scored):
        log_frame = np.ascontiguousarray(_hmm_log_emission(features[run], model))
        _, forward = _hmmc.forward_log(model["startprob"], model["transmat"], log_frame)
        posterior = np.exp(forward - logsumexp(forward, axis=1, keepdims=True))
        result[run] = posterior[:, anomaly_state].sum(axis=1)
    return result


def hmm_detect(
    frame: pd.DataFrame, model: dict[str, object], taus: np.ndarray
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Оценка HMM в два прохода, сразу для нескольких порогов.

    Второй проход — признаки по последним точкам, принятым первым проходом; фильтр пересчитывается целиком.

    Args:
        frame: DataFrame с колонками x, y и time.
        model: Результат fit_hmm_on_train.
        taus: Пороги; np.inf — без отвергнутых точек.

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray]: оценка второго прохода (C, N), NaN — точка без оценки;
        маска отвергнутых (C, N); оценка первого прохода (N,).
    """
    index, first = dbscan_segments(frame, np.ones(len(frame), dtype=bool))
    xy = enu_xy_from_frame(frame)[index]
    time_sec = time_seconds_from_frame(frame)[index]
    position = np.arange(len(index))
    first_features = apply_robust(
        dbscan_features(xy, time_sec, np.where(first, -1, position - 1)), model["center"], model["scale"]
    )
    first_scores = hmm_filter_scores(first_features, first, model)
    scored = ~np.isnan(first_scores)
    thresholds = np.atleast_1d(np.asarray(taus, dtype=float))
    scores = np.full((len(thresholds), len(frame)), np.nan)
    rejected = np.zeros((len(thresholds), len(frame)), dtype=bool)
    for row, tau in enumerate(thresholds):
        accepted = ~(first_scores > tau)
        last_accepted = np.maximum.accumulate(np.where(accepted, position, -1))
        before = np.where(first, -1, np.r_[-1, last_accepted[:-1]])
        second_features = apply_robust(dbscan_features(xy, time_sec, before), model["center"], model["scale"])
        second_scores = hmm_filter_scores(second_features, first, model)
        scores[row, index[scored]] = second_scores[scored]
        rejected[row, index[scored]] = second_scores[scored] > tau
    first_full = np.full(len(frame), np.nan)
    first_full[index[scored]] = first_scores[scored]
    return scores, rejected, first_full


def select_hmm_threshold(frame: pd.DataFrame, model: dict[str, object], mcc_band: float) -> dict[str, float]:
    """Подбирает порог: общая сетка по оценке первого прохода и 0.5, два прохода на кандидата.

    Args:
        frame: Обучающий DataFrame (time, lat, lon, x, y, l).
        model: Результат fit_hmm_on_train.
        mcc_band: Допуск по MCC относительно лучшего значения.

    Returns:
        dict[str, float]: tau, train_mcc, train_f1, train_dl_m, best_mcc, band_size.
    """
    _, _, first = hmm_detect(frame, model, np.array([np.inf]))
    candidates = np.unique(np.r_[threshold_candidates(first[np.isfinite(first)]), HMM_CANONICAL_THRESHOLD])
    scores, rejected, _ = hmm_detect(frame, model, candidates)
    ground_truth_anomaly = frame["l"].astype(str).to_numpy() == "anomaly"
    records = [
        _threshold_record(float(tau), rejected[row], frame, ground_truth_anomaly, ~np.isnan(scores[row]))
        for row, tau in enumerate(candidates)
    ]
    return select_threshold_from_records(records, mcc_band)


hmm_scenarios = (
    (DATASET_NAME, dataset),
)

hmm_rows: list[dict[str, object]] = []
hmm_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in hmm_scenarios:
    train = bundle["train"]
    test = bundle["test"]
    anomaly_train = train["l"].astype(str).to_numpy() == "anomaly"

    print("=" * 72)
    print(f"HMM: {scenario_name}")
    started = time.perf_counter()
    def train_model() -> dict[str, object]:
        """Обучение, выбор порога и оценки train для печати медиан."""
        model = fit_hmm_on_train(train)
        selection = select_hmm_threshold(train, model, MCC_BAND)
        scores = hmm_detect(train, model, np.array([selection["tau"]]))[0]
        return {"model": model, "selection": selection, "train_scores": scores}
    state = load_or_train("hmm", train, train_model)
    hmm_model, selection, train_scores = state["model"], state["selection"], state["train_scores"]
    tau = np.array([selection["tau"]])
    if np.any(~anomaly_train) and np.any(anomaly_train):
        print(
            f"  медиана оценки на train: move {np.nanmedian(train_scores[0][~anomaly_train]):.4g}, "
            f"anomaly {np.nanmedian(train_scores[0][anomaly_train]):.4g}"
        )
    stay = np.diag(hmm_model["transmat"])
    print(
        f"  EM: итераций {hmm_model['iterations']}, сошёлся {hmm_model['converged']}; "
        f"вероятность остаться в режиме: штатный ход {np.round(stay[:HMM_STATES_PER_CLASS], 3)}, "
        f"аномалия {np.round(stay[HMM_STATES_PER_CLASS:], 3)}; порог {tau[0]:.4g}: "
        f"MCC train {selection['train_mcc']:.4f} "
        f"(лучший {selection['best_mcc']:.4f}, в полосе {int(selection['band_size'])}), "
        f"DL train {selection['train_dl_m'] / 1000:.4f} км"
    )

    test_scores, test_rejected, _ = hmm_detect(test, hmm_model, tau)
    predicted_test = test_rejected[0]
    scored_test = ~np.isnan(test_scores[0])
    metrics = evaluate_test(test, predicted_test, scored_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м, без оценки {metrics['unscored_share']:.2%} "
        f"(время расчёта {elapsed:.1f} с)"
    )

    hmm_rows.append(test_report_row(scenario_name, metrics))
    hmm_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "scored_test": scored_test,
            "metrics": metrics,
        }
    )

hmm_summary = pd.DataFrame(hmm_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    hmm_summary.loc[:, report_columns],
    caption="HMM: сводные метрики на test",
)

if SHOW_PLOTS:
    _hmm_ideal_test = hmm_scenario_results[0]["test"]
    _hmm_map_tail = min(PLOT_TAIL, len(_hmm_ideal_test))
    print(f"\nИдеальная разметка: последние {nfmt(_hmm_map_tail)} точек теста (bbox: move + 5%)")
    _hmm_dummy_move = np.zeros(len(_hmm_ideal_test), dtype=bool)
    _hmm_truth_pred, _hmm_truth_gt = trajectory_series_pair(_hmm_ideal_test, _hmm_dummy_move)
    _hmm_truth_pred = slice_trajectory_series(_hmm_truth_pred, _hmm_map_tail)
    _hmm_truth_gt = slice_trajectory_series(_hmm_truth_gt, _hmm_map_tail)
    plot_track_maps(
        _hmm_truth_pred,
        _hmm_truth_gt,
        map_panel="ground_truth",
        title="HMM — идеальная разметка",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )

for item in hmm_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    if SHOW_PLOTS:
        print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
        prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"], item["scored_test"])
        prediction = slice_trajectory_series(prediction, tail)
        ground_truth = slice_trajectory_series(ground_truth, tail)
        plot_track_maps(
            prediction,
            ground_truth,
            map_panel="model",
            title=f"HMM — {item['name']}",
            figsize=MAP_FIGSIZE,
            annotate=False,
        )

## Фильтр Хэмпела

### Постановка

По модулю скорости в скользящем окне оцениваются медиана и типичный разброс. Аномалия — точка, скорость которой далеко от медианы окна. Модель пакетная (batch): окно включает следующие точки.

### Модель и оценка аномальности

$$v_i=\frac{\sqrt{\Delta x_i^2+\Delta y_i^2}}{\Delta t_i},\qquad m_i=\mathrm{med}(W_i),\qquad \mathrm{MAD}_i=\mathrm{med}_{v\in W_i}|v-m_i|,$$

$$s_i=\frac{|v_i-m_i|}{\max\bigl(1.4826\,\mathrm{MAD}_i,\ \sigma_{\min}\bigr)},\qquad \hat a_i=\mathbf{1}[s_i>\tau],$$

где $W_i$ — скорости $2w+1$ точек отрезка вокруг точки $i$, $w=10$; у краёв отрезка — ближайшие $2w+1$ точек, в коротком отрезке — весь отрезок; $1.4826\,\mathrm{MAD}_i$ — оценка СКО при нормальном распределении; $\sigma_{\min}$ — нижняя граница разброса.

Медиана устойчива, пока аномальных точек в окне не больше $w$. Свойство модели: эпизод аномалии длиннее $w$ точек сдвигает медиану и не обнаруживается; при постоянном смещении скорость аномальна только на входе в эпизод и выходе из него.

Одиночный выброс искажает и скорость следующей точки. Поэтому решение принимается в два прохода: после первого скорости пересчитываются от последней принятой точки (отвергнутые точки пропускаются), оценки и решения второго прохода — итоговые.

### Время и точки без оценки

Окно не выходит за отрезок между разрывами по времени больше 180 с. Без оценки: первая точка отрезка (скорость не определена); из точек с одинаковым временем остаётся первая; все точки отрезка, в котором меньше пяти скоростей.

### Обучение и выбор порога

1. $\sigma_{\min}$ — 5-й перцентиль $1.4826\,\mathrm{MAD}_i$ по точкам штатного хода обучающей выборки. Нижняя граница нужна, когда скорости в окне почти одинаковы (стоянка, округление координат): иначе малое отклонение даёт огромную оценку аномальности.
2. $w$ не подбирается. Подбор порога — как у AR(2); в сетку кандидатов добавляется 3, для каждого кандидата второй проход выполняется заново.

In [ ]:
HAMPEL_HALF_WINDOW = 10
HAMPEL_MIN_SPEEDS = 5
HAMPEL_MAD_SCALE = 1.4826
HAMPEL_FLOOR_PERCENTILE = 5.0
HAMPEL_CANONICAL_THRESHOLD = 3.0


def hampel_layout(frame: pd.DataFrame, half_window: int) -> dict[str, np.ndarray]:
    """Отрезки между разрывами по времени и окна скоростей для фильтра Хэмпела.

    Скорость есть у всех точек отрезка, кроме первой. Окно точки — 2w+1 ближайших скоростей отрезка
    (у краёв сдвинуто, в коротком отрезке — весь отрезок); отрезок с меньше HAMPEL_MIN_SPEEDS скоростями
    без оценки.

    Args:
        frame: DataFrame с колонками x, y и time.
        half_window: Полуокно w, точки.

    Returns:
        dict[str, np.ndarray]: index — точки отрезков подряд (P,); first — маска первой точки отрезка;
        rows — позиции (в index) точек с оценкой (R,); windows — позиции скоростей окна (R, 2w+1),
        -1 — дополнение в коротком отрезке.
    """
    if half_window < 1:
        raise ValueError("Полуокно фильтра Хэмпела должно быть не меньше 1.")
    segments = time_segments(frame, np.ones(len(frame), dtype=bool))
    index = np.concatenate(segments)
    first = np.zeros(len(index), dtype=bool)
    width = 2 * half_window + 1
    rows, windows = [], []
    offset = 0
    for segment in segments:
        first[offset] = True
        count = len(segment) - 1
        if count >= HAMPEL_MIN_SPEEDS:
            size = min(width, count)
            local = np.arange(count)
            start = np.clip(local - half_window, 0, count - size)
            window = offset + 1 + start[:, None] + np.arange(size)[None, :]
            rows.append(offset + 1 + local)
            windows.append(np.pad(window, ((0, 0), (0, width - size)), constant_values=-1))
        offset += len(segment)
    if not rows:
        raise RuntimeError("Нет отрезка с достаточным числом скоростей для фильтра Хэмпела.")
    return {"index": index, "first": first, "rows": np.concatenate(rows), "windows": np.concatenate(windows)}


def _hampel_speeds(xy: np.ndarray, time_sec: np.ndarray, previous: np.ndarray) -> np.ndarray:
    """Модуль скорости от точки ``previous`` до каждой точки, м/с; previous < 0 — NaN."""
    speed = np.full(len(xy), np.nan)
    valid = previous >= 0
    step = xy[valid] - xy[previous[valid]]
    speed[valid] = np.hypot(step[:, 0], step[:, 1]) / (time_sec[valid] - time_sec[previous[valid]])
    return speed


def _hampel_window_stats(speed: np.ndarray, layout: dict[str, np.ndarray]) -> tuple[np.ndarray, np.ndarray]:
    """Медиана и 1.4826·MAD скоростей окна для каждой точки с оценкой, м/с."""
    windows = layout["windows"]
    values = speed[np.maximum(windows, 0)]
    padded = windows < 0
    values[padded] = np.nan
    center = np.empty(len(windows))
    spread = np.empty(len(windows))
    full = ~padded.any(axis=1)
    for rows, reduce in ((full, np.median), (~full, np.nanmedian)):
        if np.any(rows):
            center[rows] = reduce(values[rows], axis=1)
            spread[rows] = reduce(np.abs(values[rows] - center[rows, None]), axis=1)
    return center, HAMPEL_MAD_SCALE * spread


def hampel_detect(
    frame: pd.DataFrame, layout: dict[str, np.ndarray], spread_floor: float, taus: np.ndarray
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Оценка аномальности фильтра Хэмпела в два прохода, сразу для нескольких порогов.

    Первый проход — скорости от предыдущей точки отрезка. Второй — скорости от последней точки, принятой
    первым проходом при данном пороге; его оценки и решения итоговые.

    Args:
        frame: DataFrame с колонками x, y и time.
        layout: Результат hampel_layout для ``frame``.
        spread_floor: Нижняя граница разброса σ_min, м/с.
        taus: Пороги; np.inf — без отвергнутых точек.

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray]: оценка второго прохода (C, N), NaN — точка без оценки;
        маска отвергнутых точек (C, N); оценка первого прохода (N,).
    """
    index, rows = layout["index"], layout["rows"]
    xy = enu_xy_from_frame(frame)[index]
    time_sec = time_seconds_from_frame(frame)[index]
    position = np.arange(len(index))
    previous = np.where(layout["first"], -1, position - 1)
    thresholds = np.atleast_1d(np.asarray(taus, dtype=float))

    def score(speed: np.ndarray) -> np.ndarray:
        center, spread = _hampel_window_stats(speed, layout)
        return np.abs(speed[rows] - center) / np.maximum(spread, spread_floor)

    first_scores = score(_hampel_speeds(xy, time_sec, previous))
    scores = np.full((len(thresholds), len(frame)), np.nan)
    rejected = np.zeros((len(thresholds), len(frame)), dtype=bool)
    for row, tau in enumerate(thresholds):
        accepted = np.ones(len(index), dtype=bool)
        accepted[rows] = first_scores <= tau
        # Последняя принятая точка до текущей; первая точка отрезка всегда принята, поэтому отрезки не смешиваются.
        last_accepted = np.maximum.accumulate(np.where(accepted, position, -1))
        before = np.where(layout["first"], -1, np.r_[-1, last_accepted[:-1]])
        second_scores = score(_hampel_speeds(xy, time_sec, before))
        scores[row, index[rows]] = second_scores
        rejected[row, index[rows]] = second_scores > tau
    first = np.full(len(frame), np.nan)
    first[index[rows]] = first_scores
    return scores, rejected, first


def fit_hampel_floor(train: pd.DataFrame, layout: dict[str, np.ndarray]) -> float:
    """Нижняя граница разброса: перцентиль 1.4826·MAD по окнам точек штатного хода обучающей выборки.

    Окна с нулевым разбросом (одинаковые скорости) в перцентиль не входят: иначе граница вырождается в нуль.

    Args:
        train: Обучающий DataFrame с колонками x, y, time и l.
        layout: Результат hampel_layout для ``train``.

    Returns:
        float: σ_min, м/с.
    """
    index = layout["index"]
    previous = np.where(layout["first"], -1, np.arange(len(index)) - 1)
    speed = _hampel_speeds(enu_xy_from_frame(train)[index], time_seconds_from_frame(train)[index], previous)
    _, spread = _hampel_window_stats(speed, layout)
    move = train["l"].astype(str).to_numpy()[index[layout["rows"]]] == "move"
    positive = spread[move & (spread > 0.0)]
    if positive.size == 0:
        raise RuntimeError("Нет окон штатного хода с ненулевым разбросом скорости.")
    return float(np.percentile(positive, HAMPEL_FLOOR_PERCENTILE))


def select_hampel_threshold(
    frame: pd.DataFrame, layout: dict[str, np.ndarray], spread_floor: float, mcc_band: float
) -> dict[str, float]:
    """Подбирает порог: сетка по оценке первого прохода и канонический порог 3, два прохода на кандидата.

    Args:
        frame: Обучающий DataFrame (time, lat, lon, x, y, l).
        layout: Результат hampel_layout для ``frame``.
        spread_floor: Нижняя граница разброса σ_min, м/с.
        mcc_band: Допуск по MCC относительно лучшего значения.

    Returns:
        dict[str, float]: tau, train_mcc, train_f1, train_dl_m, best_mcc, band_size.
    """
    _, _, first = hampel_detect(frame, layout, spread_floor, np.array([np.inf]))
    candidates = np.unique(np.r_[threshold_candidates(first[np.isfinite(first)]), HAMPEL_CANONICAL_THRESHOLD])
    scores, rejected, _ = hampel_detect(frame, layout, spread_floor, candidates)
    ground_truth_anomaly = frame["l"].astype(str).to_numpy() == "anomaly"
    records = [
        _threshold_record(float(tau), rejected[row], frame, ground_truth_anomaly, ~np.isnan(scores[row]))
        for row, tau in enumerate(candidates)
    ]
    return select_threshold_from_records(records, mcc_band)


hampel_scenarios = (
    (DATASET_NAME, dataset),
)

hampel_rows: list[dict[str, object]] = []
hampel_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in hampel_scenarios:
    train = bundle["train"]
    test = bundle["test"]
    anomaly_train = train["l"].astype(str).to_numpy() == "anomaly"

    print("=" * 72)
    print(f"Hampel: {scenario_name}")
    started = time.perf_counter()
    def train_model() -> dict[str, object]:
        """Обучение, выбор порога и оценки train для печати медиан."""
        layout = hampel_layout(train, HAMPEL_HALF_WINDOW)
        floor = fit_hampel_floor(train, layout)
        selection = select_hampel_threshold(train, layout, floor, MCC_BAND)
        scores = hampel_detect(train, layout, floor, np.array([selection["tau"]]))[0]
        return {"spread_floor": floor, "selection": selection, "train_scores": scores}
    state = load_or_train("hampel", train, train_model)
    spread_floor, selection, train_scores = state["spread_floor"], state["selection"], state["train_scores"]
    if np.any(~anomaly_train) and np.any(anomaly_train):
        print(
            f"  медиана оценки на train: move {np.nanmedian(train_scores[0][~anomaly_train]):.4g}, "
            f"anomaly {np.nanmedian(train_scores[0][anomaly_train]):.4g}"
        )
    print(
        f"  полуокно {HAMPEL_HALF_WINDOW} (окно {2 * HAMPEL_HALF_WINDOW + 1} точек); σ_min {spread_floor:.3g} м/с; "
        f"порог {selection['tau']:.4g}: MCC train {selection['train_mcc']:.4f} "
        f"(лучший {selection['best_mcc']:.4f}, в полосе {int(selection['band_size'])}), "
        f"DL train {selection['train_dl_m'] / 1000:.4f} км"
    )

    test_layout = hampel_layout(test, HAMPEL_HALF_WINDOW)
    test_scores, test_rejected, _ = hampel_detect(test, test_layout, spread_floor, np.array([selection["tau"]]))
    predicted_test = test_rejected[0]
    scored_test = ~np.isnan(test_scores[0])
    metrics = evaluate_test(test, predicted_test, scored_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м, без оценки {metrics['unscored_share']:.2%} "
        f"(время расчёта {elapsed:.1f} с)"
    )

    hampel_rows.append(test_report_row(scenario_name, metrics))
    hampel_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "scored_test": scored_test,
            "metrics": metrics,
        }
    )

hampel_summary = pd.DataFrame(hampel_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    hampel_summary.loc[:, report_columns],
    caption="Hampel: сводные метрики на test",
)

if SHOW_PLOTS:
    _hampel_ideal_test = hampel_scenario_results[0]["test"]
    _hampel_map_tail = min(PLOT_TAIL, len(_hampel_ideal_test))
    print(f"\nИдеальная разметка: последние {nfmt(_hampel_map_tail)} точек теста (bbox: move + 5%)")
    _hampel_dummy_move = np.zeros(len(_hampel_ideal_test), dtype=bool)
    _hampel_truth_pred, _hampel_truth_gt = trajectory_series_pair(_hampel_ideal_test, _hampel_dummy_move)
    _hampel_truth_pred = slice_trajectory_series(_hampel_truth_pred, _hampel_map_tail)
    _hampel_truth_gt = slice_trajectory_series(_hampel_truth_gt, _hampel_map_tail)
    plot_track_maps(
        _hampel_truth_pred,
        _hampel_truth_gt,
        map_panel="ground_truth",
        title="Hampel — идеальная разметка",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )

for item in hampel_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    if SHOW_PLOTS:
        print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
        prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"], item["scored_test"])
        prediction = slice_trajectory_series(prediction, tail)
        ground_truth = slice_trajectory_series(ground_truth, tail)
        plot_track_maps(
            prediction,
            ground_truth,
            map_panel="model",
            title=f"Hampel — {item['name']}",
            figsize=MAP_FIGSIZE,
            annotate=False,
        )

## DBSCAN

Признаки, нормировка, точки без оценки и второй проход — как у HMM.

### Постановка

Плотные области признаков штатного хода обучающей выборки задают норму. Аномалия — точка, далёкая от плотной области.

### Модель и оценка аномальности

Основная точка (core point) — точка штатного хода обучающей выборки, в $\varepsilon$-окрестности которой не меньше $\mathrm{MinPts}=6$ точек штатного хода, включая её саму. Оценка аномальности — расстояние до ближайшей основной точки:

$$s_i=\min_{c\in C_\varepsilon}\lVert\mathbf u_i-\mathbf u_c\rVert,\qquad \hat a_i=\mathbf{1}[s_i>\varepsilon],$$

где $C_\varepsilon$ — множество основных точек. Порог совпадает с радиусом: точка в $\varepsilon$-окрестности основной точки относится к её кластеру (правило DBSCAN). Решение по точке не зависит от остальных точек выборки.

### Обучение и выбор порога

1. Основные точки — по точкам штатного хода обучающих отрезков, как у MPCT.
2. $\varepsilon$ — из 30 квантилей расстояния до 6-го соседа на штатном ходу обучающей выборки, от медианы до 99.9-го перцентиля. Уровень $q$ — доля основных точек среди точек штатного хода: при $q<0.5$ область нормы сжимается до самого плотного ядра и отвергается заметная часть штатного хода; при $q=1$ кандидат задаёт самая изолированная точка штатного хода (например, ошибка разметки). Для каждого кандидата основные точки находятся заново, разметка обучающей выборки — в два прохода. Выбор — как $v_+$ у MPCT. $\mathrm{MinPts}$ не подбирается.

In [ ]:
DBSCAN_MIN_SAMPLES = 6
DBSCAN_EPS_LEVELS = np.linspace(0.5, 0.999, 30)


def fit_dbscan_on_train(train: pd.DataFrame) -> dict[str, np.ndarray]:
    """Нормировка и расстояния до MinPts-го соседа (с самой точкой) по штатному ходу обучающей выборки.

    Признаки считаются по соседним точкам обучающих отрезков штатного хода (move_segments): отрезок рвётся на
    каждой аномальной точке и на разрыве по времени.

    Args:
        train: Обучающий DataFrame с колонками x, y, time и l.

    Returns:
        dict[str, np.ndarray]: center, scale — робастная нормировка; points — нормированные признаки штатного
        хода (M, 3); k_distance — расстояние до MinPts-го соседа (M,).
    """
    index, first = segments_layout(move_segments(train))
    before = np.where(first, -1, np.arange(len(index)) - 1)
    raw = dbscan_features(enu_xy_from_frame(train)[index], time_seconds_from_frame(train)[index], before)
    raw = raw[~np.isnan(raw[:, 0])]
    if len(raw) <= DBSCAN_MIN_SAMPLES:
        raise RuntimeError("Мало точек штатного хода для DBSCAN.")
    center, scale = robust_params(raw)
    points = apply_robust(raw, center, scale)
    neighbors = NearestNeighbors(n_neighbors=DBSCAN_MIN_SAMPLES).fit(points)
    k_distance = neighbors.kneighbors(points)[0][:, -1]
    return {"center": center, "scale": scale, "points": points, "k_distance": k_distance}


def dbscan_detect(
    frame: pd.DataFrame, model: dict[str, np.ndarray], eps_values: np.ndarray
) -> tuple[np.ndarray, np.ndarray]:
    """Расстояние до ближайшей основной точки в два прохода, сразу для нескольких ε.

    Основные точки при данном ε — точки штатного хода с расстоянием до MinPts-го соседа не больше ε. Первый
    проход — признаки по предыдущим точкам отрезка; второй — по последним точкам, принятым первым проходом.

    Args:
        frame: DataFrame с колонками x, y и time.
        model: Результат fit_dbscan_on_train.
        eps_values: Радиусы ε в нормированных признаках.

    Returns:
        tuple[np.ndarray, np.ndarray]: оценка аномальности (C, N), NaN — точка без оценки; маска отвергнутых.
    """
    index, first = dbscan_segments(frame, np.ones(len(frame), dtype=bool))
    xy = enu_xy_from_frame(frame)[index]
    time_sec = time_seconds_from_frame(frame)[index]
    position = np.arange(len(index))
    radii = np.atleast_1d(np.asarray(eps_values, dtype=float))
    first_features = apply_robust(
        dbscan_features(xy, time_sec, np.where(first, -1, position - 1)), model["center"], model["scale"]
    )
    scored = ~np.isnan(first_features[:, 0])
    scores = np.full((len(radii), len(frame)), np.nan)
    rejected = np.zeros((len(radii), len(frame)), dtype=bool)
    for row, eps in enumerate(radii):
        core = model["points"][model["k_distance"] <= eps]
        if len(core) == 0:
            raise RuntimeError(f"Нет основных точек DBSCAN при ε = {eps:.4g}.")
        nearest = NearestNeighbors(n_neighbors=1).fit(core)

        def distance(features: np.ndarray) -> np.ndarray:
            return nearest.kneighbors(features[scored])[0][:, 0]

        accepted = np.ones(len(index), dtype=bool)
        accepted[scored] = distance(first_features) <= eps
        # Последняя принятая точка до текущей; первые точки отрезка без оценки и приняты — отрезки не смешиваются.
        last_accepted = np.maximum.accumulate(np.where(accepted, position, -1))
        before = np.where(first, -1, np.r_[-1, last_accepted[:-1]])
        second = apply_robust(dbscan_features(xy, time_sec, before), model["center"], model["scale"])
        second_scores = distance(second)
        scores[row, index[scored]] = second_scores
        rejected[row, index[scored]] = second_scores > eps
    return scores, rejected


def select_dbscan_eps(frame: pd.DataFrame, model: dict[str, np.ndarray], mcc_band: float) -> dict[str, float]:
    """Подбирает ε по квантилям расстояния до MinPts-го соседа: максимум MCC, в полосе — минимум DL.

    Args:
        frame: Обучающий DataFrame (time, lat, lon, x, y, l).
        model: Результат fit_dbscan_on_train.
        mcc_band: Допуск по MCC относительно лучшего значения.

    Returns:
        dict[str, float]: tau (выбранное ε), train_mcc, train_f1, train_dl_m, best_mcc, band_size.
    """
    candidates = np.unique(np.quantile(model["k_distance"], DBSCAN_EPS_LEVELS))
    scores, rejected = dbscan_detect(frame, model, candidates)
    ground_truth_anomaly = frame["l"].astype(str).to_numpy() == "anomaly"
    records = [
        _threshold_record(float(eps), rejected[row], frame, ground_truth_anomaly, ~np.isnan(scores[row]))
        for row, eps in enumerate(candidates)
    ]
    return select_threshold_from_records(records, mcc_band)


dbscan_scenarios = (
    (DATASET_NAME, dataset),
)

dbscan_rows: list[dict[str, object]] = []
dbscan_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in dbscan_scenarios:
    train = bundle["train"]
    test = bundle["test"]
    anomaly_train = train["l"].astype(str).to_numpy() == "anomaly"

    print("=" * 72)
    print(f"DBSCAN: {scenario_name}")
    started = time.perf_counter()
    def train_model() -> dict[str, object]:
        """Обучение, выбор порога и оценки train для печати медиан."""
        model = fit_dbscan_on_train(train)
        selection = select_dbscan_eps(train, model, MCC_BAND)
        scores = dbscan_detect(train, model, np.array([selection["tau"]]))[0]
        return {"model": model, "selection": selection, "train_scores": scores}
    state = load_or_train("dbscan", train, train_model)
    dbscan_model, selection, train_scores = state["model"], state["selection"], state["train_scores"]
    eps = np.array([selection["tau"]])
    if np.any(~anomaly_train) and np.any(anomaly_train):
        print(
            f"  медиана оценки на train: move {np.nanmedian(train_scores[0][~anomaly_train]):.4g}, "
            f"anomaly {np.nanmedian(train_scores[0][anomaly_train]):.4g}"
        )
    print(
        f"  MinPts {DBSCAN_MIN_SAMPLES}; точек штатного хода {len(dbscan_model['points'])}, "
        f"основных {int(np.sum(dbscan_model['k_distance'] <= eps[0]))}; ε {eps[0]:.4g}: "
        f"MCC train {selection['train_mcc']:.4f} "
        f"(лучший {selection['best_mcc']:.4f}, в полосе {int(selection['band_size'])}), "
        f"DL train {selection['train_dl_m'] / 1000:.4f} км"
    )

    test_scores, test_rejected = dbscan_detect(test, dbscan_model, eps)
    predicted_test = test_rejected[0]
    scored_test = ~np.isnan(test_scores[0])
    metrics = evaluate_test(test, predicted_test, scored_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м, без оценки {metrics['unscored_share']:.2%} "
        f"(время расчёта {elapsed:.1f} с)"
    )

    dbscan_rows.append(test_report_row(scenario_name, metrics))
    dbscan_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "scored_test": scored_test,
            "metrics": metrics,
        }
    )

dbscan_summary = pd.DataFrame(dbscan_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    dbscan_summary.loc[:, report_columns],
    caption="DBSCAN: сводные метрики на test",
)

if SHOW_PLOTS:
    _dbscan_ideal_test = dbscan_scenario_results[0]["test"]
    _dbscan_map_tail = min(PLOT_TAIL, len(_dbscan_ideal_test))
    print(f"\nИдеальная разметка: последние {nfmt(_dbscan_map_tail)} точек теста (bbox: move + 5%)")
    _dbscan_dummy_move = np.zeros(len(_dbscan_ideal_test), dtype=bool)
    _dbscan_truth_pred, _dbscan_truth_gt = trajectory_series_pair(_dbscan_ideal_test, _dbscan_dummy_move)
    _dbscan_truth_pred = slice_trajectory_series(_dbscan_truth_pred, _dbscan_map_tail)
    _dbscan_truth_gt = slice_trajectory_series(_dbscan_truth_gt, _dbscan_map_tail)
    plot_track_maps(
        _dbscan_truth_pred,
        _dbscan_truth_gt,
        map_panel="ground_truth",
        title="DBSCAN — идеальная разметка",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )

for item in dbscan_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    if SHOW_PLOTS:
        print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
        prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"], item["scored_test"])
        prediction = slice_trajectory_series(prediction, tail)
        ground_truth = slice_trajectory_series(ground_truth, tail)
        plot_track_maps(
            prediction,
            ground_truth,
            map_panel="model",
            title=f"DBSCAN — {item['name']}",
            figsize=MAP_FIGSIZE,
            annotate=False,
        )

## Local Outlier Factor (LOF)

Признаки, нормировка, точки без оценки и второй проход — как у HMM.

### Постановка

Локальная плотность точки в пространстве признаков сравнивается с плотностью её соседей среди точек штатного хода обучающей выборки (обнаружение новизны, novelty detection). Аномалия — точка в области, разреженной относительно соседей.

### Модель и оценка аномальности

Для набора $T$ нормированных признаков точек штатного хода обучающей выборки:

$$\mathrm{rd}_k(\mathbf u,\mathbf y)=\max\bigl(d_k(\mathbf y),\ \lVert\mathbf u-\mathbf y\rVert\bigr),\qquad \mathrm{lrd}_k(\mathbf u)=\Bigl(\frac1k\sum_{\mathbf y\in N_k(\mathbf u)}\mathrm{rd}_k(\mathbf u,\mathbf y)\Bigr)^{-1},$$

$$s_i=\mathrm{LOF}_k(\mathbf u_i)=\frac1k\sum_{\mathbf y\in N_k(\mathbf u_i)}\frac{\mathrm{lrd}_k(\mathbf y)}{\mathrm{lrd}_k(\mathbf u_i)},\qquad \hat a_i=\mathbf{1}[s_i>\tau],$$

где $N_k(\mathbf u)$ — $k$ ближайших к $\mathbf u$ точек набора $T$, $d_k(\mathbf y)$ — расстояние от $\mathbf y$ до её $k$-го соседа в $T$, $\mathrm{rd}_k$ — расстояние достижимости (reachability distance), $\mathrm{lrd}_k$ — локальная плотность достижимости (local reachability density). $s_i\approx1$ — плотность как у соседей; $s_i\gg1$ — точка в разреженной области. Для точки из $T$ сама точка в $N_k$ не входит.

В отличие от DBSCAN, оценка аномальности непрерывна и относительна: плотная область «стоянка» и разреженная область «ход» оцениваются каждая относительно своих соседей. Решение по точке не зависит от остальных точек выборки.

Свойство модели (риск принят): точки с совпадающими признаками (стоянка с повторяющимися координатами) дают нулевые расстояния достижимости и очень большую $\mathrm{lrd}_k$; у соседней точки с обычным шумом GPS $s_i$ тогда велика — возможны ложные срабатывания рядом со стоянками.

### Обучение и выбор порога

1. Нормировка — как у HMM; $T$ — по обучающим отрезкам, как у MPCT.
2. $k$ и порог подбираются совместно: $k\in\{10,20,35,50\}$; для каждого $k$ — общая сетка кандидатов порога по оценке первого прохода, второй проход на каждого кандидата; выбор по всем парам — как $v_+$ у MPCT. Свойство модели: оценка аномальности штатного хода слабо зависит от $k$, поэтому выбор $k$ определяется малыми различиями MCC и может меняться между обучающими выборками.

In [ ]:
LOF_K_GRID = (10, 20, 35, 50)
# Регуляризация средней достижимости, как в sklearn: совпадающие признаки не дают бесконечной плотности.
LOF_REACH_EPS = 1e-10


def fit_lof_on_train(train: pd.DataFrame) -> dict[str, object]:
    """Набор T штатного хода обучающей выборки, его соседи без самой точки и номера строк T по точкам выборки.

    Признаки и нормировка — как у DBSCAN: по обучающим отрезкам штатного хода (move_segments).

    Args:
        train: Обучающий DataFrame с колонками x, y, time и l.

    Returns:
        dict[str, object]: center, scale; points — T (M, 3); neighbor_dist, neighbor_index — соседи точек T без
        самой точки (M, max k); self_rows — строка T для каждой точки выборки, -1 — точки нет в T; nearest —
        поиск соседей по T.
    """
    index, first = segments_layout(move_segments(train))
    before = np.where(first, -1, np.arange(len(index)) - 1)
    raw = dbscan_features(enu_xy_from_frame(train)[index], time_seconds_from_frame(train)[index], before)
    valid = ~np.isnan(raw[:, 0])
    k_max = max(LOF_K_GRID)
    if np.sum(valid) <= k_max:
        raise RuntimeError("Мало точек штатного хода для LOF.")
    center, scale = robust_params(raw[valid])
    points = apply_robust(raw[valid], center, scale)
    nearest = NearestNeighbors(n_neighbors=k_max + 1).fit(points)
    distance, neighbor = nearest.kneighbors(points)
    own = neighbor == np.arange(len(points))[:, None]
    # При более чем k_max совпадающих признаках самой точки может не быть среди соседей — отбрасывается последний.
    own[~own.any(axis=1), -1] = True
    self_rows = np.full(len(train), -1, dtype=np.int64)
    self_rows[index[valid]] = np.arange(len(points))
    return {
        "center": center,
        "scale": scale,
        "points": points,
        "neighbor_dist": distance[~own].reshape(len(points), k_max),
        "neighbor_index": neighbor[~own].reshape(len(points), k_max),
        "self_rows": self_rows,
        "nearest": nearest,
    }


def lof_train_density(model: dict[str, object], k: int) -> tuple[np.ndarray, np.ndarray]:
    """Расстояние до k-го соседа и локальная плотность достижимости точек T при данном k.

    Args:
        model: Результат fit_lof_on_train.
        k: Число соседей.

    Returns:
        tuple[np.ndarray, np.ndarray]: d_k (M,); lrd_k (M,).
    """
    k_distance = model["neighbor_dist"][:, k - 1]
    reach = np.maximum(k_distance[model["neighbor_index"][:, :k]], model["neighbor_dist"][:, :k])
    return k_distance, 1.0 / (reach.mean(axis=1) + LOF_REACH_EPS)


def lof_scores(
    model: dict[str, object], k: int, density: tuple[np.ndarray, np.ndarray], features: np.ndarray, rows: np.ndarray
) -> np.ndarray:
    """LOF_k нормированных признаков относительно T; точка из T среди своих соседей не учитывается.

    Args:
        model: Результат fit_lof_on_train.
        k: Число соседей.
        density: Результат lof_train_density для k.
        features: Нормированные признаки (R, 3).
        rows: Строки T для этих точек; -1 — точки нет в T.

    Returns:
        np.ndarray: LOF_k (R,).
    """
    k_distance, lrd = density
    distance, neighbor = model["nearest"].kneighbors(features, n_neighbors=k + 1)
    own = neighbor == rows[:, None]
    own[~own.any(axis=1), -1] = True
    distance = distance[~own].reshape(len(features), k)
    neighbor = neighbor[~own].reshape(len(features), k)
    point_lrd = 1.0 / (np.maximum(k_distance[neighbor], distance).mean(axis=1) + LOF_REACH_EPS)
    return lrd[neighbor].mean(axis=1) / point_lrd


def lof_detect(
    frame: pd.DataFrame, model: dict[str, object], k: int, taus: np.ndarray, self_rows: Optional[np.ndarray] = None
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """LOF_k в два прохода, сразу для нескольких порогов.

    Второй проход — признаки по последним точкам, принятым первым проходом; LOF пересчитывается только у точек,
    признаки которых изменились, и только для ещё не встречавшихся признаков (cached_rows).

    Args:
        frame: DataFrame с колонками x, y и time.
        model: Результат fit_lof_on_train.
        k: Число соседей.
        taus: Пороги; np.inf — без отвергнутых точек.
        self_rows: Строки T по точкам ``frame`` (для обучающей выборки); None — точек T в выборке нет.

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray]: оценка второго прохода (C, N), NaN — точка без оценки;
        маска отвергнутых (C, N); оценка первого прохода (N,).
    """
    index, first = dbscan_segments(frame, np.ones(len(frame), dtype=bool))
    xy = enu_xy_from_frame(frame)[index]
    time_sec = time_seconds_from_frame(frame)[index]
    position = np.arange(len(index))
    rows = np.full(len(index), -1, dtype=np.int64) if self_rows is None else self_rows[index]
    density = lof_train_density(model, k)
    first_features = apply_robust(
        dbscan_features(xy, time_sec, np.where(first, -1, position - 1)), model["center"], model["scale"]
    )
    scored = ~np.isnan(first_features[:, 0])
    first_scores = np.full(len(index), np.nan)
    first_scores[scored] = lof_scores(model, k, density, first_features[scored], rows[scored])
    thresholds = np.atleast_1d(np.asarray(taus, dtype=float))
    scores = np.full((len(thresholds), len(frame)), np.nan)
    rejected = np.zeros((len(thresholds), len(frame)), dtype=bool)
    # Оценки второго прохода повторяются при разных порогах: кэш по ключу признаков.
    cache: dict[bytes, object] = {}
    for row, tau in enumerate(thresholds):
        accepted = ~(first_scores > tau)
        last_accepted = np.maximum.accumulate(np.where(accepted, position, -1))
        before = np.where(first, -1, np.r_[-1, last_accepted[:-1]])
        second_features = apply_robust(dbscan_features(xy, time_sec, before), model["center"], model["scale"])
        second_scores = first_scores.copy()
        changed = scored & np.any(second_features != first_features, axis=1)
        if np.any(changed):
            changed_rows = np.flatnonzero(changed)
            second_scores[changed] = cached_rows(
                feature_keys(before, changed_rows),
                cache,
                lambda missing: lof_scores(
                    model, k, density, second_features[changed_rows[missing]], rows[changed_rows[missing]]
                ),
            )
        scores[row, index[scored]] = second_scores[scored]
        rejected[row, index[scored]] = second_scores[scored] > tau
    first_full = np.full(len(frame), np.nan)
    first_full[index[scored]] = first_scores[scored]
    return scores, rejected, first_full


def select_lof_parameters(frame: pd.DataFrame, model: dict[str, object], mcc_band: float) -> dict[str, float]:
    """Совместный подбор k и порога: для каждого k — общая сетка по оценке первого прохода, два прохода.

    Args:
        frame: Обучающий DataFrame (time, lat, lon, x, y, l).
        model: Результат fit_lof_on_train.
        mcc_band: Допуск по MCC относительно лучшего значения.

    Returns:
        dict[str, float]: k, tau, train_mcc, train_f1, train_dl_m, best_mcc, band_size.
    """
    ground_truth_anomaly = frame["l"].astype(str).to_numpy() == "anomaly"
    records = []
    for k in LOF_K_GRID:
        _, _, first = lof_detect(frame, model, k, np.array([np.inf]), model["self_rows"])
        candidates = threshold_candidates(first[np.isfinite(first)])
        scores, rejected, _ = lof_detect(frame, model, k, candidates, model["self_rows"])
        for row, tau in enumerate(candidates):
            record = _threshold_record(float(tau), rejected[row], frame, ground_truth_anomaly, ~np.isnan(scores[row]))
            record["k"] = k
            records.append(record)
    selection = select_threshold_from_records(records, mcc_band)
    chosen = next(
        r
        for r in records
        if r["tau"] == selection["tau"]
        and r["loss_m"] == selection["train_dl_m"]
        and r["mcc"] == selection["train_mcc"]
    )
    return {"k": chosen["k"], **selection}


lof_scenarios = (
    (DATASET_NAME, dataset),
)

lof_rows: list[dict[str, object]] = []
lof_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in lof_scenarios:
    train = bundle["train"]
    test = bundle["test"]
    anomaly_train = train["l"].astype(str).to_numpy() == "anomaly"

    print("=" * 72)
    print(f"LOF: {scenario_name}")
    started = time.perf_counter()
    def train_model() -> dict[str, object]:
        """Обучение, выбор порога и оценки train для печати медиан."""
        model = fit_lof_on_train(train)
        selection = select_lof_parameters(train, model, MCC_BAND)
        scores = lof_detect(train, model, int(selection["k"]), np.array([selection["tau"]]), model["self_rows"])[0]
        return {"model": model, "selection": selection, "train_scores": scores}
    state = load_or_train("lof", train, train_model)
    lof_model, selection, train_scores = state["model"], state["selection"], state["train_scores"]
    k, tau = int(selection["k"]), np.array([selection["tau"]])
    if np.any(~anomaly_train) and np.any(anomaly_train):
        print(
            f"  медиана LOF на train: move {np.nanmedian(train_scores[0][~anomaly_train]):.4g}, "
            f"anomaly {np.nanmedian(train_scores[0][anomaly_train]):.4g}"
        )
    print(
        f"  точек штатного хода {len(lof_model['points'])}; k {k}, порог {tau[0]:.4g}: "
        f"MCC train {selection['train_mcc']:.4f} "
        f"(лучший {selection['best_mcc']:.4f}, в полосе {int(selection['band_size'])}), "
        f"DL train {selection['train_dl_m'] / 1000:.4f} км"
    )

    test_scores, test_rejected, _ = lof_detect(test, lof_model, k, tau)
    predicted_test = test_rejected[0]
    scored_test = ~np.isnan(test_scores[0])
    metrics = evaluate_test(test, predicted_test, scored_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м, без оценки {metrics['unscored_share']:.2%} "
        f"(время расчёта {elapsed:.1f} с)"
    )

    lof_rows.append(test_report_row(scenario_name, metrics))
    lof_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "scored_test": scored_test,
            "metrics": metrics,
        }
    )

lof_summary = pd.DataFrame(lof_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    lof_summary.loc[:, report_columns],
    caption="LOF: сводные метрики на test",
)

if SHOW_PLOTS:
    _lof_ideal_test = lof_scenario_results[0]["test"]
    _lof_map_tail = min(PLOT_TAIL, len(_lof_ideal_test))
    print(f"\nИдеальная разметка: последние {nfmt(_lof_map_tail)} точек теста (bbox: move + 5%)")
    _lof_dummy_move = np.zeros(len(_lof_ideal_test), dtype=bool)
    _lof_truth_pred, _lof_truth_gt = trajectory_series_pair(_lof_ideal_test, _lof_dummy_move)
    _lof_truth_pred = slice_trajectory_series(_lof_truth_pred, _lof_map_tail)
    _lof_truth_gt = slice_trajectory_series(_lof_truth_gt, _lof_map_tail)
    plot_track_maps(
        _lof_truth_pred,
        _lof_truth_gt,
        map_panel="ground_truth",
        title="LOF — идеальная разметка",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )

for item in lof_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    if SHOW_PLOTS:
        print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
        prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"], item["scored_test"])
        prediction = slice_trajectory_series(prediction, tail)
        ground_truth = slice_trajectory_series(ground_truth, tail)
        plot_track_maps(
            prediction,
            ground_truth,
            map_panel="model",
            title=f"LOF — {item['name']}",
            figsize=MAP_FIGSIZE,
            annotate=False,
        )

## Isolation Forest

Признаки, нормировка, точки без оценки и второй проход — как у HMM.

### Постановка

Лес случайных деревьев, обученный на штатном ходу обучающей выборки, изолирует точку последовательными разрезами по признакам (обнаружение новизны). Аномалия — точка, которая отделяется за малое число разрезов.

### Модель и оценка аномальности

Каждое из 200 деревьев строится по случайной подвыборке из $n_{\mathrm{sub}}=256$ точек набора $T$ (как у LOF): в узле выбираются случайный признак и случайное значение разреза в диапазоне точек узла.

$$s_i=2^{-\bar h(\mathbf u_i)/c(n_{\mathrm{sub}})},\qquad c(n_{\mathrm{sub}})=2H(n_{\mathrm{sub}}-1)-\frac{2(n_{\mathrm{sub}}-1)}{n_{\mathrm{sub}}},\qquad \hat a_i=\mathbf{1}[s_i>\tau],$$

где $\bar h(\mathbf u_i)$ — средняя по деревьям длина пути от корня до листа точки (с поправкой $c$ на число точек в листе), $H(n)\approx\ln n+0.5772$ — гармоническое число, $c(n_{\mathrm{sub}})$ — средняя длина пути в двоичном дереве поиска из $n_{\mathrm{sub}}$ точек. Точка вне области штатного хода уходит в крайнюю ветку и изолируется за несколько разрезов: $s_i$ близка к 1; у точки штатного хода $s_i<0.5$.

В отличие от DBSCAN и LOF, осевые разрезы оценивают положение точки относительно всей области штатного хода, а не локальную плотность. Решение по точке не зависит от остальных точек выборки.

### Обучение и выбор порога

1. Нормировка — как у HMM, набор $T$ — как у LOF; лес обучается на $T$ при фиксированном начальном значении генератора; число деревьев и $n_{\mathrm{sub}}$ не подбираются.
2. Подбор порога — как у AR(2); в сетку кандидатов добавляется 0.5 — граница аномальности из определения оценки; второй проход выполняется для каждого кандидата.

In [ ]:
IF_N_ESTIMATORS = 200
IF_MAX_SAMPLES = 256
# Граница аномальности из определения оценки: s > 0.5 — путь короче среднего в двоичном дереве поиска.
IF_CANONICAL_THRESHOLD = 0.5


def fit_isolation_forest_on_train(train: pd.DataFrame) -> dict[str, object]:
    """Лес по нормированным признакам штатного хода обучающей выборки.

    Признаки и нормировка — как у DBSCAN: по обучающим отрезкам штатного хода (move_segments).

    Args:
        train: Обучающий DataFrame с колонками x, y, time и l.

    Returns:
        dict[str, object]: center, scale — робастная нормировка; points — набор T (M, 3); forest — обученный лес.
    """
    index, first = segments_layout(move_segments(train))
    before = np.where(first, -1, np.arange(len(index)) - 1)
    raw = dbscan_features(enu_xy_from_frame(train)[index], time_seconds_from_frame(train)[index], before)
    raw = raw[~np.isnan(raw[:, 0])]
    if len(raw) < 2:
        raise RuntimeError("Мало точек штатного хода для Isolation Forest.")
    center, scale = robust_params(raw)
    points = apply_robust(raw, center, scale)
    forest = IsolationForest(
        n_estimators=IF_N_ESTIMATORS, max_samples=min(IF_MAX_SAMPLES, len(points)), random_state=SEED, n_jobs=-1
    ).fit(points)
    return {"center": center, "scale": scale, "points": points, "forest": forest}


def isolation_forest_detect(
    frame: pd.DataFrame, model: dict[str, object], taus: np.ndarray
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Оценка s = 2^(-h̄/c) в два прохода, сразу для нескольких порогов.

    Второй проход — признаки по последним точкам, принятым первым проходом; оценка пересчитывается только у точек,
    признаки которых изменились, и только для ещё не встречавшихся признаков (cached_rows).

    Args:
        frame: DataFrame с колонками x, y и time.
        model: Результат fit_isolation_forest_on_train.
        taus: Пороги; np.inf — без отвергнутых точек.

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray]: оценка второго прохода (C, N), NaN — точка без оценки;
        маска отвергнутых (C, N); оценка первого прохода (N,).
    """
    index, first = dbscan_segments(frame, np.ones(len(frame), dtype=bool))
    xy = enu_xy_from_frame(frame)[index]
    time_sec = time_seconds_from_frame(frame)[index]
    position = np.arange(len(index))
    first_features = apply_robust(
        dbscan_features(xy, time_sec, np.where(first, -1, position - 1)), model["center"], model["scale"]
    )
    scored = ~np.isnan(first_features[:, 0])
    first_scores = np.full(len(index), np.nan)
    # score_samples в sklearn — оценка статьи со знаком минус.
    first_scores[scored] = -model["forest"].score_samples(first_features[scored])
    thresholds = np.atleast_1d(np.asarray(taus, dtype=float))
    scores = np.full((len(thresholds), len(frame)), np.nan)
    rejected = np.zeros((len(thresholds), len(frame)), dtype=bool)
    # Оценки второго прохода повторяются при разных порогах: кэш по ключу признаков.
    cache: dict[bytes, object] = {}
    for row, tau in enumerate(thresholds):
        accepted = ~(first_scores > tau)
        last_accepted = np.maximum.accumulate(np.where(accepted, position, -1))
        before = np.where(first, -1, np.r_[-1, last_accepted[:-1]])
        second_features = apply_robust(dbscan_features(xy, time_sec, before), model["center"], model["scale"])
        second_scores = first_scores.copy()
        changed = scored & np.any(second_features != first_features, axis=1)
        if np.any(changed):
            changed_rows = np.flatnonzero(changed)
            second_scores[changed] = cached_rows(
                feature_keys(before, changed_rows),
                cache,
                lambda missing: -model["forest"].score_samples(second_features[changed_rows[missing]]),
            )
        scores[row, index[scored]] = second_scores[scored]
        rejected[row, index[scored]] = second_scores[scored] > tau
    first_full = np.full(len(frame), np.nan)
    first_full[index[scored]] = first_scores[scored]
    return scores, rejected, first_full


def select_isolation_forest_threshold(
    frame: pd.DataFrame, model: dict[str, object], mcc_band: float
) -> dict[str, float]:
    """Подбирает порог: общая сетка по оценке первого прохода и 0.5, два прохода на кандидата.

    Args:
        frame: Обучающий DataFrame (time, lat, lon, x, y, l).
        model: Результат fit_isolation_forest_on_train.
        mcc_band: Допуск по MCC относительно лучшего значения.

    Returns:
        dict[str, float]: tau, train_mcc, train_f1, train_dl_m, best_mcc, band_size.
    """
    _, _, first = isolation_forest_detect(frame, model, np.array([np.inf]))
    candidates = np.unique(np.r_[threshold_candidates(first[np.isfinite(first)]), IF_CANONICAL_THRESHOLD])
    scores, rejected, _ = isolation_forest_detect(frame, model, candidates)
    ground_truth_anomaly = frame["l"].astype(str).to_numpy() == "anomaly"
    records = [
        _threshold_record(float(tau), rejected[row], frame, ground_truth_anomaly, ~np.isnan(scores[row]))
        for row, tau in enumerate(candidates)
    ]
    return select_threshold_from_records(records, mcc_band)


if_scenarios = (
    (DATASET_NAME, dataset),
)

if_rows: list[dict[str, object]] = []
if_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in if_scenarios:
    train = bundle["train"]
    test = bundle["test"]
    anomaly_train = train["l"].astype(str).to_numpy() == "anomaly"

    print("=" * 72)
    print(f"Isolation Forest: {scenario_name}")
    started = time.perf_counter()
    def train_model() -> dict[str, object]:
        """Обучение, выбор порога и оценки train для печати медиан."""
        model = fit_isolation_forest_on_train(train)
        selection = select_isolation_forest_threshold(train, model, MCC_BAND)
        scores = isolation_forest_detect(train, model, np.array([selection["tau"]]))[0]
        return {"model": model, "selection": selection, "train_scores": scores}
    state = load_or_train("isolation_forest", train, train_model)
    if_model, selection, train_scores = state["model"], state["selection"], state["train_scores"]
    tau = np.array([selection["tau"]])
    if np.any(~anomaly_train) and np.any(anomaly_train):
        print(
            f"  медиана оценки на train: move {np.nanmedian(train_scores[0][~anomaly_train]):.4g}, "
            f"anomaly {np.nanmedian(train_scores[0][anomaly_train]):.4g}"
        )
    print(
        f"  {IF_N_ESTIMATORS} деревьев, подвыборка {IF_MAX_SAMPLES}, точек штатного хода {len(if_model['points'])}; "
        f"порог {tau[0]:.4g}: MCC train {selection['train_mcc']:.4f} "
        f"(лучший {selection['best_mcc']:.4f}, в полосе {int(selection['band_size'])}), "
        f"DL train {selection['train_dl_m'] / 1000:.4f} км"
    )

    test_scores, test_rejected, _ = isolation_forest_detect(test, if_model, tau)
    predicted_test = test_rejected[0]
    scored_test = ~np.isnan(test_scores[0])
    metrics = evaluate_test(test, predicted_test, scored_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м, без оценки {metrics['unscored_share']:.2%} "
        f"(время расчёта {elapsed:.1f} с)"
    )

    if_rows.append(test_report_row(scenario_name, metrics))
    if_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "scored_test": scored_test,
            "metrics": metrics,
        }
    )

if_summary = pd.DataFrame(if_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    if_summary.loc[:, report_columns],
    caption="Isolation Forest: сводные метрики на test",
)

if SHOW_PLOTS:
    _if_ideal_test = if_scenario_results[0]["test"]
    _if_map_tail = min(PLOT_TAIL, len(_if_ideal_test))
    print(f"\nИдеальная разметка: последние {nfmt(_if_map_tail)} точек теста (bbox: move + 5%)")
    _if_dummy_move = np.zeros(len(_if_ideal_test), dtype=bool)
    _if_truth_pred, _if_truth_gt = trajectory_series_pair(_if_ideal_test, _if_dummy_move)
    _if_truth_pred = slice_trajectory_series(_if_truth_pred, _if_map_tail)
    _if_truth_gt = slice_trajectory_series(_if_truth_gt, _if_map_tail)
    plot_track_maps(
        _if_truth_pred,
        _if_truth_gt,
        map_panel="ground_truth",
        title="Isolation Forest — идеальная разметка",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )

for item in if_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    if SHOW_PLOTS:
        print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
        prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"], item["scored_test"])
        prediction = slice_trajectory_series(prediction, tail)
        ground_truth = slice_trajectory_series(ground_truth, tail)
        plot_track_maps(
            prediction,
            ground_truth,
            map_panel="model",
            title=f"Isolation Forest — {item['name']}",
            figsize=MAP_FIGSIZE,
            annotate=False,
        )

## One-Class SVM

Признаки, нормировка, точки без оценки и второй проход — как у HMM.

### Постановка

По точкам штатного хода обучающей выборки строится граница области штатного хода в пространстве признаков (обнаружение новизны). Аномалия — точка, далёкая от этой области.

### Модель и оценка аномальности

$$f(\mathbf u)=\sum_j\alpha_j K(\mathbf u_j,\mathbf u)-\rho,\qquad K(\mathbf u,\mathbf u')=\exp\bigl(-\gamma\lVert\mathbf u-\mathbf u'\rVert^2\bigr),\qquad \gamma=\frac{1}{d\,\mathrm{Var}(T)},$$

$$s_i=-f(\mathbf u_i),\qquad \hat a_i=\mathbf{1}[s_i>\tau],$$

где сумма — по опорным векторам $\mathbf u_j$ набора $T$ (как у LOF), $\alpha_j\ge0$ и $\rho$ — решение задачи One-Class SVM при $\nu=0.1$, $d=3$ — число признаков, $\mathrm{Var}(T)$ — дисперсия всех значений признаков набора $T$. Внутри области штатного хода $f>0$; при $\tau=0$ вне области оказывается не больше доли $\nu$ точек $T$.

$\nu$ ограничивает снизу долю опорных векторов: при $\nu=0.1$ их не меньше 10% точек $T$, и оценка ближе к сглаженной оценке плотности — информативна не только у границы, но и во всём пространстве признаков.

В отличие от DBSCAN, LOF и Isolation Forest, граница гладкая и определяется взвешенной суммой ядер опорных векторов. Решение по точке не зависит от остальных точек выборки.

Свойство модели: далеко от области штатного хода значения ядра близки к нулю и оценка насыщается на $\rho$ — далёкие точки между собой не различаются; на решение это не влияет, пока порог ниже $\rho$.

### Обучение и выбор порога

1. Нормировка — как у HMM, набор $T$ — как у LOF; модель обучается на всех точках $T$; $\nu$ и $\gamma$ не подбираются.
2. Подбор порога — как у AR(2); в сетку кандидатов добавляется 0 — граница SVM; второй проход выполняется для каждого кандидата.

In [ ]:
OCSVM_NU = 0.1
# Граница SVM: s > 0 — точка вне области штатного хода.
OCSVM_CANONICAL_THRESHOLD = 0.0
OCSVM_CACHE_MB = 2000


def fit_ocsvm_on_train(train: pd.DataFrame) -> dict[str, object]:
    """One-Class SVM по нормированным признакам всех точек штатного хода обучающей выборки.

    Признаки и нормировка — как у DBSCAN: по обучающим отрезкам штатного хода (move_segments).

    Args:
        train: Обучающий DataFrame с колонками x, y, time и l.

    Returns:
        dict[str, object]: center, scale — робастная нормировка; points — набор T (M, 3); svm — обученная модель.
    """
    index, first = segments_layout(move_segments(train))
    before = np.where(first, -1, np.arange(len(index)) - 1)
    raw = dbscan_features(enu_xy_from_frame(train)[index], time_seconds_from_frame(train)[index], before)
    raw = raw[~np.isnan(raw[:, 0])]
    if len(raw) < 2:
        raise RuntimeError("Мало точек штатного хода для One-Class SVM.")
    center, scale = robust_params(raw)
    points = apply_robust(raw, center, scale)
    # gamma="scale": γ = 1 / (d · Var(T)) по всем значениям признаков.
    svm = OneClassSVM(kernel="rbf", gamma="scale", nu=OCSVM_NU, cache_size=OCSVM_CACHE_MB).fit(points)
    return {"center": center, "scale": scale, "points": points, "svm": svm}


def ocsvm_detect(
    frame: pd.DataFrame, model: dict[str, object], taus: np.ndarray
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Оценка s = -f(u) в два прохода, сразу для нескольких порогов.

    Второй проход — признаки по последним точкам, принятым первым проходом; оценка пересчитывается только у точек,
    признаки которых изменились, и только для ещё не встречавшихся признаков (cached_rows).

    Args:
        frame: DataFrame с колонками x, y и time.
        model: Результат fit_ocsvm_on_train.
        taus: Пороги; np.inf — без отвергнутых точек.

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray]: оценка второго прохода (C, N), NaN — точка без оценки;
        маска отвергнутых (C, N); оценка первого прохода (N,).
    """
    index, first = dbscan_segments(frame, np.ones(len(frame), dtype=bool))
    xy = enu_xy_from_frame(frame)[index]
    time_sec = time_seconds_from_frame(frame)[index]
    position = np.arange(len(index))
    first_features = apply_robust(
        dbscan_features(xy, time_sec, np.where(first, -1, position - 1)), model["center"], model["scale"]
    )
    scored = ~np.isnan(first_features[:, 0])
    first_scores = np.full(len(index), np.nan)
    first_scores[scored] = -model["svm"].decision_function(first_features[scored])
    thresholds = np.atleast_1d(np.asarray(taus, dtype=float))
    scores = np.full((len(thresholds), len(frame)), np.nan)
    rejected = np.zeros((len(thresholds), len(frame)), dtype=bool)
    # Оценки второго прохода повторяются при разных порогах: кэш по ключу признаков.
    cache: dict[bytes, object] = {}
    for row, tau in enumerate(thresholds):
        accepted = ~(first_scores > tau)
        last_accepted = np.maximum.accumulate(np.where(accepted, position, -1))
        before = np.where(first, -1, np.r_[-1, last_accepted[:-1]])
        second_features = apply_robust(dbscan_features(xy, time_sec, before), model["center"], model["scale"])
        second_scores = first_scores.copy()
        changed = scored & np.any(second_features != first_features, axis=1)
        if np.any(changed):
            changed_rows = np.flatnonzero(changed)
            second_scores[changed] = cached_rows(
                feature_keys(before, changed_rows),
                cache,
                lambda missing: -model["svm"].decision_function(second_features[changed_rows[missing]]),
            )
        scores[row, index[scored]] = second_scores[scored]
        rejected[row, index[scored]] = second_scores[scored] > tau
    first_full = np.full(len(frame), np.nan)
    first_full[index[scored]] = first_scores[scored]
    return scores, rejected, first_full


def select_ocsvm_threshold(frame: pd.DataFrame, model: dict[str, object], mcc_band: float) -> dict[str, float]:
    """Подбирает порог: общая сетка по оценке первого прохода и 0, два прохода на кандидата.

    Args:
        frame: Обучающий DataFrame (time, lat, lon, x, y, l).
        model: Результат fit_ocsvm_on_train.
        mcc_band: Допуск по MCC относительно лучшего значения.

    Returns:
        dict[str, float]: tau, train_mcc, train_f1, train_dl_m, best_mcc, band_size.
    """
    _, _, first = ocsvm_detect(frame, model, np.array([np.inf]))
    candidates = np.unique(np.r_[threshold_candidates(first[np.isfinite(first)]), OCSVM_CANONICAL_THRESHOLD])
    scores, rejected, _ = ocsvm_detect(frame, model, candidates)
    ground_truth_anomaly = frame["l"].astype(str).to_numpy() == "anomaly"
    records = [
        _threshold_record(float(tau), rejected[row], frame, ground_truth_anomaly, ~np.isnan(scores[row]))
        for row, tau in enumerate(candidates)
    ]
    return select_threshold_from_records(records, mcc_band)


ocsvm_scenarios = (
    (DATASET_NAME, dataset),
)

ocsvm_rows: list[dict[str, object]] = []
ocsvm_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in ocsvm_scenarios:
    train = bundle["train"]
    test = bundle["test"]
    anomaly_train = train["l"].astype(str).to_numpy() == "anomaly"

    print("=" * 72)
    print(f"One-Class SVM: {scenario_name}")
    started = time.perf_counter()
    def train_model() -> dict[str, object]:
        """Обучение, выбор порога и оценки train для печати медиан."""
        model = fit_ocsvm_on_train(train)
        selection = select_ocsvm_threshold(train, model, MCC_BAND)
        scores = ocsvm_detect(train, model, np.array([selection["tau"]]))[0]
        return {"model": model, "selection": selection, "train_scores": scores}
    state = load_or_train("ocsvm", train, train_model)
    ocsvm_model, selection, train_scores = state["model"], state["selection"], state["train_scores"]
    tau = np.array([selection["tau"]])
    if np.any(~anomaly_train) and np.any(anomaly_train):
        print(
            f"  медиана оценки на train: move {np.nanmedian(train_scores[0][~anomaly_train]):.4g}, "
            f"anomaly {np.nanmedian(train_scores[0][anomaly_train]):.4g}"
        )
    print(
        f"  ν {OCSVM_NU}, точек штатного хода {len(ocsvm_model['points'])}, "
        f"опорных векторов {len(ocsvm_model['svm'].support_)}; порог {tau[0]:.4g}: "
        f"MCC train {selection['train_mcc']:.4f} "
        f"(лучший {selection['best_mcc']:.4f}, в полосе {int(selection['band_size'])}), "
        f"DL train {selection['train_dl_m'] / 1000:.4f} км"
    )

    test_scores, test_rejected, _ = ocsvm_detect(test, ocsvm_model, tau)
    predicted_test = test_rejected[0]
    scored_test = ~np.isnan(test_scores[0])
    metrics = evaluate_test(test, predicted_test, scored_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м, без оценки {metrics['unscored_share']:.2%} "
        f"(время расчёта {elapsed:.1f} с)"
    )

    ocsvm_rows.append(test_report_row(scenario_name, metrics))
    ocsvm_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "scored_test": scored_test,
            "metrics": metrics,
        }
    )

ocsvm_summary = pd.DataFrame(ocsvm_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    ocsvm_summary.loc[:, report_columns],
    caption="One-Class SVM: сводные метрики на test",
)

if SHOW_PLOTS:
    _ocsvm_ideal_test = ocsvm_scenario_results[0]["test"]
    _ocsvm_map_tail = min(PLOT_TAIL, len(_ocsvm_ideal_test))
    print(f"\nИдеальная разметка: последние {nfmt(_ocsvm_map_tail)} точек теста (bbox: move + 5%)")
    _ocsvm_dummy_move = np.zeros(len(_ocsvm_ideal_test), dtype=bool)
    _ocsvm_truth_pred, _ocsvm_truth_gt = trajectory_series_pair(_ocsvm_ideal_test, _ocsvm_dummy_move)
    _ocsvm_truth_pred = slice_trajectory_series(_ocsvm_truth_pred, _ocsvm_map_tail)
    _ocsvm_truth_gt = slice_trajectory_series(_ocsvm_truth_gt, _ocsvm_map_tail)
    plot_track_maps(
        _ocsvm_truth_pred,
        _ocsvm_truth_gt,
        map_panel="ground_truth",
        title="One-Class SVM — идеальная разметка",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )

for item in ocsvm_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    if SHOW_PLOTS:
        print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
        prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"], item["scored_test"])
        prediction = slice_trajectory_series(prediction, tail)
        ground_truth = slice_trajectory_series(ground_truth, tail)
        plot_track_maps(
            prediction,
            ground_truth,
            map_panel="model",
            title=f"One-Class SVM — {item['name']}",
            figsize=MAP_FIGSIZE,
            annotate=False,
        )

## Robust PCA (Principal Component Pursuit)

Признаки, нормировка и второй проход — как у HMM.

### Постановка

Окно из подряд идущих точек раскладывается на часть, объяснимую типичным поведением штатного хода, и разреженное отклонение. Аномалия — точка, на которую приходится отклонение.

### Модель и оценка аномальности

Окно $W$ — вектор $\mathbf x_W=(\mathbf u_j,\ldots,\mathbf u_{j+n_W-1})$ нормированных признаков $n_W=16$ подряд идущих точек.

Типичное поведение — подпространство окон штатного хода обучающей выборки. Матрица $X$ этих окон (по строке на окно) раскладывается на низкоранговую и разреженную части:

$$\min_{L,S}\ \lVert L\rVert_*+\lambda\lVert S\rVert_1\quad\text{при}\quad X=L+S,\qquad \lambda=\frac{1}{\sqrt{\max(n,m)}},$$

где $\lVert L\rVert_*$ — сумма сингулярных чисел (ядерная норма), $\lVert S\rVert_1$ — сумма модулей элементов, $n\times m$ — размер $X$. Базис подпространства $U$ — правые сингулярные векторы $L$ с ненулевыми сингулярными числами; ранг определяется разложением. Единичные выбросы и ошибки разметки штатного хода уходят в $S$ и подпространство не искажают.

Оценка окна любой выборки — разреженный остаток проекции на фиксированное подпространство:

$$\mathbf c^*=\arg\min_{\mathbf c}\lVert\mathbf x_W-U\mathbf c\rVert_1,\qquad \mathbf s_W=\mathbf x_W-U\mathbf c^*,\qquad s_i=\max_{W\ni i}\lVert\mathbf s_W(i)\rVert,\qquad \hat a_i=\mathbf{1}[s_i>\tau],$$

где $\mathbf s_W(i)$ — три компоненты остатка, относящиеся к точке $i$; максимум — по окнам, содержащим точку. Это разложение $X=L+S$ при известном подпространстве: остаток разрежен без отдельного штрафа. Отклонение одной точки не объясняется комбинацией окон штатного хода и попадает в остаток на позициях этой точки; остаток обычной проекции размазал бы его по всему окну.

В отличие от точечных моделей, оценивается согласованность точки с соседями: аномальна последовательность, даже если значения признаков по отдельности обычны. Свойство модели: эпизод длиннее окна с обычной кинематикой не отличается от штатного хода; аномальны окна, захватывающие вход в эпизод и выход из него.

### Время и точки без оценки

Окна — по $n_W$ подряд идущих точек внутри отрезка между разрывами по времени больше 180 с, с шагом 1. Без оценки: первые две точки отрезка (как у HMM); все точки отрезка, в котором меньше $n_W+2$ точек.

### Обучение и выбор порога

1. Нормировка и окна штатного хода — по точкам штатного хода обучающей выборки, как у HMM; разложение $X=L+S$ — до сходимости.
2. $n_W$ не подбирается. Подбор порога — как у AR(2); второй проход выполняется для каждого кандидата.

In [ ]:
RPCA_WINDOW = 16
RPCA_MAX_ITER = 1000
RPCA_TOL = 1e-7
# Ранг L: сингулярные числа меньше этой доли наибольшего считаются нулевыми.
RPCA_RANK_TOL = 1e-6


def principal_component_pursuit(data: np.ndarray) -> tuple[np.ndarray, np.ndarray, int, bool]:
    """Разложение X = L + S: min ||L||_* + λ||S||_1, λ = 1/√max(n, m), inexact ALM до сходимости.

    Args:
        data: Матрица окон (n, m).

    Returns:
        tuple[np.ndarray, np.ndarray, int, bool]: L, S, число итераций, достигнута ли сходимость.
    """
    work = np.asarray(data, dtype=float)
    lam = 1.0 / np.sqrt(max(work.shape))
    spectral = np.linalg.norm(work, 2)
    mu = 1.25 / max(spectral, 1e-8)
    mu_max = mu * 1e7
    lagrange = work / max(spectral, np.max(np.abs(work)) / lam)
    sparse = np.zeros_like(work)
    low_rank = np.zeros_like(work)
    data_norm = max(np.linalg.norm(work, "fro"), 1e-8)
    for iteration in range(1, RPCA_MAX_ITER + 1):
        basis, singular, right = np.linalg.svd(work - sparse + lagrange / mu, full_matrices=False)
        low_rank = (basis * np.maximum(singular - 1.0 / mu, 0.0)) @ right
        residual = work - low_rank + lagrange / mu
        sparse = np.sign(residual) * np.maximum(np.abs(residual) - lam / mu, 0.0)
        error = work - low_rank - sparse
        lagrange = lagrange + mu * error
        if np.linalg.norm(error, "fro") / data_norm < RPCA_TOL:
            return low_rank, sparse, iteration, True
        mu = min(mu * 1.6, mu_max)
    return low_rank, sparse, RPCA_MAX_ITER, False


def fit_rpca_on_train(train: pd.DataFrame) -> dict[str, object]:
    """Нормировка и базис подпространства окон штатного хода обучающей выборки.

    Признаки и нормировка — как у DBSCAN: по обучающим отрезкам штатного хода (move_segments).

    Args:
        train: Обучающий DataFrame с колонками x, y, time и l.

    Returns:
        dict[str, object]: center, scale; basis — U (3·n_W, r); windows — число окон; iterations, converged.
    """
    index, first = segments_layout(move_segments(train))
    before = np.where(first, -1, np.arange(len(index)) - 1)
    raw = dbscan_features(enu_xy_from_frame(train)[index], time_seconds_from_frame(train)[index], before)
    scored = ~np.isnan(raw[:, 0])
    starts = rpca_window_starts(first, scored, RPCA_WINDOW)
    if len(starts) == 0:
        raise RuntimeError("Нет окна штатного хода для Robust PCA.")
    center, scale = robust_params(raw[scored])
    features = apply_robust(raw, center, scale)
    matrix = features[starts[:, None] + np.arange(RPCA_WINDOW)].reshape(len(starts), -1)
    low_rank, _, iterations, converged = principal_component_pursuit(matrix)
    if not converged:
        print(f"  предупреждение: разложение Robust PCA не сошлось за {RPCA_MAX_ITER} итераций")
    _, singular, right = np.linalg.svd(low_rank, full_matrices=False)
    rank = int(np.sum(singular > RPCA_RANK_TOL * singular[0]))
    return {
        "center": center,
        "scale": scale,
        "basis": right[:rank].T,
        "windows": len(starts),
        "iterations": iterations,
        "converged": converged,
    }


def rpca_window_residual(basis: np.ndarray, windows: np.ndarray) -> np.ndarray:
    """Остаток s = x - U c*, c* = argmin ||x - U c||_1, для каждого окна — точно, линейным программированием.

    Задача: min Σ t при -t ≤ x - U c ≤ t по переменным (c, t).

    Args:
        basis: Ортонормированный базис U (m, r).
        windows: Окна (k, m).

    Returns:
        np.ndarray: Остатки (k, m).
    """
    size, rank = basis.shape
    cost = np.r_[np.zeros(rank), np.ones(size)]
    bound_matrix = np.block([[basis, -np.eye(size)], [-basis, -np.eye(size)]])
    bounds = [(None, None)] * rank + [(0.0, None)] * size
    residual = np.empty_like(windows, dtype=float)
    for row, window in enumerate(windows):
        solution = linprog(cost, A_ub=bound_matrix, b_ub=np.r_[window, -window], bounds=bounds, method="highs")
        if not solution.success:
            raise RuntimeError(f"Линейное программирование Robust PCA не решено: {solution.message}")
        residual[row] = window - basis @ solution.x[:rank]
    return residual


def rpca_first_pass(frame: pd.DataFrame, model: dict[str, object]) -> dict[str, np.ndarray]:
    """Окна, признаки и остатки первого прохода — общие для всех порогов и повторных вызовов.

    Args:
        frame: DataFrame с колонками x, y и time.
        model: Результат fit_rpca_on_train.

    Returns:
        dict[str, np.ndarray]: index, first, xy, time_sec, features, cover, scored, residual.
    """
    index, first = dbscan_segments(frame, np.ones(len(frame), dtype=bool))
    xy = enu_xy_from_frame(frame)[index]
    time_sec = time_seconds_from_frame(frame)[index]
    features = apply_robust(
        dbscan_features(xy, time_sec, np.where(first, -1, np.arange(len(index)) - 1)), model["center"], model["scale"]
    )
    starts = rpca_window_starts(first, ~np.isnan(features[:, 0]), RPCA_WINDOW)
    cover = starts[:, None] + np.arange(RPCA_WINDOW)
    scored = np.zeros(len(index), dtype=bool)
    scored[cover.ravel()] = True
    residual = rpca_window_residual(model["basis"], features[cover].reshape(len(starts), -1))
    return {
        "index": index,
        "first": first,
        "xy": xy,
        "time_sec": time_sec,
        "features": features,
        "cover": cover,
        "scored": scored,
        "residual": residual,
    }


def rpca_detect(
    frame: pd.DataFrame,
    model: dict[str, object],
    taus: np.ndarray,
    first_pass: Optional[dict[str, np.ndarray]] = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Оценка точки — максимум нормы её компонент остатка по окнам, в два прохода, сразу для нескольких порогов.

    Второй проход — признаки по последним точкам, принятым первым проходом; остаток пересчитывается только у окон
    с изменёнными признаками и только для ещё не встречавшихся окон (cached_rows по окну и ключам признаков его точек).

    Args:
        frame: DataFrame с колонками x, y и time.
        model: Результат fit_rpca_on_train.
        taus: Пороги; np.inf — без отвергнутых точек.
        first_pass: Результат rpca_first_pass для ``frame``; None — считается заново.

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray]: оценка второго прохода (C, N), NaN — точка без оценки;
        маска отвергнутых (C, N); оценка первого прохода (N,).
    """
    prepared = rpca_first_pass(frame, model) if first_pass is None else first_pass
    index, first, cover, scored = prepared["index"], prepared["first"], prepared["cover"], prepared["scored"]
    xy, time_sec, first_features = prepared["xy"], prepared["time_sec"], prepared["features"]
    position = np.arange(len(index))

    def window_norms(residual: np.ndarray) -> np.ndarray:
        return np.linalg.norm(residual.reshape(len(residual), RPCA_WINDOW, -1), axis=2)

    def point_scores(norms: np.ndarray) -> np.ndarray:
        result = np.full(len(index), -np.inf)
        np.maximum.at(result, cover.ravel(), norms.ravel())
        return np.where(scored, result, np.nan)

    first_norms = window_norms(prepared["residual"])
    first_scores = point_scores(first_norms)
    thresholds = np.atleast_1d(np.asarray(taus, dtype=float))
    scores = np.full((len(thresholds), len(frame)), np.nan)
    rejected = np.zeros((len(thresholds), len(frame)), dtype=bool)
    # Оценки второго прохода повторяются при разных порогах: кэш по ключу признаков.
    cache: dict[bytes, object] = {}
    for row, tau in enumerate(thresholds):
        accepted = ~(first_scores > tau)
        last_accepted = np.maximum.accumulate(np.where(accepted, position, -1))
        before = np.where(first, -1, np.r_[-1, last_accepted[:-1]])
        second_features = apply_robust(dbscan_features(xy, time_sec, before), model["center"], model["scale"])
        changed_point = np.any(second_features != first_features, axis=1) & ~np.isnan(first_features[:, 0])
        changed = changed_point[cover].any(axis=1)
        norms = first_norms.copy()
        if np.any(changed):
            changed_windows = np.flatnonzero(changed)
            points = cover[changed_windows]
            point_keys = feature_keys(before, points.ravel())[:, 1:].reshape(len(points), -1)
            norms[changed_windows] = cached_rows(
                np.column_stack([changed_windows, point_keys]),
                cache,
                lambda missing: window_norms(
                    rpca_window_residual(
                        model["basis"], second_features[points[missing]].reshape(int(missing.sum()), -1)
                    )
                ),
            )
        second_scores = point_scores(norms)
        scores[row, index[scored]] = second_scores[scored]
        rejected[row, index[scored]] = second_scores[scored] > tau
    first_full = np.full(len(frame), np.nan)
    first_full[index[scored]] = first_scores[scored]
    return scores, rejected, first_full


def select_rpca_threshold(
    frame: pd.DataFrame, model: dict[str, object], mcc_band: float, first_pass: dict[str, np.ndarray]
) -> dict[str, float]:
    """Подбирает порог: общая сетка по оценке первого прохода, два прохода на кандидата.

    Args:
        frame: Обучающий DataFrame (time, lat, lon, x, y, l).
        model: Результат fit_rpca_on_train.
        mcc_band: Допуск по MCC относительно лучшего значения.
        first_pass: Результат rpca_first_pass для ``frame``.

    Returns:
        dict[str, float]: tau, train_mcc, train_f1, train_dl_m, best_mcc, band_size.
    """
    _, _, first = rpca_detect(frame, model, np.array([np.inf]), first_pass)
    candidates = threshold_candidates(first[np.isfinite(first)])
    scores, rejected, _ = rpca_detect(frame, model, candidates, first_pass)
    ground_truth_anomaly = frame["l"].astype(str).to_numpy() == "anomaly"
    records = [
        _threshold_record(float(tau), rejected[row], frame, ground_truth_anomaly, ~np.isnan(scores[row]))
        for row, tau in enumerate(candidates)
    ]
    return select_threshold_from_records(records, mcc_band)


rpca_scenarios = (
    (DATASET_NAME, dataset),
)

rpca_rows: list[dict[str, object]] = []
rpca_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in rpca_scenarios:
    train = bundle["train"]
    test = bundle["test"]
    anomaly_train = train["l"].astype(str).to_numpy() == "anomaly"

    print("=" * 72)
    print(f"Robust PCA: {scenario_name}")
    started = time.perf_counter()
    def train_model() -> dict[str, object]:
        """Обучение, выбор порога и оценки train для печати медиан."""
        model = fit_rpca_on_train(train)
        first_pass = rpca_first_pass(train, model)
        selection = select_rpca_threshold(train, model, MCC_BAND, first_pass)
        scores = rpca_detect(train, model, np.array([selection["tau"]]), first_pass)[0]
        return {"model": model, "selection": selection, "train_scores": scores}
    state = load_or_train("rpca", train, train_model)
    rpca_model, selection, train_scores = state["model"], state["selection"], state["train_scores"]
    tau = np.array([selection["tau"]])
    if np.any(~anomaly_train) and np.any(anomaly_train):
        print(
            f"  медиана оценки на train: move {np.nanmedian(train_scores[0][~anomaly_train]):.4g}, "
            f"anomaly {np.nanmedian(train_scores[0][anomaly_train]):.4g}"
        )
    print(
        f"  окон штатного хода {rpca_model['windows']}, ранг {rpca_model['basis'].shape[1]} из "
        f"{rpca_model['basis'].shape[0]}, итераций разложения {rpca_model['iterations']}; порог {tau[0]:.4g}: "
        f"MCC train {selection['train_mcc']:.4f} "
        f"(лучший {selection['best_mcc']:.4f}, в полосе {int(selection['band_size'])}), "
        f"DL train {selection['train_dl_m'] / 1000:.4f} км"
    )

    test_scores, test_rejected, _ = rpca_detect(test, rpca_model, tau)
    predicted_test = test_rejected[0]
    scored_test = ~np.isnan(test_scores[0])
    metrics = evaluate_test(test, predicted_test, scored_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м, без оценки {metrics['unscored_share']:.2%} "
        f"(время расчёта {elapsed:.1f} с)"
    )

    rpca_rows.append(test_report_row(scenario_name, metrics))
    rpca_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "scored_test": scored_test,
            "metrics": metrics,
        }
    )

rpca_summary = pd.DataFrame(rpca_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    rpca_summary.loc[:, report_columns],
    caption="Robust PCA: сводные метрики на test",
)

if SHOW_PLOTS:
    _rpca_ideal_test = rpca_scenario_results[0]["test"]
    _rpca_map_tail = min(PLOT_TAIL, len(_rpca_ideal_test))
    print(f"\nИдеальная разметка: последние {nfmt(_rpca_map_tail)} точек теста (bbox: move + 5%)")
    _rpca_dummy_move = np.zeros(len(_rpca_ideal_test), dtype=bool)
    _rpca_truth_pred, _rpca_truth_gt = trajectory_series_pair(_rpca_ideal_test, _rpca_dummy_move)
    _rpca_truth_pred = slice_trajectory_series(_rpca_truth_pred, _rpca_map_tail)
    _rpca_truth_gt = slice_trajectory_series(_rpca_truth_gt, _rpca_map_tail)
    plot_track_maps(
        _rpca_truth_pred,
        _rpca_truth_gt,
        map_panel="ground_truth",
        title="Robust PCA — идеальная разметка",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )

for item in rpca_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    if SHOW_PLOTS:
        print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
        prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"], item["scored_test"])
        prediction = slice_trajectory_series(prediction, tail)
        ground_truth = slice_trajectory_series(ground_truth, tail)
        plot_track_maps(
            prediction,
            ground_truth,
            map_panel="model",
            title=f"Robust PCA — {item['name']}",
            figsize=MAP_FIGSIZE,
            annotate=False,
        )

## LSTM-автоэнкодер (EncDec-AD)

Признаки, нормировка и второй проход — как у HMM; окна и точки без оценки — как у Robust PCA.

### Постановка

Сеть кодировщик–декодер на LSTM, обученная на окнах штатного хода обучающей выборки, восстанавливает окно признаков. Аномалия — точка, признаки которой восстанавливаются плохо.

### Модель и оценка аномальности

Кодировщик LSTM читает окно $\mathbf x_W=(\mathbf u_j,\ldots,\mathbf u_{j+n_W-1})$ и сжимает его в скрытое состояние размера 32; декодер LSTM восстанавливает из этого состояния окно $\hat{\mathbf x}_W$. Для точки $i$ на позиции $p$ окна $W$:

$$\mathbf e_W(i)=\mathbf u_i-\hat{\mathbf u}_W(i),\qquad d_W(i)=\sum_{k=1}^{3}\frac{\bigl(e_{W,k}(i)-\mu_{p,k}\bigr)^2}{\sigma_{p,k}^2},$$

$$s_i=\min_{W\ni i}d_W(i),\qquad \hat a_i=\mathbf{1}[s_i>\tau],$$

где $\hat{\mathbf u}_W(i)$ — восстановленные признаки точки $i$, $\mu_{p,k}$ и $\sigma_{p,k}^2$ — среднее и дисперсия ошибки признака $k$ на позиции $p$ по калибровочным окнам; минимум — по окнам, содержащим точку.

Сеть восстанавливает окно из сжатого состояния, поэтому выброс портит восстановление и соседних точек того же окна. Выброс восстанавливается плохо во всех своих окнах, а у соседней точки есть окно без выброса — минимум по окнам относит отклонение к точке выброса.

В отличие от Robust PCA, норма — нелинейная модель последовательности, а не линейное подпространство. Свойство модели — как у Robust PCA: эпизод длиннее окна с обычной кинематикой не отличается от штатного хода.

### Обучение и выбор порога

1. Нормировка — как у HMM. Окна штатного хода обучающей выборки упорядочиваются по времени: последние 15% — калибровочные, окна, пересекающиеся с ними, отбрасываются; из остальных берётся до 8000 случайных окон для обучения.
2. Сеть обучается по среднему квадрату ошибки восстановления (Adam), до 300 эпох с ранней остановкой: 10 эпох без улучшения ошибки на калибровочных окнах больше чем на $10^{-4}$ (относительно), веса — лучшей эпохи; скорость обучения уменьшается вдвое после 3 эпох без улучшения; при остановке по лимиту эпох выводится предупреждение; начальное значение генератора фиксировано, вычисления детерминированы. Размер сети и число эпох не подбираются.
3. $\mu_{p,k}$, $\sigma_{p,k}^2$ — по калибровочным окнам. Подбор порога — как у AR(2); второй проход выполняется для каждого кандидата.

In [ ]:
LSTM_WINDOW = 16
LSTM_HIDDEN = 32
LSTM_N_FEATURES = 3


class LSTMAutoencoder(nn.Module):
    """Кодировщик–декодер: восстановление окна из финального состояния LSTM."""

    def __init__(self, n_features: int, hidden: int):
        """Создаёт кодировщик и декодер LSTM.

        Args:
            n_features: Число признаков точки.
            hidden: Размер скрытого состояния.
        """
        super().__init__()
        self.encoder = nn.LSTM(n_features, hidden, batch_first=True)
        self.decoder = nn.LSTM(n_features, hidden, batch_first=True)
        self.proj = nn.Linear(hidden, n_features)

    def forward(self, sequence: torch.Tensor) -> torch.Tensor:
        """Восстанавливает окно из финального состояния кодировщика.

        Args:
            sequence: Окно shape (batch, length, n_features).

        Returns:
            torch.Tensor: Восстановленное окно той же формы.
        """
        _, state = self.encoder(sequence)
        decoded, _ = self.decoder(torch.zeros_like(sequence), state)
        return self.proj(decoded)


def _lstm_reconstruct(model: nn.Module, windows: np.ndarray) -> np.ndarray:
    """Восстановленные окна (k, n_W, d) без градиентов."""
    if len(windows) == 0:
        return np.empty_like(windows)
    return batched_model_forward(model, windows.astype(np.float32), lambda m, b: m(b), TORCH_INFER_BATCH)


def _lstm_mse(model: nn.Module, windows: np.ndarray) -> float:
    """Средний квадрат ошибки восстановления окон."""
    return float(np.mean((_lstm_reconstruct(model, windows) - windows) ** 2))


def fit_lstm_on_train(train: pd.DataFrame) -> dict[str, object]:
    """Обучает LSTM-автоэнкодер на окнах штатного хода и калибрует ошибку по последним по времени окнам.

    Признаки и нормировка — как у DBSCAN; окна — как у Robust PCA. Последние NN_CALIB_FRACTION окон по времени —
    калибровочные, пересекающиеся с ними окна отбрасываются, из остальных — до N_WINDOWS случайных для обучения.
    Ранняя остановка — по ошибке на калибровочных окнах.

    Args:
        train: Обучающий DataFrame с колонками x, y, time и l.

    Returns:
        dict[str, object]: center, scale, model, error_mean и error_var (n_W, d), epochs, n_train, n_calib.
    """
    index, first = segments_layout(move_segments(train))
    before = np.where(first, -1, np.arange(len(index)) - 1)
    raw = dbscan_features(enu_xy_from_frame(train)[index], time_seconds_from_frame(train)[index], before)
    scored = ~np.isnan(raw[:, 0])
    starts = rpca_window_starts(first, scored, LSTM_WINDOW)
    if len(starts) < 20:
        raise RuntimeError("Мало окон штатного хода для LSTM-автоэнкодера.")
    center, scale = robust_params(raw[scored])
    features = apply_robust(raw, center, scale)
    calib_starts = starts[int(len(starts) * (1.0 - NN_CALIB_FRACTION)) :]
    fit_starts = starts[starts + LSTM_WINDOW <= calib_starts[0]]
    fit_starts = take_window_starts(fit_starts, N_WINDOWS, SEED)
    offsets = np.arange(LSTM_WINDOW)
    fit_windows = features[fit_starts[:, None] + offsets].astype(np.float32)
    calib_windows = features[calib_starts[:, None] + offsets].astype(np.float32)

    torch.manual_seed(SEED)
    model = LSTMAutoencoder(LSTM_N_FEATURES, LSTM_HIDDEN).to(DEVICE)
    loader = DataLoader(
        TensorDataset(torch.from_numpy(fit_windows)),
        batch_size=TORCH_BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    training = train_with_early_stopping(
        model,
        loader,
        lambda net, batch: ((net(batch[0]) - batch[0]) ** 2).mean(),
        lambda net: _lstm_mse(net, calib_windows),
    )
    error = _lstm_reconstruct(model, calib_windows) - calib_windows
    return {
        "center": center,
        "scale": scale,
        "model": model,
        "error_mean": error.mean(axis=0),
        "error_var": np.maximum(error.var(axis=0), 1e-12),
        "epochs": training["epochs"],
        "calib_loss": training["calib_loss"],
        "n_train": len(fit_starts),
        "n_calib": len(calib_starts),
    }


def lstm_detect(
    frame: pd.DataFrame, model: dict[str, object], taus: np.ndarray
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Оценка точки — минимум по окнам нормированной ошибки её признаков, в два прохода, для нескольких порогов.

    Второй проход — признаки по последним точкам, принятым первым проходом; восстанавливаются только окна
    с изменёнными признаками.

    Args:
        frame: DataFrame с колонками x, y и time.
        model: Результат fit_lstm_on_train.
        taus: Пороги; np.inf — без отвергнутых точек.

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray]: оценка второго прохода (C, N), NaN — точка без оценки;
        маска отвергнутых (C, N); оценка первого прохода (N,).
    """
    index, first = dbscan_segments(frame, np.ones(len(frame), dtype=bool))
    xy = enu_xy_from_frame(frame)[index]
    time_sec = time_seconds_from_frame(frame)[index]
    position = np.arange(len(index))
    first_features = apply_robust(
        dbscan_features(xy, time_sec, np.where(first, -1, position - 1)), model["center"], model["scale"]
    )
    starts = rpca_window_starts(first, ~np.isnan(first_features[:, 0]), LSTM_WINDOW)
    cover = starts[:, None] + np.arange(LSTM_WINDOW)
    scored = np.zeros(len(index), dtype=bool)
    scored[cover.ravel()] = True

    def window_errors(features: np.ndarray, rows: np.ndarray) -> np.ndarray:
        windows = features[cover[rows]].astype(np.float32)
        error = _lstm_reconstruct(model["model"], windows) - windows
        return np.sum((error - model["error_mean"]) ** 2 / model["error_var"], axis=2)

    def point_scores(per_window: np.ndarray) -> np.ndarray:
        result = np.full(len(index), np.inf)
        np.minimum.at(result, cover.ravel(), per_window.ravel())
        return np.where(scored, result, np.nan)

    first_errors = window_errors(first_features, np.arange(len(starts)))
    first_scores = point_scores(first_errors)
    thresholds = np.atleast_1d(np.asarray(taus, dtype=float))
    scores = np.full((len(thresholds), len(frame)), np.nan)
    rejected = np.zeros((len(thresholds), len(frame)), dtype=bool)
    for row, tau in enumerate(thresholds):
        accepted = ~(first_scores > tau)
        last_accepted = np.maximum.accumulate(np.where(accepted, position, -1))
        before = np.where(first, -1, np.r_[-1, last_accepted[:-1]])
        second_features = apply_robust(dbscan_features(xy, time_sec, before), model["center"], model["scale"])
        changed_point = np.any(second_features != first_features, axis=1) & ~np.isnan(first_features[:, 0])
        changed = np.flatnonzero(changed_point[cover].any(axis=1))
        errors = first_errors.copy()
        if len(changed):
            errors[changed] = window_errors(second_features, changed)
        second_scores = point_scores(errors)
        scores[row, index[scored]] = second_scores[scored]
        rejected[row, index[scored]] = second_scores[scored] > tau
    first_full = np.full(len(frame), np.nan)
    first_full[index[scored]] = first_scores[scored]
    return scores, rejected, first_full


def select_lstm_threshold(frame: pd.DataFrame, model: dict[str, object], mcc_band: float) -> dict[str, float]:
    """Подбирает порог: общая сетка по оценке первого прохода, два прохода на кандидата.

    Args:
        frame: Обучающий DataFrame (time, lat, lon, x, y, l).
        model: Результат fit_lstm_on_train.
        mcc_band: Допуск по MCC относительно лучшего значения.

    Returns:
        dict[str, float]: tau, train_mcc, train_f1, train_dl_m, best_mcc, band_size.
    """
    _, _, first = lstm_detect(frame, model, np.array([np.inf]))
    candidates = threshold_candidates(first[np.isfinite(first)])
    scores, rejected, _ = lstm_detect(frame, model, candidates)
    ground_truth_anomaly = frame["l"].astype(str).to_numpy() == "anomaly"
    records = [
        _threshold_record(float(tau), rejected[row], frame, ground_truth_anomaly, ~np.isnan(scores[row]))
        for row, tau in enumerate(candidates)
    ]
    return select_threshold_from_records(records, mcc_band)


lstm_scenarios = (
    (DATASET_NAME, dataset),
)

lstm_rows: list[dict[str, object]] = []
lstm_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in lstm_scenarios:
    train = bundle["train"]
    test = bundle["test"]
    anomaly_train = train["l"].astype(str).to_numpy() == "anomaly"

    print("=" * 72)
    print(f"LSTM-автоэнкодер: {scenario_name}")
    started = time.perf_counter()
    def train_model() -> dict[str, object]:
        """Обучение, выбор порога и оценки train для печати медиан."""
        model = fit_lstm_on_train(train)
        selection = select_lstm_threshold(train, model, MCC_BAND)
        scores = lstm_detect(train, model, np.array([selection["tau"]]))[0]
        return {"model": model, "selection": selection, "train_scores": scores}
    state = load_or_train("lstm", train, train_model)
    lstm_model, selection, train_scores = state["model"], state["selection"], state["train_scores"]
    tau = np.array([selection["tau"]])
    if np.any(~anomaly_train) and np.any(anomaly_train):
        print(
            f"  медиана оценки на train: move {np.nanmedian(train_scores[0][~anomaly_train]):.4g}, "
            f"anomaly {np.nanmedian(train_scores[0][anomaly_train]):.4g}"
        )
    print(
        f"  окон обучения {lstm_model['n_train']}, калибровочных {lstm_model['n_calib']}, "
        f"эпох {lstm_model['epochs']} (ошибка калибровки {lstm_model['calib_loss']:.4g}); порог {tau[0]:.4g}: "
        f"MCC train {selection['train_mcc']:.4f} "
        f"(лучший {selection['best_mcc']:.4f}, в полосе {int(selection['band_size'])}), "
        f"DL train {selection['train_dl_m'] / 1000:.4f} км"
    )

    test_scores, test_rejected, _ = lstm_detect(test, lstm_model, tau)
    predicted_test = test_rejected[0]
    scored_test = ~np.isnan(test_scores[0])
    metrics = evaluate_test(test, predicted_test, scored_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м, без оценки {metrics['unscored_share']:.2%} "
        f"(время расчёта {elapsed:.1f} с)"
    )

    lstm_rows.append(test_report_row(scenario_name, metrics))
    lstm_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "scored_test": scored_test,
            "metrics": metrics,
        }
    )

lstm_summary = pd.DataFrame(lstm_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    lstm_summary.loc[:, report_columns],
    caption="LSTM-автоэнкодер: сводные метрики на test",
)

if SHOW_PLOTS:
    _lstm_ideal_test = lstm_scenario_results[0]["test"]
    _lstm_map_tail = min(PLOT_TAIL, len(_lstm_ideal_test))
    print(f"\nИдеальная разметка: последние {nfmt(_lstm_map_tail)} точек теста (bbox: move + 5%)")
    _lstm_dummy_move = np.zeros(len(_lstm_ideal_test), dtype=bool)
    _lstm_truth_pred, _lstm_truth_gt = trajectory_series_pair(_lstm_ideal_test, _lstm_dummy_move)
    _lstm_truth_pred = slice_trajectory_series(_lstm_truth_pred, _lstm_map_tail)
    _lstm_truth_gt = slice_trajectory_series(_lstm_truth_gt, _lstm_map_tail)
    plot_track_maps(
        _lstm_truth_pred,
        _lstm_truth_gt,
        map_panel="ground_truth",
        title="LSTM-автоэнкодер — идеальная разметка",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )

for item in lstm_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    if SHOW_PLOTS:
        print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
        prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"], item["scored_test"])
        prediction = slice_trajectory_series(prediction, tail)
        ground_truth = slice_trajectory_series(ground_truth, tail)
        plot_track_maps(
            prediction,
            ground_truth,
            map_panel="model",
            title=f"LSTM-автоэнкодер — {item['name']}",
            figsize=MAP_FIGSIZE,
            annotate=False,
        )

## Temporal Convolutional Network (TCN)

Признаки, нормировка и второй проход — как у HMM; обучение и калибровка — как у LSTM-автоэнкодера.

### Постановка

Причинная свёрточная сеть по признакам предыдущих принятых точек прогнозирует признаки следующей точки. Аномалия — точка, признаки которой далеки от прогноза.

### Модель и оценка аномальности

$$\hat{\mathbf u}_i=g\bigl(\mathbf u_{i-15},\ldots,\mathbf u_{i-1}\bigr),\qquad s_i=\sum_{k=1}^{3}\frac{\bigl(u_{i,k}-\hat u_{i,k}-\mu_k\bigr)^2}{\sigma_k^2},\qquad \hat a_i=\mathbf{1}[s_i>\tau],$$

где $g$ — сеть из трёх причинных свёрточных уровней (ядро 3, расширения 1, 2, 4, 32 канала), её поле зрения — ровно 15 предыдущих точек; $\mu_k$, $\sigma_k^2$ — среднее и дисперсия ошибки прогноза признака $k$ по калибровочным окнам. Шаг по времени до прогнозируемой точки входит в её признаки: скорость и ускорения нормированы по $\Delta t$.

В истории — только принятые точки: история — 15 последних принятых точек отрезка, отвергнутая точка в неё не входит.

В отличие от LSTM-автоэнкодера, модель прогнозирует следующую точку по прошлому, а не восстанавливает окно; в отличие от AR и фильтра Калмана — нелинейная модель по признакам, а не линейная модель положения.

### Время и точки без оценки

Без оценки: первые 17 точек отрезка (2 — на признаки, 15 — на историю); из точек с одинаковым временем остаётся первая.

### Обучение и выбор порога

1. Окно обучения — 15 точек истории и прогнозируемая точка штатного хода; калибровка, число окон, обучение и ранняя остановка — как у LSTM-автоэнкодера.
2. $\mu_k$, $\sigma_k^2$ — по калибровочным окнам. Подбор порога — как у AR(2); второй проход выполняется для каждого кандидата.

In [ ]:
TCN_HISTORY = 15
TCN_CHANNELS = 32
TCN_LEVELS = 3
TCN_KERNEL = 3
TCN_N_FEATURES = 3


class CausalTCN(nn.Module):
    """Причинная TCN: прогноз признаков следующей точки по истории."""

    def __init__(self, n_features: int, channels: int, levels: int, kernel: int):
        """Создаёт стек причинных свёрточных уровней с расширениями 1, 2, 4, ….

        Args:
            n_features: Число признаков точки.
            channels: Число каналов.
            levels: Число уровней.
            kernel: Размер ядра свёртки.
        """
        super().__init__()
        blocks, pads = [], []
        in_channels, dilation = n_features, 1
        for _ in range(levels):
            blocks.append(nn.Conv1d(in_channels, channels, kernel, dilation=dilation))
            pads.append((kernel - 1) * dilation)
            in_channels, dilation = channels, dilation * 2
        self.blocks = nn.ModuleList(blocks)
        self.pads = pads
        self.head = nn.Linear(channels, n_features)

    def forward(self, sequence: torch.Tensor) -> torch.Tensor:
        """Прогнозирует признаки следующей точки.

        Args:
            sequence: История shape (batch, length, n_features).

        Returns:
            torch.Tensor: Прогноз shape (batch, n_features).
        """
        hidden = sequence.transpose(1, 2)
        for conv, pad in zip(self.blocks, self.pads):
            hidden = torch.relu(conv(nn.functional.pad(hidden, (pad, 0))))
        return self.head(hidden[:, :, -1])


def _tcn_predict(model: nn.Module, history: np.ndarray) -> np.ndarray:
    """Прогноз (k, d) по историям (k, TCN_HISTORY, d) без градиентов."""
    if len(history) == 0:
        return np.empty((0, TCN_N_FEATURES), dtype=np.float32)
    return batched_model_forward(model, history.astype(np.float32), lambda m, b: m(b), TORCH_INFER_BATCH)


def fit_tcn_on_train(train: pd.DataFrame) -> dict[str, object]:
    """Обучает TCN на окнах штатного хода (15 точек истории и прогнозируемая точка), калибрует остаток.

    Признаки и нормировка — как у DBSCAN; калибровка, число окон и ранняя остановка — как у LSTM-автоэнкодера.

    Args:
        train: Обучающий DataFrame с колонками x, y, time и l.

    Returns:
        dict[str, object]: center, scale, model, error_mean и error_var (d,), epochs, calib_loss, n_train, n_calib.
    """
    index, first = segments_layout(move_segments(train))
    before = np.where(first, -1, np.arange(len(index)) - 1)
    raw = dbscan_features(enu_xy_from_frame(train)[index], time_seconds_from_frame(train)[index], before)
    scored = ~np.isnan(raw[:, 0])
    starts = rpca_window_starts(first, scored, TCN_HISTORY + 1)
    if len(starts) < 20:
        raise RuntimeError("Мало окон штатного хода для TCN.")
    center, scale = robust_params(raw[scored])
    features = apply_robust(raw, center, scale).astype(np.float32)
    calib_starts = starts[int(len(starts) * (1.0 - NN_CALIB_FRACTION)) :]
    fit_starts = starts[starts + TCN_HISTORY + 1 <= calib_starts[0]]
    fit_starts = take_window_starts(fit_starts, N_WINDOWS, SEED)
    offsets = np.arange(TCN_HISTORY)

    def pairs(window_starts: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        return features[window_starts[:, None] + offsets], features[window_starts + TCN_HISTORY]

    fit_history, fit_target = pairs(fit_starts)
    calib_history, calib_target = pairs(calib_starts)
    torch.manual_seed(SEED)
    model = CausalTCN(TCN_N_FEATURES, TCN_CHANNELS, TCN_LEVELS, TCN_KERNEL).to(DEVICE)
    loader = DataLoader(
        TensorDataset(torch.from_numpy(fit_history), torch.from_numpy(fit_target)),
        batch_size=TORCH_BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    training = train_with_early_stopping(
        model,
        loader,
        lambda net, batch: ((net(batch[0]) - batch[1]) ** 2).mean(),
        lambda net: float(np.mean((_tcn_predict(net, calib_history) - calib_target) ** 2)),
    )
    error = calib_target - _tcn_predict(model, calib_history)
    return {
        "center": center,
        "scale": scale,
        "model": model,
        "error_mean": error.mean(axis=0),
        "error_var": np.maximum(error.var(axis=0), 1e-12),
        "epochs": training["epochs"],
        "calib_loss": training["calib_loss"],
        "n_train": len(fit_starts),
        "n_calib": len(calib_starts),
    }


def _tcn_scores(model: dict[str, object], history: np.ndarray, target: np.ndarray) -> np.ndarray:
    """Нормированный квадрат ошибки прогноза признаков, сумма по признакам."""
    error = target - _tcn_predict(model["model"], history)
    return np.sum((error - model["error_mean"]) ** 2 / model["error_var"], axis=1)


def tcn_detect(
    frame: pd.DataFrame, model: dict[str, object], taus: np.ndarray
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Оценка точки — нормированная ошибка прогноза её признаков, в два прохода, сразу для нескольких порогов.

    Первый проход — история из 15 предыдущих точек отрезка. Второй — история из 15 последних точек отрезка,
    принятых первым проходом, признаки по принятым точкам; прогноз пересчитывается только у точек, история или
    признаки которых изменились. Если принятых точек в отрезке до точки меньше 15, остаётся оценка первого прохода.

    Args:
        frame: DataFrame с колонками x, y и time.
        model: Результат fit_tcn_on_train.
        taus: Пороги; np.inf — без отвергнутых точек.

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray]: оценка второго прохода (C, N), NaN — точка без оценки;
        маска отвергнутых (C, N); оценка первого прохода (N,).
    """
    index, first = dbscan_segments(frame, np.ones(len(frame), dtype=bool))
    xy = enu_xy_from_frame(frame)[index]
    time_sec = time_seconds_from_frame(frame)[index]
    position = np.arange(len(index))
    segment = np.cumsum(first)
    first_features = apply_robust(
        dbscan_features(xy, time_sec, np.where(first, -1, position - 1)), model["center"], model["scale"]
    )
    starts = rpca_window_starts(first, ~np.isnan(first_features[:, 0]), TCN_HISTORY + 1)
    targets = starts + TCN_HISTORY
    first_history = starts[:, None] + np.arange(TCN_HISTORY)
    first_target_scores = _tcn_scores(model, first_features[first_history], first_features[targets])
    first_scores = np.full(len(index), np.nan)
    first_scores[targets] = first_target_scores
    thresholds = np.atleast_1d(np.asarray(taus, dtype=float))
    scores = np.full((len(thresholds), len(frame)), np.nan)
    rejected = np.zeros((len(thresholds), len(frame)), dtype=bool)
    for row, tau in enumerate(thresholds):
        accepted = ~(first_scores > tau)
        last_accepted = np.maximum.accumulate(np.where(accepted, position, -1))
        before = np.where(first, -1, np.r_[-1, last_accepted[:-1]])
        second_features = apply_robust(dbscan_features(xy, time_sec, before), model["center"], model["scale"])
        # История — 15 последних принятых точек с признаками до прогнозируемой точки в том же отрезке.
        usable = np.flatnonzero(accepted & ~np.isnan(second_features[:, 0]))
        count_before = np.searchsorted(usable, targets)
        has_history = count_before >= TCN_HISTORY
        history = np.zeros((len(targets), TCN_HISTORY), dtype=np.int64)
        history[has_history] = usable[count_before[has_history, None] - TCN_HISTORY + np.arange(TCN_HISTORY)]
        has_history &= segment[history[:, 0]] == segment[targets]
        history[~has_history] = first_history[~has_history]
        changed = has_history & (
            np.any(history != first_history, axis=1)
            | np.any(second_features[history] != first_features[history], axis=(1, 2))
            | np.any(second_features[targets] != first_features[targets], axis=1)
        )
        target_scores = first_target_scores.copy()
        if np.any(changed):
            target_scores[changed] = _tcn_scores(
                model, second_features[history[changed]], second_features[targets[changed]]
            )
        scores[row, index[targets]] = target_scores
        rejected[row, index[targets]] = target_scores > tau
    first_full = np.full(len(frame), np.nan)
    first_full[index[targets]] = first_target_scores
    return scores, rejected, first_full


def select_tcn_threshold(frame: pd.DataFrame, model: dict[str, object], mcc_band: float) -> dict[str, float]:
    """Подбирает порог: общая сетка по оценке первого прохода, два прохода на кандидата.

    Args:
        frame: Обучающий DataFrame (time, lat, lon, x, y, l).
        model: Результат fit_tcn_on_train.
        mcc_band: Допуск по MCC относительно лучшего значения.

    Returns:
        dict[str, float]: tau, train_mcc, train_f1, train_dl_m, best_mcc, band_size.
    """
    _, _, first = tcn_detect(frame, model, np.array([np.inf]))
    candidates = threshold_candidates(first[np.isfinite(first)])
    scores, rejected, _ = tcn_detect(frame, model, candidates)
    ground_truth_anomaly = frame["l"].astype(str).to_numpy() == "anomaly"
    records = [
        _threshold_record(float(tau), rejected[row], frame, ground_truth_anomaly, ~np.isnan(scores[row]))
        for row, tau in enumerate(candidates)
    ]
    return select_threshold_from_records(records, mcc_band)


tcn_scenarios = (
    (DATASET_NAME, dataset),
)

tcn_rows: list[dict[str, object]] = []
tcn_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in tcn_scenarios:
    train = bundle["train"]
    test = bundle["test"]
    anomaly_train = train["l"].astype(str).to_numpy() == "anomaly"

    print("=" * 72)
    print(f"TCN: {scenario_name}")
    started = time.perf_counter()
    def train_model() -> dict[str, object]:
        """Обучение, выбор порога и оценки train для печати медиан."""
        model = fit_tcn_on_train(train)
        selection = select_tcn_threshold(train, model, MCC_BAND)
        scores = tcn_detect(train, model, np.array([selection["tau"]]))[0]
        return {"model": model, "selection": selection, "train_scores": scores}
    state = load_or_train("tcn", train, train_model)
    tcn_model, selection, train_scores = state["model"], state["selection"], state["train_scores"]
    tau = np.array([selection["tau"]])
    if np.any(~anomaly_train) and np.any(anomaly_train):
        print(
            f"  медиана оценки на train: move {np.nanmedian(train_scores[0][~anomaly_train]):.4g}, "
            f"anomaly {np.nanmedian(train_scores[0][anomaly_train]):.4g}"
        )
    print(
        f"  окон обучения {tcn_model['n_train']}, калибровочных {tcn_model['n_calib']}, "
        f"эпох {tcn_model['epochs']} (ошибка калибровки {tcn_model['calib_loss']:.4g}); порог {tau[0]:.4g}: "
        f"MCC train {selection['train_mcc']:.4f} "
        f"(лучший {selection['best_mcc']:.4f}, в полосе {int(selection['band_size'])}), "
        f"DL train {selection['train_dl_m'] / 1000:.4f} км"
    )

    test_scores, test_rejected, _ = tcn_detect(test, tcn_model, tau)
    predicted_test = test_rejected[0]
    scored_test = ~np.isnan(test_scores[0])
    metrics = evaluate_test(test, predicted_test, scored_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м, без оценки {metrics['unscored_share']:.2%} "
        f"(время расчёта {elapsed:.1f} с)"
    )

    tcn_rows.append(test_report_row(scenario_name, metrics))
    tcn_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "scored_test": scored_test,
            "metrics": metrics,
        }
    )

tcn_summary = pd.DataFrame(tcn_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    tcn_summary.loc[:, report_columns],
    caption="TCN: сводные метрики на test",
)

if SHOW_PLOTS:
    _tcn_ideal_test = tcn_scenario_results[0]["test"]
    _tcn_map_tail = min(PLOT_TAIL, len(_tcn_ideal_test))
    print(f"\nИдеальная разметка: последние {nfmt(_tcn_map_tail)} точек теста (bbox: move + 5%)")
    _tcn_dummy_move = np.zeros(len(_tcn_ideal_test), dtype=bool)
    _tcn_truth_pred, _tcn_truth_gt = trajectory_series_pair(_tcn_ideal_test, _tcn_dummy_move)
    _tcn_truth_pred = slice_trajectory_series(_tcn_truth_pred, _tcn_map_tail)
    _tcn_truth_gt = slice_trajectory_series(_tcn_truth_gt, _tcn_map_tail)
    plot_track_maps(
        _tcn_truth_pred,
        _tcn_truth_gt,
        map_panel="ground_truth",
        title="TCN — идеальная разметка",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )

for item in tcn_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    if SHOW_PLOTS:
        print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
        prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"], item["scored_test"])
        prediction = slice_trajectory_series(prediction, tail)
        ground_truth = slice_trajectory_series(ground_truth, tail)
        plot_track_maps(
            prediction,
            ground_truth,
            map_panel="model",
            title=f"TCN — {item['name']}",
            figsize=MAP_FIGSIZE,
            annotate=False,
        )

## DONUT (вариационный автоэнкодер)

Признаки, нормировка и второй проход — как у HMM; окно и точки без оценки — как у TCN.

### Постановка

Вариационный автоэнкодер задаёт распределение окна признаков штатного хода. Обучение — на всех окнах обучающей выборки, аномальные точки исключаются из правдоподобия (modified ELBO, M-ELBO). Аномалия — точка с малой вероятностью восстановления.

### Модель и оценка аномальности

Окно $\mathbf x_W=(\mathbf u_{i-15},\ldots,\mathbf u_i)$ — 15 предыдущих принятых точек и точка $i$:

$$z\sim\mathcal N(0,I),\qquad \mathbf u_j\mid z\sim\mathcal N\bigl(\boldsymbol\mu_j(z),\ \mathrm{diag}\,\boldsymbol\sigma_j^2(z)\bigr),$$

$$q(z\mid\mathbf x_W)=\mathcal N\bigl(\boldsymbol\mu_q(\mathbf x_W),\ \mathrm{diag}\,\boldsymbol\sigma_q^2(\mathbf x_W)\bigr),$$

где $z$ — скрытый вектор размерности 8; декодер ($\boldsymbol\mu_j$, $\boldsymbol\sigma_j$) и кодировщик ($\boldsymbol\mu_q$, $\boldsymbol\sigma_q$) — по два полносвязных слоя из 100 нейронов; $\boldsymbol\sigma_j\ge10^{-4}$.

Обучение — максимум M-ELBO:

$$\mathcal L(\mathbf x_W)=\mathbb E_{q(z\mid\mathbf x_W)}\Bigl[\sum_{j\in W}\alpha_j\log p(\mathbf u_j\mid z)\Bigr]-\beta\,D_{\mathrm{KL}}\bigl(q(z\mid\mathbf x_W)\,\Vert\,\mathcal N(0,I)\bigr)$$

$$\beta=\frac{1}{|W|}\sum_{j\in W}\alpha_j,$$

где $\alpha_j=1$ у точки штатного хода и $\alpha_j=0$ у аномальной; при обучении ещё 1% точек случайно получает $\alpha_j=0$ — модель учится восстанавливать окно с невидимыми точками. Разметка обучающей выборки используется уже при обучении, а не только при выборе порога.

Оценка аномальности — минус вероятность восстановления последней точки окна:

$$s_i=-\frac{1}{L}\sum_{l=1}^{L}\log p(\mathbf u_i\mid z_l),\qquad z_l\sim q(z\mid\mathbf x_W),\qquad L=100,\qquad \hat a_i=\mathbf{1}[s_i>\tau].$$

Во втором проходе отвергнутые точки в окно не входят, поэтому восстанавливать пропущенные точки при оценке не требуется.

В отличие от LSTM-автоэнкодера и TCN, модель вероятностная: оценка — логарифм правдоподобия, отдельная калибровка ошибки не нужна; разметка участвует в обучении.

### Обучение и выбор порога

1. Окна — все окна обучающей выборки, кроме окон из одних аномальных точек; признаки при обучении считаются по последней точке штатного хода, чтобы след аномальной точки не попадал в признаки соседней; точка штатного хода после серии аномалий начинает отрезок заново, как в обучающих отрезках MPCT. Калибровочный блок, число окон, ранняя остановка (по M-ELBO) и начальное значение генератора — как у LSTM-автоэнкодера.
2. Подбор порога — как у AR(2); второй проход выполняется для каждого кандидата.

In [ ]:
DONUT_HISTORY = 15
DONUT_LATENT = 8
DONUT_HIDDEN = 100
DONUT_STD_MIN = 1e-4
DONUT_MISSING_RATE = 0.01
DONUT_SAMPLES = 100
DONUT_N_FEATURES = 3


class DonutVAE(nn.Module):
    """VAE окна признаков: кодировщик q(z|x) и декодер p(x|z) со средним и СКО каждого признака."""

    def __init__(self, dim: int, hidden: int, latent: int):
        """Создаёт кодировщик и декодер — по два полносвязных слоя.

        Args:
            dim: Размер плоского окна.
            hidden: Число нейронов в скрытых слоях.
            latent: Размерность скрытого вектора z.
        """
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU())
        self.q_mean = nn.Linear(hidden, latent)
        self.q_std = nn.Linear(hidden, latent)
        self.decoder = nn.Sequential(nn.Linear(latent, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU())
        self.x_mean = nn.Linear(hidden, dim)
        self.x_std = nn.Linear(hidden, dim)

    def encode(self, flat: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Среднее и СКО q(z|x)."""
        hidden = self.encoder(flat)
        return self.q_mean(hidden), nn.functional.softplus(self.q_std(hidden)) + DONUT_STD_MIN

    def decode(self, latent: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Среднее и СКО p(x|z); СКО не меньше DONUT_STD_MIN."""
        hidden = self.decoder(latent)
        return self.x_mean(hidden), nn.functional.softplus(self.x_std(hidden)) + DONUT_STD_MIN


def _donut_neg_m_elbo(model: DonutVAE, flat: torch.Tensor, alpha: torch.Tensor) -> torch.Tensor:
    """Минус M-ELBO, среднее по окнам: правдоподобие точек с весом α, KL с множителем β — долей α = 1 в окне."""
    q_mean, q_std = model.encode(flat)
    latent = q_mean + q_std * torch.randn_like(q_std)
    x_mean, x_std = model.decode(latent)
    log_prob = torch.distributions.Normal(x_mean, x_std).log_prob(flat)
    point_alpha = alpha.repeat_interleave(DONUT_N_FEATURES, dim=1)
    beta = alpha.mean(dim=1)
    kl = (-torch.log(q_std) + 0.5 * (q_std**2 + q_mean**2) - 0.5).sum(dim=1)
    return (-(point_alpha * log_prob).sum(dim=1) + beta * kl).mean()


def fit_donut_on_train(train: pd.DataFrame) -> dict[str, object]:
    """Обучает DONUT по M-ELBO на всех окнах обучающей выборки.

    Признаки — по последней точке штатного хода (след аномальной точки не попадает в признаки соседней); точка
    штатного хода после серии аномалий начинает отрезок заново, как в move_segments; нормировка — по штатному ходу,
    как у DBSCAN. α = 1 у штатного хода, 0 у аномалии; при обучении ещё DONUT_MISSING_RATE точек
    случайно получают α = 0. Калибровочный блок, число окон и ранняя остановка — как у LSTM-автоэнкодера.

    Args:
        train: Обучающий DataFrame с колонками x, y, time и l.

    Returns:
        dict[str, object]: center, scale, model, noise — фиксированные выборки ε для оценки, epochs, calib_loss,
        n_train, n_calib.
    """
    move_all = train["l"].astype(str).to_numpy() == "move"
    index, first = dbscan_segments(train, np.ones(len(train), dtype=bool))
    move = move_all[index]
    position = np.arange(len(index))
    # Признаки по последней точке штатного хода; первая точка отрезка считается опорной, отрезки не смешиваются.
    anchor = move | first
    last_move = np.maximum.accumulate(np.where(anchor, position, -1))
    # Точка штатного хода после серии аномалий — без предшественника: прыжок через серию не попадает в признаки.
    restart = move & ~first & ~np.r_[True, move[:-1]]
    before = np.where(first | restart, -1, np.r_[-1, last_move[:-1]])
    raw = dbscan_features(enu_xy_from_frame(train)[index], time_seconds_from_frame(train)[index], before)
    scored = ~np.isnan(raw[:, 0])
    starts = rpca_window_starts(first, scored, DONUT_HISTORY + 1)
    cover = starts[:, None] + np.arange(DONUT_HISTORY + 1)
    keep = move[cover].any(axis=1)
    starts, cover = starts[keep], cover[keep]
    if len(starts) < 20 or not np.any(move & scored):
        raise RuntimeError("Мало окон для DONUT.")
    center, scale = robust_params(raw[move & scored])
    features = apply_robust(raw, center, scale).astype(np.float32)
    calib_rows = np.arange(int(len(starts) * (1.0 - NN_CALIB_FRACTION)), len(starts))
    fit_rows = np.flatnonzero(starts + DONUT_HISTORY + 1 <= starts[calib_rows[0]])
    fit_rows = take_window_starts(fit_rows, N_WINDOWS, SEED)
    flat = features[cover].reshape(len(starts), -1)
    alpha = move[cover].astype(np.float32)

    torch.manual_seed(SEED)
    model = DonutVAE(flat.shape[1], DONUT_HIDDEN, DONUT_LATENT).to(DEVICE)
    loader = DataLoader(
        TensorDataset(torch.from_numpy(flat[fit_rows]), torch.from_numpy(alpha[fit_rows])),
        batch_size=TORCH_BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    calib_flat = torch.from_numpy(flat[calib_rows]).to(DEVICE)
    calib_alpha = torch.from_numpy(alpha[calib_rows]).to(DEVICE)
    training = train_with_early_stopping(
        model,
        loader,
        lambda net, batch: _donut_neg_m_elbo(
            net, batch[0], batch[1] * ~(torch.rand(batch[1].shape, device=DEVICE) < DONUT_MISSING_RATE)
        ),
        lambda net: float(_donut_neg_m_elbo(net, calib_flat, calib_alpha)),
    )
    # Одни и те же выборки ε для всех окон: оценка воспроизводима и не зависит от состава пакета.
    noise = torch.randn(DONUT_SAMPLES, DONUT_LATENT, generator=torch.Generator().manual_seed(SEED)).to(DEVICE)
    return {
        "center": center,
        "scale": scale,
        "model": model,
        "noise": noise,
        "epochs": training["epochs"],
        "calib_loss": training["calib_loss"],
        "n_train": len(fit_rows),
        "n_calib": len(calib_rows),
    }


def _donut_scores(model: dict[str, object], history: np.ndarray, target: np.ndarray) -> np.ndarray:
    """Минус средний логарифм вероятности признаков последней точки окна по выборкам z ~ q(z|x)."""
    if len(target) == 0:
        return np.empty(0)
    flat = np.concatenate([history.reshape(len(history), -1), target], axis=1).astype(np.float32)
    vae, noise = model["model"], model["noise"]
    pieces = []
    with torch.no_grad():
        for start in range(0, len(flat), TORCH_INFER_BATCH):
            batch = torch.from_numpy(flat[start : start + TORCH_INFER_BATCH]).to(DEVICE)
            q_mean, q_std = vae.encode(batch)
            latent = q_mean[:, None, :] + q_std[:, None, :] * noise[None, :, :]
            x_mean, x_std = vae.decode(latent)
            last = slice(-DONUT_N_FEATURES, None)
            log_prob = torch.distributions.Normal(x_mean[..., last], x_std[..., last]).log_prob(batch[:, None, last])
            pieces.append((-log_prob.sum(dim=2).mean(dim=1)).cpu().numpy())
    return np.concatenate(pieces)


def donut_detect(
    frame: pd.DataFrame, model: dict[str, object], taus: np.ndarray
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Оценка точки — минус вероятность восстановления её признаков, в два прохода, сразу для нескольких порогов.

    Окно — 15 предыдущих точек отрезка и точка; во втором проходе — 15 последних принятых точек, признаки
    по принятым точкам; пересчитываются только окна с изменёнными точками или признаками. Если принятых точек
    в отрезке до точки меньше 15, остаётся оценка первого прохода.

    Args:
        frame: DataFrame с колонками x, y и time.
        model: Результат fit_donut_on_train.
        taus: Пороги; np.inf — без отвергнутых точек.

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray]: оценка второго прохода (C, N), NaN — точка без оценки;
        маска отвергнутых (C, N); оценка первого прохода (N,).
    """
    index, first = dbscan_segments(frame, np.ones(len(frame), dtype=bool))
    xy = enu_xy_from_frame(frame)[index]
    time_sec = time_seconds_from_frame(frame)[index]
    position = np.arange(len(index))
    segment = np.cumsum(first)
    first_features = apply_robust(
        dbscan_features(xy, time_sec, np.where(first, -1, position - 1)), model["center"], model["scale"]
    )
    starts = rpca_window_starts(first, ~np.isnan(first_features[:, 0]), DONUT_HISTORY + 1)
    targets = starts + DONUT_HISTORY
    first_history = starts[:, None] + np.arange(DONUT_HISTORY)
    first_target_scores = _donut_scores(model, first_features[first_history], first_features[targets])
    first_scores = np.full(len(index), np.nan)
    first_scores[targets] = first_target_scores
    thresholds = np.atleast_1d(np.asarray(taus, dtype=float))
    scores = np.full((len(thresholds), len(frame)), np.nan)
    rejected = np.zeros((len(thresholds), len(frame)), dtype=bool)
    for row, tau in enumerate(thresholds):
        accepted = ~(first_scores > tau)
        last_accepted = np.maximum.accumulate(np.where(accepted, position, -1))
        before = np.where(first, -1, np.r_[-1, last_accepted[:-1]])
        second_features = apply_robust(dbscan_features(xy, time_sec, before), model["center"], model["scale"])
        usable = np.flatnonzero(accepted & ~np.isnan(second_features[:, 0]))
        count_before = np.searchsorted(usable, targets)
        has_history = count_before >= DONUT_HISTORY
        history = np.zeros((len(targets), DONUT_HISTORY), dtype=np.int64)
        history[has_history] = usable[count_before[has_history, None] - DONUT_HISTORY + np.arange(DONUT_HISTORY)]
        has_history &= segment[history[:, 0]] == segment[targets]
        history[~has_history] = first_history[~has_history]
        changed = has_history & (
            np.any(history != first_history, axis=1)
            | np.any(second_features[history] != first_features[history], axis=(1, 2))
            | np.any(second_features[targets] != first_features[targets], axis=1)
        )
        target_scores = first_target_scores.copy()
        if np.any(changed):
            target_scores[changed] = _donut_scores(
                model, second_features[history[changed]], second_features[targets[changed]]
            )
        scores[row, index[targets]] = target_scores
        rejected[row, index[targets]] = target_scores > tau
    first_full = np.full(len(frame), np.nan)
    first_full[index[targets]] = first_target_scores
    return scores, rejected, first_full


def select_donut_threshold(frame: pd.DataFrame, model: dict[str, object], mcc_band: float) -> dict[str, float]:
    """Подбирает порог: общая сетка по оценке первого прохода, два прохода на кандидата.

    Args:
        frame: Обучающий DataFrame (time, lat, lon, x, y, l).
        model: Результат fit_donut_on_train.
        mcc_band: Допуск по MCC относительно лучшего значения.

    Returns:
        dict[str, float]: tau, train_mcc, train_f1, train_dl_m, best_mcc, band_size.
    """
    _, _, first = donut_detect(frame, model, np.array([np.inf]))
    candidates = threshold_candidates(first[np.isfinite(first)])
    scores, rejected, _ = donut_detect(frame, model, candidates)
    ground_truth_anomaly = frame["l"].astype(str).to_numpy() == "anomaly"
    records = [
        _threshold_record(float(tau), rejected[row], frame, ground_truth_anomaly, ~np.isnan(scores[row]))
        for row, tau in enumerate(candidates)
    ]
    return select_threshold_from_records(records, mcc_band)


donut_scenarios = (
    (DATASET_NAME, dataset),
)

donut_rows: list[dict[str, object]] = []
donut_scenario_results: list[dict[str, object]] = []

for scenario_name, bundle in donut_scenarios:
    train = bundle["train"]
    test = bundle["test"]
    anomaly_train = train["l"].astype(str).to_numpy() == "anomaly"

    print("=" * 72)
    print(f"DONUT: {scenario_name}")
    started = time.perf_counter()
    def train_model() -> dict[str, object]:
        """Обучение, выбор порога и оценки train для печати медиан."""
        model = fit_donut_on_train(train)
        selection = select_donut_threshold(train, model, MCC_BAND)
        scores = donut_detect(train, model, np.array([selection["tau"]]))[0]
        return {"model": model, "selection": selection, "train_scores": scores}
    state = load_or_train("donut", train, train_model)
    donut_model, selection, train_scores = state["model"], state["selection"], state["train_scores"]
    tau = np.array([selection["tau"]])
    if np.any(~anomaly_train) and np.any(anomaly_train):
        print(
            f"  медиана оценки на train: move {np.nanmedian(train_scores[0][~anomaly_train]):.4g}, "
            f"anomaly {np.nanmedian(train_scores[0][anomaly_train]):.4g}"
        )
    print(
        f"  окон обучения {donut_model['n_train']}, калибровочных {donut_model['n_calib']}, "
        f"эпох {donut_model['epochs']} (−M-ELBO калибровки {donut_model['calib_loss']:.4g}); порог {tau[0]:.4g}: "
        f"MCC train {selection['train_mcc']:.4f} "
        f"(лучший {selection['best_mcc']:.4f}, в полосе {int(selection['band_size'])}), "
        f"DL train {selection['train_dl_m'] / 1000:.4f} км"
    )

    test_scores, test_rejected, _ = donut_detect(test, donut_model, tau)
    predicted_test = test_rejected[0]
    scored_test = ~np.isnan(test_scores[0])
    metrics = evaluate_test(test, predicted_test, scored_test)
    elapsed = time.perf_counter() - started
    print(
        f"  тест: F1 {metrics['f1']:.4f}, DL {metrics['distance_loss_m']:.2f} м, "
        f"RMSE {metrics['rmse_m']:.2f} м, без оценки {metrics['unscored_share']:.2%} "
        f"(время расчёта {elapsed:.1f} с)"
    )

    donut_rows.append(test_report_row(scenario_name, metrics))
    donut_scenario_results.append(
        {
            "name": scenario_name,
            "test": test,
            "predicted_test": predicted_test,
            "scored_test": scored_test,
            "metrics": metrics,
        }
    )

donut_summary = pd.DataFrame(donut_rows)
report_columns = ["Сценарий"] + [key for key, _ in TEST_REPORT_METRICS]
display_test_report_table(
    donut_summary.loc[:, report_columns],
    caption="DONUT: сводные метрики на test",
)

if SHOW_PLOTS:
    _donut_ideal_test = donut_scenario_results[0]["test"]
    _donut_map_tail = min(PLOT_TAIL, len(_donut_ideal_test))
    print(f"\nИдеальная разметка: последние {nfmt(_donut_map_tail)} точек теста (bbox: move + 5%)")
    _donut_dummy_move = np.zeros(len(_donut_ideal_test), dtype=bool)
    _donut_truth_pred, _donut_truth_gt = trajectory_series_pair(_donut_ideal_test, _donut_dummy_move)
    _donut_truth_pred = slice_trajectory_series(_donut_truth_pred, _donut_map_tail)
    _donut_truth_gt = slice_trajectory_series(_donut_truth_gt, _donut_map_tail)
    plot_track_maps(
        _donut_truth_pred,
        _donut_truth_gt,
        map_panel="ground_truth",
        title="DONUT — идеальная разметка",
        figsize=MAP_FIGSIZE,
        annotate=False,
    )

for item in donut_scenario_results:
    tail = min(PLOT_TAIL, len(item["test"]))
    if SHOW_PLOTS:
        print(f"  Карта разметки модели: последние {nfmt(tail)} точек теста (bbox: move + 5%)")
        prediction, ground_truth = trajectory_series_pair(item["test"], item["predicted_test"], item["scored_test"])
        prediction = slice_trajectory_series(prediction, tail)
        ground_truth = slice_trajectory_series(ground_truth, tail)
        plot_track_maps(
            prediction,
            ground_truth,
            map_panel="model",
            title=f"DONUT — {item['name']}",
            figsize=MAP_FIGSIZE,
            annotate=False,
        )

## Общая сводка по моделям



In [ ]:
def _model_summary_catalog() -> list[tuple[str, str]]:
    """Порядок строк в общей сводке: (название модели, имя DataFrame в kernel).

    Returns:
        list[tuple[str, str]]: Пары (название модели, имя переменной-сводки).
    """
    return [
        ("MPCT", "mpct_summary"),
        ("AR(2)", "ar_summary"),
        ("ARIMA(2,1,1)", "arima_summary"),
        ("Фильтр Калмана", "kalman_summary"),
        ("Gaussian process", "gp_summary"),
        ("GP-EKF", "gp_ekf_summary"),
        ("HMM", "hmm_summary"),
        ("Hampel", "hampel_summary"),
        ("DBSCAN", "dbscan_summary"),
        ("LOF", "lof_summary"),
        ("Isolation Forest", "if_summary"),
        ("One-Class SVM", "ocsvm_summary"),
        ("Robust PCA", "rpca_summary"),
        ("LSTM-автоэнкодер", "lstm_summary"),
        ("TCN", "tcn_summary"),
        ("DONUT", "donut_summary"),
    ]


def build_master_test_table(scenario_index: int) -> pd.DataFrame:
    """Сводная таблица test по моделям.

    Args:
        scenario_index: Индекс сценария в сводках моделей.

    Returns:
        pd.DataFrame: Строки «Модель» + метрики test.
    """
    rows: list[dict[str, object]] = []
    for model_name, var_name in _model_summary_catalog():
        table = globals().get(var_name)
        if not isinstance(table, pd.DataFrame) or len(table) <= scenario_index:
            continue
        summary_row = table.iloc[scenario_index]
        row: dict[str, object] = {"Модель": model_name}
        for key, _ in TEST_REPORT_METRICS:
            row[key] = summary_row.get(key, float("nan"))
        rows.append(row)
    return pd.DataFrame(rows)


def display_master_test_table(frame: pd.DataFrame, caption: str) -> None:
    """Таблица сводки: колонка «Модель» и метрики test с форматированием fmt_num.

    Args:
        frame: Таблица из build_master_test_table.
        caption: Подпись таблицы.
    """
    metric_labels = {key: label for key, label in TEST_REPORT_METRICS}
    renamed = frame.rename(columns=metric_labels)
    for key, label in TEST_REPORT_METRICS:
        if label in renamed.columns:
            renamed[label] = renamed[label].map(lambda v, k=key: _format_metric_value(k, v))
    styled = renamed.style
    if caption:
        styled = styled.set_caption(caption)
    display(styled)


_master_scenarios = ((0, f"{DATASET_NAME} (train {nfmt(len(train))}, test {nfmt(len(test))} точек)"),)

for scenario_index, scenario_caption in _master_scenarios:
    master_table = build_master_test_table(scenario_index)
    print("=" * 72)
    print(scenario_caption)
    if master_table.empty:
        print("  (нет данных — выполните ячейки моделей выше)")
        continue
    display_master_test_table(
        master_table,
        caption=f"Метрики test, все модели: {scenario_caption}",
    )


## Сводка: сортировка по F1 и Distance loss

Итоговая таблица test: строки моделей по убыванию F1, затем по возрастанию ошибки длины траектории (DL).



In [ ]:
_master_sort_specs = (
    ("f1", False, "F1 (убывание)"),
    ("distance_loss_m", True, "Distance loss (возрастание)"),
)

for sort_key, ascending, sort_title in _master_sort_specs:
    print("=" * 72)
    print(f"Общая сводка — сортировка: {sort_title}")
    for scenario_index, scenario_caption in _master_scenarios:
        master_table = build_master_test_table(scenario_index)
        print("=" * 72)
        print(scenario_caption)
        if master_table.empty:
            print("  (нет данных — выполните ячейки моделей выше)")
            continue
        sorted_table = master_table.sort_values(
            sort_key, ascending=ascending, na_position="last"
        ).reset_index(drop=True)
        display_master_test_table(
            sorted_table,
            caption=f"{sort_title}: {scenario_caption}",
        )



## Итоговые таблицы test

### Геопространственные метрики (по возрастанию DL)

| Модель | RMSE, м | Max(d_i), м | DL, м |
| --- | --- | --- | --- |
| TCN | 31 167.6 | 58 565.04 | 259 905.39 |
| DONUT | 30 218.53 | 58 519.42 | 279 001.31 |
| Gaussian process | 84 425.06 | 473 118.46 | 2 979 142.9 |
| ARIMA(2,1,1) | 79 937.26 | 473 118.46 | 3 117 421.56 |
| HMM | 97 690.99 | 924 578.82 | 6 610 074.16 |
| LSTM-автоэнкодер | 228 917.39 | 927 606.9 | 10 912 832.12 |
| Isolation Forest | 114 593.03 | 925 219.76 | 11 995 819.35 |
| AR(2) | 176 947.93 | 1 000 117.28 | 12 801 590.35 |
| GP-EKF | 192 082.87 | 1 004 326.86 | 12 996 571.67 |
| Фильтр Калмана | 200 583.62 | 1 004 326.86 | 13 887 583.38 |
| One-Class SVM | 164 979.44 | 927 443.02 | 16 642 528.37 |
| DBSCAN | 200 299.48 | 927 575.21 | 23 531 505.13 |
| Robust PCA | 200 077.35 | 927 570.27 | 27 644 710.55 |
| LOF | 190 996.41 | 1 000 309.45 | 32 376 592.88 |
| Hampel | 200 530.09 | 1 032 230.96 | 53 948 074.66 |
| MPCT | 228 905.27 | 6 380 999.81 | 92 793 153.6 |

### Временные метрики (по возрастанию средней задержки)

| Модель | Задержка мед., с | Задержка ср., с | Задержка p95, с | Задержка макс., с | Вовремя | С опозданием или пропущено |
| --- | --- | --- | --- | --- | --- | --- |
| MPCT | 15.00 | 674.00 | 3 617 | 6 480 | 19 | 10 |
| LSTM-автоэнкодер | 30.00 | 1 898.87 | 8 673.6 | 25 552 | 14 | 15 |
| Robust PCA | 30.00 | 1 981.47 | 10 353.6 | 25 552 | 14 | 15 |
| DONUT | 170.00 | 2 004.2 | 8 925.6 | 25 762 | 14 | 15 |
| Isolation Forest | 25.00 | 2 068.94 | 10 843 | 25 552 | 13 | 16 |
| Hampel | 30.00 | 2 090.15 | 10 948.8 | 25 572 | 16 | 13 |
| HMM | 55.00 | 2 091.38 | 10 828 | 25 552 | 13 | 16 |
| DBSCAN | 55.00 | 2 123.94 | 10 858 | 25 552 | 13 | 16 |
| TCN | 145.00 | 2 203 | 11 519.2 | 25 762 | 15 | 14 |
| One-Class SVM | 40.00 | 2 212.2 | 11 830.6 | 25 552 | 14 | 15 |
| Gaussian process | 40.00 | 2 295.54 | 12 552.8 | 25 562 | 16 | 13 |
| AR(2) | 25.00 | 2 477.67 | 13 621.4 | 25 552 | 17 | 12 |
| Фильтр Калмана | 35.00 | 2 481 | 13 632.4 | 25 552 | 17 | 12 |
| GP-EKF | 35.00 | 2 481 | 13 632.4 | 25 552 | 17 | 12 |
| ARIMA(2,1,1) | 40.00 | 2 486 | 13 636.9 | 25 562 | 17 | 12 |
| LOF | 30.00 | 3 302.59 | 26 399 | 29 792 | 12 | 17 |

### Метрики классификации (по убыванию F1)

| Модель | Precision | Recall | F1 | MCC | Аномалий без оценки, % |
| --- | --- | --- | --- | --- | --- |
| GP-EKF | 0.8192 | 0.7316 | 0.7729 | 0.6997 | 6.03 |
| Фильтр Калмана | 0.8298 | 0.7146 | 0.7679 | 0.6960 | 5.58 |
| AR(2) | 0.7984 | 0.7221 | 0.7583 | 0.6780 | 9.46 |
| ARIMA(2,1,1) | 0.8305 | 0.6918 | 0.7548 | 0.6832 | 11.85 |
| Gaussian process | 0.8336 | 0.6822 | 0.7503 | 0.6797 | 7.81 |
| TCN | 0.8485 | 0.6237 | 0.7189 | 0.6576 | 15.61 |
| One-Class SVM | 0.5829 | 0.9105 | 0.7108 | 0.6017 | 2.64 |
| Isolation Forest | 0.5725 | 0.9271 | 0.7079 | 0.6004 | 2.64 |
| DONUT | 0.7565 | 0.6149 | 0.6784 | 0.5930 | 15.61 |
| LSTM-автоэнкодер | 0.7444 | 0.6071 | 0.6688 | 0.5714 | 6.23 |
| Robust PCA | 0.8717 | 0.5283 | 0.6579 | 0.6021 | 6.23 |
| HMM | 0.5193 | 0.8880 | 0.6554 | 0.5199 | 2.64 |
| MPCT | 0.9808 | 0.4872 | 0.6510 | 0.6286 | 0.00 |
| DBSCAN | 0.7757 | 0.5326 | 0.6315 | 0.5432 | 2.64 |
| LOF | 0.6679 | 0.5003 | 0.5721 | 0.4511 | 2.64 |
| Hampel | 0.5246 | 0.1761 | 0.2637 | 0.1802 | 1.93 |

## Разбор результатов и выводы

### Условия сравнения

Все модели обучаются на одном обучающем блоке (66 622 точки, доля аномалий 51.5%) и проверяются на одной тестовой выборке (86 263 точки, доля аномалий 27.5%, 29 эпизодов аномалии). Аномалии в данных — в основном длинные эпизоды: средняя длина эпизода в тестовой выборке — около 820 точек, самый длинный необнаруженный эпизод длится 25 552 с (7.1 ч). Метрики классификации считаются только по точкам с оценкой, поэтому у моделей разные подмножества тестовой выборки. Тестовая выборка одна, и 95%-й интервал $F_1$ и MCC, оценённый блочным бутстрепом для фильтра Калмана и Isolation Forest, — около $\pm0.08$; различия меньше 0.1 не считаются значимыми.

### Метрики классификации

Лучшие $F_1$ и MCC — у моделей прогноза: GP-EKF и фильтр Калмана ($F_1=0.77$, MCC 0.70), AR(2) (0.76; 0.68), ARIMA(2,1,1) и гауссовский процесс (0.75; 0.68). Различия между ними лежат внутри погрешности. Модели прогноза сочетают высокую точность (0.80–0.83) и полноту (0.68–0.73): аномальная точка далека от прогноза, а возврат на трек после эпизода обеспечивает проверка достижимости при перезапуске.

Следующая группа — TCN (0.72; 0.66), One-Class SVM и Isolation Forest (0.71; 0.60). У One-Class SVM и Isolation Forest полнота наибольшая (0.91–0.93) при точности 0.57–0.58: признаки одной точки отделяют аномалию от штатного хода с запасом, но отвергается и часть штатного хода — вероятно, участки с редкими значениями признаков (манёвры, стоянки). Robust PCA и MPCT — противоположный случай: точность 0.87 и 0.98 при полноте 0.53 и 0.49.

Наименьшие значения — у LOF ($F_1=0.57$, MCC 0.45) и фильтра Хэмпела (0.26; 0.18). Фильтр Хэмпела в этой задаче неприменим: окно лежит внутри длинного эпизода, и аномалии сравниваются с аномалиями.

### Геопространственные метрики

Наименьшие RMSE и ошибка длины траектории — у TCN и DONUT (RMSE около 30 км, DL 0.26–0.28 млн м, Max$(d_i)$ 59 км). Это преимущество частично следует из окна: у этих моделей без оценки 15.6% аномальных точек, а точки без оценки в заполненной траектории восстанавливаются интерполяцией, как отвергнутые. Среди моделей с малой долей точек без оценки лучшая геометрия — у гауссовского процесса и ARIMA(2,1,1) (RMSE 80–84 км, DL около 3 млн м) и у HMM (98 км; 6.61 млн м).

У большинства моделей Max$(d_i)$ — 925–1032 км: принимается один и тот же далёкий смещённый эпизод. Фильтр Калмана, GP-EKF и AR(2) при лучших метриках классификации дают DL 12.8–13.9 млн м: метрики классификации не отражают того, что несколько принятых далёких эпизодов определяют ошибку длины. Наибольшая ошибка длины — у MPCT (92.79 млн м, Max$(d_i)$ 6381 км): модель проверяет только скорость между выбранными точками и принимает длинные смещённые эпизоды, согласованные внутри.

### Временные метрики

Медианная задержка обнаружения у всех моделей, кроме TCN и DONUT, — 15–55 с, то есть 2–6 точек; у TCN и DONUT — 145–170 с. Средняя задержка (1 900–3 300 с) и 95-й перцентиль (8 700–26 400 с) определяются пропущенными эпизодами: у большинства моделей максимальная задержка 25 552 с — это один и тот же длинный эпизод, не обнаруженный ни в одной точке. Его обнаруживает только MPCT: максимальная задержка 6 480 с, наименьшая средняя задержка (674 с) и наибольшее число эпизодов, обнаруженных в первой точке (19 из 29).

### Свойства отдельных моделей

- GP-EKF совпадает с фильтром Калмана в пределах погрешности: длины корреляции поправки GP малы (0.07–0.13 м/с), доля дисперсии ускорения, объяснённая поправкой, отрицательна. В этих данных ускорение не зависит от скорости, и поправка GP не даёт улучшения.
- AR(2) после оценки параметров по участкам с шагом 10 с сводится к модели постоянной скорости: $\phi\approx(2,-1)$, $\sigma$ 3–4 м.
- HMM: оценка аномальности — апостериорная вероятность режима аномалии — почти нулевая на обоих классах (медианы на обучающей выборке $2.9\cdot10^{-12}$ и $6.6\cdot10^{-12}$). Внутри длинного эпизода признаки одной точки неотличимы от штатного хода, и вероятность перехода не удерживает цепь в режимах аномалии.
- MPCT — единственная модель без непрерывной оценки аномальности и без точек без оценки; наибольшая точность (0.98) и лучшие временные метрики при наибольшей ошибке длины.
- Нейросети (LSTM-автоэнкодер, TCN, DONUT) обучаются и рассчитываются не дольше моделей прогноза и по метрикам классификации не превосходят моделей прогноза.

### Ограничения

Данные одного судна; тестовая выборка одна и содержит 29 эпизодов аномалии; доля аномалий в обучающем блоке (51.5%) почти вдвое выше, чем в тестовой выборке (27.5%). Метрики классификации у моделей с окном посчитаны на подмножестве без начальных точек отрезков.

### Выводы

1. Лучший компромисс метрик классификации, скорости и простоты — фильтр Калмана постоянной скорости: два параметра, $F_1=0.77$, MCC 0.70. Усложнения модели прогноза (GP-EKF, гауссовский процесс, ARIMA) значимо не улучшают классификацию.
2. Для геометрии траекторий из моделей с малой долей точек без оценки предпочтительны гауссовский процесс и ARIMA(2,1,1): DL в 4–5 раз меньше, чем у фильтра Калмана, при тех же $F_1$ и MCC.
3. Для полноты предпочтительны One-Class SVM и Isolation Forest (полнота 0.91–0.93) ценой точности 0.57–0.58.
4. Для раннего обнаружения эпизодов предпочтителен MPCT (19 из 29 эпизодов — в первой точке), но он принимает длинные смещённые эпизоды и даёт наибольшую ошибку длины.
5. Основной источник ошибок всех моделей — длинные смещённые эпизоды, согласованные внутри: признаки одной точки и проверка скорости их не различают, и такие эпизоды определяют RMSE, ошибку длины и максимальную задержку.
6. Возможно, сильные стороны разных моделей удастся объединить в ансамбле моделей.